# NS-HAGRAG public-release notebook

> **Security and data notice:** Credentials and private paths have been removed. Set `HF_TOKEN`, `NEO4J_URI`, `NEO4J_USERNAME`, and `NEO4J_PASSWORD` as environment variables before running. Keep `.env` files, UMLS resources, datasets, checkpoints, generated results, and Neo4j backups out of version control. UMLS content is licence-restricted and must not be redistributed. Only load pickle or checkpoint files that you created or trust. The graph-clearing function is destructive; keep `clear_existing=False` unless deletion is intentional.


# NS-HAGRAG: Neurosymbolic Layer for HAGRAG

## Overview
This notebook adds a **neurosymbolic verification layer** to your existing HAGRAG system.

### What this does:
1. Takes your existing `ArchRAGQueryEngine` (unchanged)
2. Adds symbolic verification on top (new)
3. Provides explainability traces (new)

### Integration:
- **No changes** to your existing PDF processing, KG construction, or retrieval
- **One wrapper class** that enhances query answering
- **Plugs in** with a single line of code

---

It still does the stuff for both Biosq and Pubmed, evalautes both on questions but the issue is it has problem in output. output is not clean and has special characters in it

To fix output draft and final both,refer to 1.2-w

In [ ]:
# ============================================================
# CELL 1: PDFProcessor with Checkpointing
# ============================================================
# Replace your current Cell 1 with this version

import os
import pdfplumber
from tqdm import tqdm
import json
import pickle
import hashlib
from typing import List, Dict, Optional
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class PDFProcessor:
    """Handles PDF processing and text extraction with checkpointing"""
    
    def __init__(self, pdf_dir: str = './test/', checkpoint_dir: str = './checkpoints/hagragpipeline'):
        self.pdf_dir = pdf_dir
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        
        # Checkpoint files
        self.texts_checkpoint = self.checkpoint_dir / 'extracted_texts.json'
        self.chunks_checkpoint = self.checkpoint_dir / 'chunks.json'
        self.config_checkpoint = self.checkpoint_dir / 'config.json'
        
    def _get_pdf_hash(self) -> str:
        """Generate hash of PDF directory state for cache validation"""
        pdf_files = sorted([f for f in os.listdir(self.pdf_dir) if f.endswith('.pdf')])
        hash_input = ''.join(pdf_files) + str(len(pdf_files))
        return hashlib.md5(hash_input.encode()).hexdigest()
    
    def _load_config(self) -> Optional[Dict]:
        """Load checkpoint config"""
        if self.config_checkpoint.exists():
            with open(self.config_checkpoint, 'r') as f:
                return json.load(f)
        return None
    
    def _save_config(self, pdf_hash: str, num_texts: int, num_chunks: int):
        """Save checkpoint config"""
        config = {
            'pdf_hash': pdf_hash,
            'num_texts': num_texts,
            'num_chunks': num_chunks,
            'pdf_dir': self.pdf_dir
        }
        with open(self.config_checkpoint, 'w') as f:
            json.dump(config, f, indent=2)
    
    def extract_texts(self, force_reprocess: bool = False) -> List[str]:
        """Extract text from all PDFs with checkpointing"""
        
        current_hash = self._get_pdf_hash()
        config = self._load_config()
        
        # Check if we can use cached texts
        if not force_reprocess and self.texts_checkpoint.exists() and config:
            if config.get('pdf_hash') == current_hash:
                logging.info(f"✓ Loading cached texts from {self.texts_checkpoint}")
                with open(self.texts_checkpoint, 'r', encoding='utf-8') as f:
                    texts = json.load(f)
                logging.info(f"✓ Loaded {len(texts)} texts from cache")
                return texts
            else:
                logging.info("PDF directory changed, reprocessing...")
        
        # Process PDFs
        texts = []
        pdf_files = sorted([f for f in os.listdir(self.pdf_dir) if f.endswith('.pdf')])
        
        if not pdf_files:
            raise ValueError(f"No PDF files found in {self.pdf_dir}")
        
        logging.info(f"Processing {len(pdf_files)} PDF files...")
        
        for filename in tqdm(pdf_files, desc="Extracting text from PDFs"):
            pdf_path = os.path.join(self.pdf_dir, filename)
            try:
                with pdfplumber.open(pdf_path) as pdf:
                    doc_text = []
                    for page_num, page in enumerate(pdf.pages):
                        text = page.extract_text()
                        if text and text.strip():
                            text = text.strip()
                            doc_text.append(text)
                    
                    if doc_text:
                        full_text = '\n\n'.join(doc_text)
                        texts.append(full_text)
                        
            except Exception as e:
                logging.error(f"Error processing {filename}: {e}")
                continue
        
        # Save checkpoint
        logging.info(f"Saving {len(texts)} texts to checkpoint...")
        with open(self.texts_checkpoint, 'w', encoding='utf-8') as f:
            json.dump(texts, f, ensure_ascii=False, indent=2)
        
        logging.info(f"✓ Extracted {len(texts)} document texts from {len(pdf_files)} PDFs")
        return texts
    
    def create_chunks(self, texts: List[str], chunk_size: int = 1024, 
                     chunk_overlap: int = 20, force_reprocess: bool = False) -> List[str]:
        """Split texts into chunks with checkpointing"""
        
        from langchain_text_splitters import RecursiveCharacterTextSplitter
        
        current_hash = self._get_pdf_hash()
        config = self._load_config()
        
        # Check if we can use cached chunks
        if not force_reprocess and self.chunks_checkpoint.exists() and config:
            if config.get('pdf_hash') == current_hash:
                logging.info(f"✓ Loading cached chunks from {self.chunks_checkpoint}")
                with open(self.chunks_checkpoint, 'r', encoding='utf-8') as f:
                    chunks = json.load(f)
                logging.info(f"✓ Loaded {len(chunks)} chunks from cache")
                return chunks
        
        # Create chunks
        logging.info(f"Creating chunks (size={chunk_size}, overlap={chunk_overlap})...")
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
        
        documents = splitter.create_documents(texts)
        chunks = [doc.page_content for doc in documents]
        
        # Save checkpoint
        logging.info(f"Saving {len(chunks)} chunks to checkpoint...")
        with open(self.chunks_checkpoint, 'w', encoding='utf-8') as f:
            json.dump(chunks, f, ensure_ascii=False, indent=2)
        
        # Save config
        self._save_config(current_hash, len(texts), len(chunks))
        
        logging.info(f"✓ Created {len(chunks)} chunks from {len(texts)} documents")
        
        return chunks

print("✓ Cell 1 loaded: PDFProcessor with checkpointing")

In [ ]:
# ============================================================
# CELL 2: TripletExtractor + ArchRAGCHNSW 
# ============================================================
# FIXED VERSION - includes exact NS-HAGRAG extract_claim_triplets
# Combines: Baseline checkpointing + NS-HAGRAG claim extraction

from typing import List, Tuple, Dict, Optional, Any
import json
import re
import logging
from pathlib import Path
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
from sklearn.neighbors import NearestNeighbors
import heapq

class TripletExtractor:
    """
    Handles knowledge graph triplet extraction + answer->claim triplets.
    Combines:
      - Baseline checkpointing for KG building
      - NS-HAGRAG extract_claim_triplets for verification
    """

    def __init__(self, llm_pipeline, checkpoint_dir: str = './checkpoints/hagragpipeline'):
        self.llm_pipeline = llm_pipeline
        self.extraction_stats = {"success": 0, "failed": 0, "total": 0}
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        
        # Progress tracking for KG building
        self.progress_file = self.checkpoint_dir / 'triplets_progress.json'
        self.triplets_cache = self.checkpoint_dir / 'triplets_cache.jsonl'
        self.processed_chunks = self._load_progress()
    
    # ==================== CHECKPOINTING (from Baseline) ====================
    
    def _load_progress(self) -> set:
        """Load set of already-processed chunk indices"""
        if self.progress_file.exists():
            with open(self.progress_file, 'r') as f:
                data = json.load(f)
                logging.info(f"✓ Resuming: {len(data['processed'])} chunks already processed")
                return set(data['processed'])
        return set()
    
    def _save_progress(self, chunk_idx: int, entities: List[tuple], relationships: List[tuple]):
        """Save progress after processing each chunk"""
        self.processed_chunks.add(chunk_idx)
        
        # Append to cache file
        with open(self.triplets_cache, 'a', encoding='utf-8') as f:
            cache_entry = {
                'chunk_idx': chunk_idx,
                'entities': entities,
                'relationships': relationships
            }
            f.write(json.dumps(cache_entry, ensure_ascii=False) + '\n')
        
        # Update progress file
        with open(self.progress_file, 'w') as f:
            json.dump({
                'processed': sorted(list(self.processed_chunks)),
                'total_processed': len(self.processed_chunks)
            }, f, indent=2)
    
    def is_chunk_processed(self, chunk_idx: int) -> bool:
        """Check if chunk already processed"""
        return chunk_idx in self.processed_chunks
    
    def load_cached_triplets(self) -> Dict[int, Dict]:
        """Load all cached triplets"""
        if not self.triplets_cache.exists():
            return {}
        
        cached = {}
        with open(self.triplets_cache, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    entry = json.loads(line)
                    cached[entry['chunk_idx']] = {
                        'entities': entry['entities'],
                        'relationships': entry['relationships']
                    }
        return cached
    
    def clear_cache(self):
        """Clear all cached triplets and progress"""
        if self.triplets_cache.exists():
            self.triplets_cache.unlink()
        if self.progress_file.exists():
            self.progress_file.unlink()
        self.processed_chunks = set()
        logging.info("✓ Cleared triplet extraction cache")
    
    def print_stats(self):
        """Print statistics"""
        total = self.extraction_stats["total"]
        if total > 0:
            success_pct = (self.extraction_stats["success"] / total) * 100
            print(f"\n{'='*60}")
            print(f"EXTRACTION STATS: {self.extraction_stats['success']}/{total} ({success_pct:.1f}%)")
            print(f"Cached chunks: {len(self.processed_chunks)}")
            print(f"{'='*60}\n")

    # ==================== KG EXTRACTION (for building graph) ====================
    
    def extract_triplets(self, text: str, max_entities: int = 5, chunk_idx: int = None) -> Tuple[List[tuple], List[tuple]]:
        """Extract entities+relationships from PDF chunk text (KG building)."""
        self.extraction_stats["total"] += 1
        
        try:
            prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id>
You are an expert in extracting entity-relation triplets from text related to diabetes.
Extract up to {max_entities} triplets and return ONLY valid JSON in this format:

{{
  "entities": [
    {{
      "entity_name": "string",
      "entity_type": "string",
      "entity_description": "string",
      "entity_attributes": {{}}
    }}
  ],
  "relationships": [
    {{
      "source_entity": "string",
      "target_entity": "string",
      "relation": "string",
      "relationship_description": "string",
      "relationship_attributes": {{}}
    }}
  ]
}}

Input Text:
{text[:4000]}
<|end_header_id><|start_header_id|>user<|end_header_id>
Return ONLY the JSON.
<|end_header_id>"""

            resp = self.llm_pipeline(
                prompt,
                max_new_tokens=512,
                do_sample=True,
                temperature=0.1,
                top_p=0.95,
                repetition_penalty=1.2,
                return_full_text=False
            )

            resp_text = resp[0].get("generated_text", "") if isinstance(resp, list) else str(resp)
            entities, relationships = self.parse_response(resp_text)
            
            if len(entities) > 0:
                self.extraction_stats["success"] += 1
            else:
                self.extraction_stats["failed"] += 1
                
            # Save progress
            if chunk_idx is not None:
                self._save_progress(chunk_idx, entities, relationships)
                
            return entities, relationships

        except Exception as e:
            self.extraction_stats["failed"] += 1
            logging.error(f"Error extracting triplets: {e}")
            return [], []

    # ==================== CLAIM EXTRACTION (NS-HAGRAG - EXACT COPY) ====================
    
    def extract_claim_triplets(self, text: str, max_claims: int = 8) -> Tuple[List[tuple], List[tuple]]:
        """
        Stable ANSWER->CLAIMS extraction (relationships-only JSON).
        THIS IS THE EXACT NS-HAGRAG VERSION.
        
        Returns:
          entities: [] (unused)
          relationships: List[(src, tgt, rel, rel_desc, rel_attrs)]
        """
        text = (text or "")[:3500]

        prompt = (
            "You are an expert biomedical information extractor.\n"
            f"Extract up to {max_claims} factual relationship triples.\n"
            "Return ONLY valid JSON between markers. No prose. No markdown.\n\n"
            "BEGIN_JSON\n"
            "{\n"
            '  "relationships": [\n'
            '    {"source_entity":"...", "target_entity":"...", "relation":"CAUSES|TREATS|ASSOCIATED_WITH|INCREASES|DECREASES|IS_A",'
            ' "relationship_description":"", "relationship_attributes":{}}\n'
            "  ]\n"
            "}\n"
            "END_JSON\n\n"
            "Text:\n"
            f"{text}\n"
        )

        try:
            resp = self.llm_pipeline(
                prompt,
                max_new_tokens=800,
                do_sample=False,  # CRITICAL: deterministic
                top_p=1.0,
                repetition_penalty=1.0,
                return_full_text=False
            )

            resp_text = resp[0].get("generated_text", "") if isinstance(resp, list) else str(resp)
            entities, rels = self.parse_response(resp_text)

            logging.info(f"Claim extraction parsed relationships: {len(rels)}")

            # Ensure 5-field tuples
            fixed = []
            for r in rels:
                if len(r) == 5:
                    fixed.append(r)
                elif len(r) == 3:
                    fixed.append((r[0], r[1], r[2], "", {}))

            # One retry if empty
            if not fixed:
                logging.warning("Retrying claim extraction (strict reminder)...")
                resp = self.llm_pipeline(
                    prompt + "\nREMINDER: ONLY JSON between BEGIN_JSON and END_JSON.",
                    max_new_tokens=800,
                    do_sample=False,
                    top_p=1.0,
                    repetition_penalty=1.0,
                    return_full_text=False
                )
                resp_text = resp[0].get("generated_text", "") if isinstance(resp, list) else str(resp)
                entities, rels = self.parse_response(resp_text)

                fixed = []
                for r in rels:
                    if len(r) == 5:
                        fixed.append(r)
                    elif len(r) == 3:
                        fixed.append((r[0], r[1], r[2], "", {}))

                logging.info(f"Claim extraction retry relationships: {len(fixed)}")

            return [], fixed

        except Exception as e:
            logging.error(f"Error extracting claim triplets: {e}")
            return [], []

    # ==================== PARSING (NS-HAGRAG version with BEGIN_JSON/END_JSON) ====================
    
    def parse_response(self, response_str: str) -> Tuple[List[tuple], List[tuple]]:
        """
        Robust parsing - handles:
          1. BEGIN_JSON...END_JSON markers
          2. Standard JSON objects
          3. JSON arrays
        """
        entities, relationships = [], []
        if not response_str:
            return entities, relationships

        response_clean = re.sub(r"<\|.*?\|>", "", response_str, flags=re.DOTALL)
        response_clean = re.sub(r"```(?:json)?", "", response_clean, flags=re.I).replace("```", "")
        response_clean = response_clean.strip()

        # 1) Try marker JSON (NS-HAGRAG format)
        m = re.search(r"BEGIN_JSON(.*?)END_JSON", response_clean, flags=re.DOTALL)
        if m:
            json_str = m.group(1).strip()
        else:
            # 2) Try any {...}
            start = response_clean.find("{")
            end = response_clean.rfind("}")
            json_str = response_clean[start:end+1].strip() if (start != -1 and end != -1 and end > start) else ""

        # Try JSON load if we have candidate
        if json_str:
            try:
                data = json.loads(json_str)

                # relationships-only JSON supported (NS-HAGRAG claim extraction)
                if isinstance(data, dict) and "relationships" in data and isinstance(data["relationships"], list):
                    for rel in data["relationships"]:
                        if all(k in rel for k in ["source_entity", "target_entity", "relation"]):
                            relationships.append((
                                rel["source_entity"],
                                rel["target_entity"],
                                str(rel["relation"]).strip().upper().replace(" ", "_"),
                                rel.get("relationship_description", ""),
                                rel.get("relationship_attributes", {})
                            ))
                    return [], relationships

                # Full schema (entities + relationships) for KG building
                if isinstance(data, dict):
                    entity_list = data.get("entities", [])
                    rel_list = data.get("relationships", [])
                elif isinstance(data, list):
                    entity_list = data
                    rel_list = []
                else:
                    return entities, relationships

                # Extract entities
                for entity in entity_list:
                    if not isinstance(entity, dict):
                        continue
                        
                    name = (entity.get("entity_name") or entity.get("name") or 
                           entity.get("entity") or "").strip()
                    etype = (entity.get("entity_type") or entity.get("type") or 
                            "Entity").strip()
                    desc = (entity.get("entity_description") or entity.get("description") or 
                           "").strip()
                    attrs = entity.get("entity_attributes") or entity.get("attributes") or {}
                    
                    if name and len(name) > 2:
                        entities.append((name, etype, desc, attrs))

                # Extract relationships
                for relation in rel_list:
                    if not isinstance(relation, dict):
                        continue
                        
                    source = (relation.get("source_entity") or relation.get("source") or 
                             relation.get("from") or "").strip()
                    target = (relation.get("target_entity") or relation.get("target") or 
                             relation.get("to") or "").strip()
                    rel_type = (relation.get("relation") or relation.get("relationship") or 
                               relation.get("type") or "RELATED_TO").strip()
                    rel_desc = (relation.get("relationship_description") or 
                               relation.get("description") or "").strip()
                    rel_attrs = relation.get("relationship_attributes") or relation.get("attributes") or {}
                    
                    if source and target:
                        relationships.append((source, target, rel_type, rel_desc, rel_attrs))

                return entities, relationships

            except json.JSONDecodeError:
                pass

        # Fallback: try to find array
        array_match = re.search(r'\[(.*?)\]', response_clean, re.DOTALL)
        if array_match:
            try:
                entity_list = json.loads(f'[{array_match.group(1)}]')
                for entity in entity_list:
                    if isinstance(entity, dict):
                        name = (entity.get("entity_name") or entity.get("name") or "").strip()
                        etype = (entity.get("entity_type") or entity.get("type") or "Entity").strip()
                        desc = (entity.get("entity_description") or entity.get("description") or "").strip()
                        attrs = entity.get("entity_attributes") or entity.get("attributes") or {}
                        if name and len(name) > 2:
                            entities.append((name, etype, desc, attrs))
            except:
                pass

        return entities, relationships


class ArchRAGCHNSW:
    """C-HNSW Index Implementation for ArchRAG"""

    def __init__(self, embedding_dim: int = 384, M: int = 16, ef_construction: int = 200):
        self.embedding_dim = embedding_dim
        self.M = M
        self.ef_construction = ef_construction
        self.layers = []
        self.node_embeddings = {}
        self.inter_layer_links = {}
        self.node_data = {}
        self.nn_index = None

    def add_node(self, node_id: str, embedding: np.ndarray, layer: int, node_data: dict = None):
        """Add a node to the specified layer"""
        while len(self.layers) <= layer:
            self.layers.append(nx.Graph())

        self.layers[layer].add_node(node_id)
        self.node_embeddings[node_id] = embedding
        if node_data:
            self.node_data[node_id] = node_data

    def distance(self, a: np.ndarray, b: np.ndarray) -> float:
        """Compute cosine distance between two vectors"""
        return 1 - cosine_similarity([a], [b])[0][0]

    def build_intra_layer_links(self):
        """Build links within each layer"""
        for layer_idx, layer_graph in enumerate(self.layers):
            nodes = list(layer_graph.nodes())
            if len(nodes) <= 1:
                continue

            embeddings = np.array([self.node_embeddings[n] for n in nodes])
            nn = NearestNeighbors(n_neighbors=min(self.M, len(nodes)), metric="cosine")
            nn.fit(embeddings)

            distances, indices = nn.kneighbors(embeddings)
            for i, node_id in enumerate(nodes):
                for j, dist in zip(indices[i], distances[i]):
                    if j != i:
                        neighbor_id = nodes[j]
                        layer_graph.add_edge(node_id, neighbor_id, weight=dist, similarity=1-dist)

    def build_inter_layer_links(self):
        """Build links between adjacent layers"""
        for layer_idx in range(1, len(self.layers)):
            higher_layer_nodes = list(self.layers[layer_idx].nodes())
            lower_layer_nodes = list(self.layers[layer_idx-1].nodes())

            if not higher_layer_nodes or not lower_layer_nodes:
                continue

            lower_embeddings = np.array([self.node_embeddings[n] for n in lower_layer_nodes])
            nn = NearestNeighbors(n_neighbors=1, metric="cosine")
            nn.fit(lower_embeddings)

            for node_id in higher_layer_nodes:
                if node_id not in self.node_embeddings:
                    continue

                node_emb = self.node_embeddings[node_id]
                distances, indices = nn.kneighbors([node_emb])
                nearest = lower_layer_nodes[indices[0][0]]
                self.inter_layer_links[node_id] = nearest

    def search_layer(self, layer_idx: int, query_embedding: np.ndarray,
                    entry_point: str, k: int = 1) -> List[str]:
        """Search for k nearest neighbors in a specific layer"""
        if layer_idx >= len(self.layers) or not self.layers[layer_idx].nodes():
            return []

        if entry_point not in self.node_embeddings:
            return []

        layer = self.layers[layer_idx]
        visited = set([entry_point])
        candidates = [(self.distance(query_embedding, self.node_embeddings[entry_point]), entry_point)]
        results = [(self.distance(query_embedding, self.node_embeddings[entry_point]), entry_point)]

        while candidates:
            current_dist, current = heapq.heappop(candidates)

            if results and current_dist > max(results, key=lambda x: x[0])[0]:
                break

            for neighbor in layer.neighbors(current):
                if neighbor not in visited and neighbor in self.node_embeddings:
                    visited.add(neighbor)
                    dist = self.distance(query_embedding, self.node_embeddings[neighbor])
                    edge_data = layer.get_edge_data(current, neighbor)
                    weight = edge_data.get("weight", 1.0)

                    adjusted_dist = dist * weight
                    if len(results) < k or adjusted_dist < max(results, key=lambda x: x[0])[0]:
                        heapq.heappush(candidates, (adjusted_dist, neighbor))
                        heapq.heappush(results, (adjusted_dist, neighbor))

                        if len(results) > k:
                            results.remove(max(results, key=lambda x: x[0]))

        return [node_id for _, node_id in sorted(results, key=lambda x: x[0])[:k]]

    def hierarchical_search(self, query_embedding: np.ndarray, k: int = 3) -> Dict[int, List[str]]:
        """Perform hierarchical search across all layers"""
        if not self.layers or all(not layer.nodes() for layer in self.layers):
            return {}

        results = {}
        top_layer = len(self.layers) - 1
        while top_layer >= 0 and not self.layers[top_layer].nodes():
            top_layer -= 1

        if top_layer < 0:
            return {}

        entry_point = list(self.layers[top_layer].nodes())[0]

        for layer_idx in range(top_layer, -1, -1):
            if not self.layers[layer_idx].nodes():
                continue

            layer_results = self.search_layer(layer_idx, query_embedding, entry_point, k)
            results[layer_idx] = layer_results

            if layer_results and layer_idx > 0:
                best_node = layer_results[0]
                entry_point = self.inter_layer_links.get(best_node,
                    list(self.layers[layer_idx - 1].nodes())[0] if self.layers[layer_idx - 1].nodes() else entry_point)

        return results

print("✓ Cell 2 loaded: TripletExtractor with checkpointing + NS-HAGRAG extract_claim_triplets + ArchRAGCHNSW")

In [ ]:
# ============================================================
# CELL 3: ArchRAGStore with Backup/Restore Capabilities
# ============================================================
# Replace your current Cell 3 with this version

from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import networkx as nx
from graspologic.partition import hierarchical_leiden
from collections import defaultdict
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import json
import logging
import pickle
from pathlib import Path
from typing import List, Set, Dict, Any
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

class ArchRAGStore:
    """Enhanced ArchRAG Store with local backup and Neo4j sync"""

    def __init__(self, uri: str, username: str, password: str, 
                 checkpoint_dir: str = './checkpoints/hagragpipeline'):
        self.driver = GraphDatabase.driver(uri, auth=(username, password))
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        
        # Storage
        self.entity_embeddings = {}
        self.hierarchical_communities = {}
        self.community_summaries = {}
        self.chnsw_index = ArchRAGCHNSW(embedding_dim=384)
        
        # Configuration
        self.max_layers = 4
        self.min_nodes_per_layer = 2
        self.connection_threshold = 0.1
        
        # Checkpoint files
        self.embeddings_file = self.checkpoint_dir / 'entity_embeddings.pkl'
        self.communities_file = self.checkpoint_dir / 'hierarchical_communities.pkl'
        self.chnsw_file = self.checkpoint_dir / 'chnsw_index.pkl'
        self.neo4j_backup_file = self.checkpoint_dir / 'neo4j_backup.json'
        
    def close(self):
        """Close the database connection"""
        self.driver.close()
    
    # ==================== NEO4J BACKUP/RESTORE ====================
    
    def backup_neo4j_to_local(self):
        """Backup entire Neo4j graph to local file."""
        logging.info("Backing up Neo4j graph to local storage...")
        
        backup_data = {
            'entities': [],
            'relationships': []
        }
        
        with self.driver.session() as session:
            # Backup entities
            result = session.run(
                "MATCH (e:Entity) RETURN e.name, e.type, e.description, e.doc_id, e.attributes"
            )
            for record in result:
                backup_data['entities'].append({
                    'name': record['e.name'],
                    'type': record['e.type'],
                    'description': record['e.description'],
                    'doc_id': record['e.doc_id'],
                    'attributes': record['e.attributes']
                })
            
            # Backup relationships
            result = session.run(
                """
                MATCH (source:Entity)-[r:RELATION]->(target:Entity)
                RETURN source.name, target.name, r.type, r.description, r.doc_id, r.attributes
                """
            )
            for record in result:
                backup_data['relationships'].append({
                    'source': record['source.name'],
                    'target': record['target.name'],
                    'type': record['r.type'],
                    'description': record['r.description'],
                    'doc_id': record['r.doc_id'],
                    'attributes': record['r.attributes']
                })
        
        with open(self.neo4j_backup_file, 'w', encoding='utf-8') as f:
            json.dump(backup_data, f, ensure_ascii=False, indent=2)
        
        logging.info(f"✓ Backed up {len(backup_data['entities'])} entities, "
                    f"{len(backup_data['relationships'])} relationships")
    
    def restore_neo4j_from_local(self):
        """Restore Neo4j graph from local backup."""
        if not self.neo4j_backup_file.exists():
            logging.error(f"No backup file found at {self.neo4j_backup_file}")
            return False
        
        logging.info("Restoring Neo4j graph from local backup...")
        
        with open(self.neo4j_backup_file, 'r', encoding='utf-8') as f:
            backup_data = json.load(f)
        
        with self.driver.session() as session:
            # Clear existing data
            session.run("MATCH (n) DETACH DELETE n")
            
            # Restore entities
            for entity in backup_data['entities']:
                session.run(
                    """
                    CREATE (e:Entity {
                        name: $name,
                        type: $type,
                        description: $desc,
                        doc_id: $doc_id,
                        attributes: $attrs
                    })
                    """,
                    name=entity['name'],
                    type=entity['type'],
                    desc=entity['description'],
                    doc_id=entity['doc_id'],
                    attrs=entity['attributes']
                )
            
            # Restore relationships
            for rel in backup_data['relationships']:
                session.run(
                    """
                    MATCH (source:Entity {name: $source})
                    MATCH (target:Entity {name: $target})
                    CREATE (source)-[r:RELATION {
                        type: $type,
                        description: $desc,
                        doc_id: $doc_id,
                        attributes: $attrs
                    }]->(target)
                    """,
                    source=rel['source'],
                    target=rel['target'],
                    type=rel['type'],
                    desc=rel['description'],
                    doc_id=rel['doc_id'],
                    attrs=rel['attributes']
                )
        
        logging.info(f"✓ Restored {len(backup_data['entities'])} entities, "
                    f"{len(backup_data['relationships'])} relationships")
        return True
    
    def entity_exists_in_neo4j(self, entity_name: str) -> bool:
        """Check if entity already exists in Neo4j"""
        with self.driver.session() as session:
            result = session.run(
                "MATCH (e:Entity {name: $name}) RETURN count(e) as count",
                name=entity_name
            )
            count = result.single()['count']
            return count > 0
    
    def get_neo4j_entity_count(self) -> int:
        """Get total entity count in Neo4j"""
        with self.driver.session() as session:
            result = session.run("MATCH (e:Entity) RETURN count(e) as count")
            return result.single()['count']
    
    def clear_neo4j_graph(self):
        """Clear entire Neo4j graph"""
        logging.info("Clearing Neo4j graph...")
        with self.driver.session() as session:
            session.run("MATCH (n) DETACH DELETE n")
        logging.info("✓ Neo4j graph cleared")

    # ==================== TRIPLET ADDITION ====================
    
    def add_triplets(self, entities: List[tuple], relationships: List[tuple], 
                    doc_id: str, skip_existing: bool = False):
        """Add triplets to Neo4j with optional skip for existing entities"""
        
        with self.driver.session() as session:
            for entity_name, entity_type, entity_desc, entity_attrs in entities:
                if skip_existing and self.entity_exists_in_neo4j(entity_name):
                    continue
                
                entity_text = f"{entity_name} {entity_type} {entity_desc} {json.dumps(entity_attrs)}"
                embedding = self.embedding_model.encode(entity_text)
                self.entity_embeddings[entity_name] = embedding

                session.run(
                    """
                    MERGE (e:Entity {name: $name})
                    SET e.type = $type, e.description = $desc, e.doc_id = $doc_id, e.attributes = $attrs
                    """,
                    name=entity_name, type=entity_type, desc=entity_desc,
                    doc_id=doc_id, attrs=json.dumps(entity_attrs)
                )

            for source, target, rel, rel_desc, rel_attrs in relationships:
                session.run(
                    """
                    MATCH (source:Entity {name: $source})
                    MATCH (target:Entity {name: $target})
                    MERGE (source)-[r:RELATION {type: $rel}]->(target)
                    SET r.description = $desc, r.doc_id = $doc_id, r.attributes = $attrs
                    """,
                    source=source, target=target, rel=rel, desc=rel_desc,
                    doc_id=doc_id, attrs=json.dumps(rel_attrs)
                )
    
    # ==================== LOCAL CHECKPOINT SAVE/LOAD ====================
    
    def save_local_checkpoints(self):
        """Save embeddings, communities, and CHNSW index to local files"""
        logging.info("Saving local checkpoints...")
        
        with open(self.embeddings_file, 'wb') as f:
            pickle.dump(self.entity_embeddings, f)
        
        with open(self.communities_file, 'wb') as f:
            pickle.dump(self.hierarchical_communities, f)
        
        with open(self.chnsw_file, 'wb') as f:
            pickle.dump(self.chnsw_index, f)
        
        logging.info(f"✓ Saved embeddings ({len(self.entity_embeddings)} entities), "
                    f"communities ({len(self.hierarchical_communities)} layers), "
                    f"CHNSW index")
    
    def load_local_checkpoints(self) -> bool:
        """Load embeddings, communities, and CHNSW index from local files"""
        if not all([self.embeddings_file.exists(), 
                   self.communities_file.exists(),
                   self.chnsw_file.exists()]):
            logging.warning("Some checkpoint files missing, cannot load")
            return False
        
        logging.info("Loading local checkpoints...")
        
        with open(self.embeddings_file, 'rb') as f:
            self.entity_embeddings = pickle.load(f)
        
        with open(self.communities_file, 'rb') as f:
            self.hierarchical_communities = pickle.load(f)
        
        with open(self.chnsw_file, 'rb') as f:
            self.chnsw_index = pickle.load(f)
        
        logging.info(f"✓ Loaded embeddings ({len(self.entity_embeddings)} entities), "
                    f"communities ({len(self.hierarchical_communities)} layers), "
                    f"CHNSW index")
        return True
    
    # ==================== COMMUNITY BUILDING ====================
    
    def _calculate_community_connection(self, comm1: dict, comm2: dict, graph: nx.Graph) -> float:
        """Calculate connection strength between two communities"""
        members1 = set(comm1["members"])
        members2 = set(comm2["members"])

        direct_connections = 0
        total_possible = len(members1) * len(members2)

        if total_possible == 0:
            return 0.0

        for m1 in members1:
            for m2 in members2:
                if graph.has_edge(m1, m2):
                    direct_connections += 1

        shared_members = len(members1.intersection(members2))
        connection_strength = (direct_connections / total_possible) + (shared_members / max(len(members1), len(members2)))

        return min(connection_strength, 1.0)

    def compute_similarity_threshold(self, similarities: List[float], percentile: float = 0.7) -> float:
        """Compute dynamic similarity threshold"""
        if not similarities:
            return 0.5
        return np.percentile(similarities, percentile * 100)

    def augment_graph_with_attributes(self, nx_graph: nx.Graph, k: int = 5) -> nx.Graph:
        """Augment graph by connecting entities with similar attributes"""
        augmented_graph = nx_graph.copy()
        nodes = list(nx_graph.nodes())

        if len(nodes) <= 1:
            return augmented_graph

        all_similarities = []
        node_similarities = {}

        for i, node1 in enumerate(nodes):
            if node1 not in self.entity_embeddings:
                continue

            similarities = []
            for j, node2 in enumerate(nodes):
                if i != j and node2 in self.entity_embeddings:
                    sim = cosine_similarity(
                        [self.entity_embeddings[node1]],
                        [self.entity_embeddings[node2]]
                    )[0][0]
                    similarities.append((node2, sim))
                    all_similarities.append(sim)

            node_similarities[node1] = similarities

        threshold = self.compute_similarity_threshold(all_similarities)

        for node1, similarities in node_similarities.items():
            top_k = sorted(similarities, key=lambda x: x[1], reverse=True)[:k]
            for node2, sim in top_k:
                if sim > threshold:
                    weight = 1 - sim
                    augmented_graph.add_edge(node1, node2, weight=weight, similarity=sim)

        return augmented_graph

    def cluster_graph(self, graph: nx.Graph, method: str = "leiden") -> List[Set[str]]:
        """Cluster graph using specified method"""
        if len(graph.nodes()) < 2:
            return [set(graph.nodes())]

        try:
            if method == "leiden":
                clusters = hierarchical_leiden(graph)
                community_map = defaultdict(set)
                for item in clusters:
                    community_map[item.cluster].add(item.node)
                return list(community_map.values())
            else:
                communities = nx.algorithms.community.greedy_modularity_communities(graph)
                return [set(c) for c in communities]
        except Exception as e:
            logging.error(f"Clustering failed: {e}, falling back to single community")
            return [set(graph.nodes())]

    def generate_attributed_community_summary(self, community: Set[str], llm) -> str:
        """Generate summary for attributed community using LLM"""
        community_ids = {entity for entity in community if entity.startswith('L')}
        actual_entities = {entity for entity in community if not entity.startswith('L')}

        if community_ids and not actual_entities:
            return self._generate_meta_community_summary(community_ids, llm)
        else:
            return self._generate_entity_community_summary(actual_entities, llm)

    def _generate_meta_community_summary(self, community_ids: Set[str], llm) -> str:
        """Generate summary for meta-community containing other communities"""
        sub_summaries = []
        total_members = 0

        for comm_id in community_ids:
            for layer_data in self.hierarchical_communities.values():
                for comm_data in layer_data:
                    if comm_data['id'] == comm_id:
                        sub_summaries.append(comm_data['summary'])
                        total_members += comm_data['size']
                        break

        if not sub_summaries:
            return f"Meta-community grouping {len(community_ids)} sub-communities"

        combined_text = "\n\n".join(sub_summaries)

        meta_prompt = ChatPromptTemplate.from_messages([
            ("system",
            "You are analyzing a higher-level community that groups several sub-communities. "
            "Create a concise summary that identifies the overarching themes and patterns "
            "across these sub-communities. Focus on what connects them at a higher level."),
            ("human", "Sub-community summaries:\n{summaries}")
        ])

        meta_chain = LLMChain(llm=llm, prompt=meta_prompt)
        try:
            result = meta_chain.invoke({"summaries": combined_text})
            return f"Meta-community ({total_members} total entities): {result['text'].strip()}"
        except Exception as e:
            logging.error(f"Error generating meta-community summary: {e}")
            return f"Meta-community grouping {len(community_ids)} sub-communities with {total_members} total entities"

    def _generate_entity_community_summary(self, actual_entities: Set[str], llm) -> str:
        """Generate summary for regular entity community"""
        entity_details = []

        with self.driver.session() as session:
            for entity_name in actual_entities:
                result = session.run(
                    "MATCH (e:Entity {name: $name}) RETURN e.name, e.type, e.description, e.attributes",
                    name=entity_name
                )
                record = result.single()
                if record:
                    entity_details.append({
                        "name": record["e.name"],
                        "type": record["e.type"] or "Unknown",
                        "description": record["e.description"] or "No description",
                        "attributes": json.loads(record["e.attributes"] or "{}")
                    })

        if not entity_details:
            return f"Community with {len(actual_entities)} entities"

        summary_text = f"Entities: {json.dumps(entity_details, indent=2)}"

        summary_prompt = ChatPromptTemplate.from_messages([
            ("system", "Create a concise summary of this entity community."),
            ("human", "Community data:\n{community_data}")
        ])

        summary_chain = LLMChain(llm=llm, prompt=summary_prompt)
        try:
            result = summary_chain.invoke({"community_data": summary_text})
            return result["text"].strip()
        except Exception as e:
            logging.error(f"Error generating entity summary: {e}")
            return f"Community with {len(actual_entities)} entities"

    def build_hierarchical_attributed_communities(self, llm):
        """Build hierarchical attributed communities"""
        current_graph = self._create_nx_graph()
        layer = 0
        all_communities = {}

        logging.info(f"Starting hierarchical clustering with {len(current_graph.nodes())} nodes")

        while layer < self.max_layers and len(current_graph.nodes()) >= self.min_nodes_per_layer:
            logging.info(f"Processing layer {layer} with {len(current_graph.nodes())} nodes")

            augmented_graph = self.augment_graph_with_attributes(current_graph)
            communities = self.cluster_graph(augmented_graph)
            logging.info(f"Found {len(communities)} communities in layer {layer}")

            layer_communities = []
            for i, community in enumerate(communities):
                if len(community) == 0:
                    continue

                community_id = f"L{layer}_C{i}"
                summary = self.generate_attributed_community_summary(community, llm)

                community_data = {
                    "id": community_id,
                    "layer": layer,
                    "members": list(community),
                    "summary": summary,
                    "size": len(community),
                    "is_meta_community": any(member.startswith('L') for member in community)
                }
                layer_communities.append(community_data)

                try:
                    self.entity_embeddings[community_id] = self.embedding_model.encode(summary)
                except Exception as e:
                    logging.error(f"Error generating embedding for community {community_id}: {e}")
                    self.entity_embeddings[community_id] = np.zeros(384)

            all_communities[layer] = layer_communities
            self.hierarchical_communities[layer] = layer_communities

            # Build next layer graph
            new_graph = nx.Graph()
            for comm_data in layer_communities:
                new_graph.add_node(comm_data["id"])

            for i, comm1 in enumerate(layer_communities):
                for j, comm2 in enumerate(layer_communities):
                    if i < j:
                        connection_strength = self._calculate_community_connection(
                            comm1, comm2, augmented_graph
                        )
                        if connection_strength > self.connection_threshold:
                            new_graph.add_edge(comm1["id"], comm2["id"], weight=1-connection_strength)

            current_graph = new_graph
            layer += 1

        logging.info(f"Built {layer} layers of hierarchical communities")
        return all_communities

    def build_chnsw_index(self):
        """Build C-HNSW index from hierarchical communities"""
        logging.info("Building C-HNSW index...")

        for layer, communities in self.hierarchical_communities.items():
            for community in communities:
                community_id = community["id"]
                if community_id in self.entity_embeddings:
                    embedding = self.entity_embeddings[community_id]
                    self.chnsw_index.add_node(
                        community_id,
                        embedding,
                        layer,
                        node_data=community
                    )

        for entity_name, embedding in self.entity_embeddings.items():
            if not entity_name.startswith('L'):
                self.chnsw_index.add_node(
                    entity_name,
                    embedding,
                    0,
                    node_data={"type": "entity", "name": entity_name}
                )

        self.chnsw_index.build_intra_layer_links()
        self.chnsw_index.build_inter_layer_links()
        logging.info("C-HNSW index construction completed")

    def _create_nx_graph(self) -> nx.Graph:
        """Create NetworkX graph from Neo4j database"""
        nx_graph = nx.Graph()
        with self.driver.session() as session:
            result = session.run(
                "MATCH (e1)-[r:RELATION]->(e2) RETURN e1.name, r.type, e2.name, r.description, r.attributes"
            )
            for record in result:
                e1, rel, e2, desc, attrs = (
                    record["e1.name"], record["r.type"], record["e2.name"],
                    record["r.description"], record["r.attributes"]
                )
                nx_graph.add_node(e1)
                nx_graph.add_node(e2)
                nx_graph.add_edge(e1, e2, relationship=rel, description=desc, attributes=attrs)
        return nx_graph

    def build_communities(self, llm):
        """Main method to build the complete ArchRAG structure"""
        logging.info("Building ArchRAG hierarchical attributed communities...")
        self.build_hierarchical_attributed_communities(llm)
        self.build_chnsw_index()
        
        # Save everything locally
        self.save_local_checkpoints()
        self.backup_neo4j_to_local()
        
        logging.info("ArchRAG community structure completed and backed up!")

print("✓ Cell 3 loaded: ArchRAGStore with backup/restore capabilities")

In [ ]:
# ============================================================
# CELL 4: ArchRAGQueryEngine + ArchRAGPipeline with Checkpoint Support
# ============================================================
# Replace your current Cell 4 with this version

import json
import numpy as np
import logging
import re
from typing import Dict, Any, List, Optional, Tuple
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline
from tqdm import tqdm
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain
from pathlib import Path

# Fix langchain verbose issue
import langchain
if not hasattr(langchain, "verbose"):
    langchain.verbose = False


class ArchRAGQueryEngine:
    """Enhanced Query Engine for ArchRAG"""

    def __init__(self, graph_store, llm, relevance_threshold: float = 0.3):
        self.graph_store = graph_store
        self.llm = llm
        self.embedding_model = graph_store.embedding_model
        self.relevance_threshold = relevance_threshold

    def retrieve_hierarchical_info(self, query: str, k: int = 3) -> Dict[int, List[Dict]]:
        """Retrieve relevant information across all hierarchical layers"""
        query_embedding = self.embedding_model.encode(query)
        search_results = self.graph_store.chnsw_index.hierarchical_search(query_embedding, k)

        hierarchical_info = {}
        for layer, node_ids in search_results.items():
            layer_info = []
            for node_id in node_ids:
                if node_id in self.graph_store.chnsw_index.node_data:
                    node_data = self.graph_store.chnsw_index.node_data[node_id]
                    if node_data.get("type") == "entity":
                        entity_details = self._get_entity_details(node_id)
                        layer_info.append({
                            "id": node_id,
                            "type": "entity",
                            "content": entity_details,
                            "layer": layer
                        })
                    else:
                        layer_info.append({
                            "id": node_id,
                            "type": "community",
                            "content": node_data,
                            "layer": layer
                        })
            hierarchical_info[layer] = layer_info
        return hierarchical_info

    def _get_entity_details(self, entity_name: str) -> Dict:
        """Get detailed information about an entity from Neo4j"""
        with self.graph_store.driver.session() as session:
            result = session.run(
                """
                MATCH (e:Entity {name: $name})
                OPTIONAL MATCH (e)-[r:RELATION]-(connected)
                RETURN e.name, e.type, e.description, e.attributes,
                       collect({relation: r.type, connected_entity: connected.name,
                               relation_desc: r.description}) as relationships
                """,
                name=entity_name
            )
            record = result.single()
            if record:
                return {
                    "name": record["e.name"],
                    "type": record["e.type"] or "Unknown",
                    "description": record["e.description"] or "No description",
                    "attributes": json.loads(record["e.attributes"] or "{}"),
                    "relationships": record["relationships"]
                }
            return {}

    def adaptive_filter_information(self, query: str, hierarchical_info: Dict[int, List[Dict]]) -> List[Dict]:
        """Apply adaptive filtering to retrieved information with layer weighting"""
        analysis_reports = []
        max_layer = max(hierarchical_info.keys(), default=0)

        for layer, layer_info in sorted(hierarchical_info.items(), reverse=True):
            if not layer_info:
                continue

            layer_text = self._format_layer_info(layer_info)
            layer_weight = 1.0 + (layer / max_layer) * 0.5 if max_layer > 0 else 1.0

            filter_prompt = ChatPromptTemplate.from_messages([
                ("system",
                "Analyze the relevance of the provided information for the query. "
                "Respond with ONLY a valid JSON object in this exact format:\n"
                '{"relevance_score": 7.5, "analysis": "brief analysis text", "relevant_content": "most relevant parts"}\n'
                "relevance_score must be a number from 0-10. Do not include any text before or after the JSON."),
                ("human", "Query: {query}\n\nRetrieved Information:\n{info}")
            ])

            filter_chain = LLMChain(llm=self.llm, prompt=filter_prompt)

            response_text = ""
            try:
                response = filter_chain.invoke({"query": query, "info": layer_text})
                response_text = response["text"].strip()
                logging.debug(f"LLM response for layer {layer}: {response_text}")

                json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
                matches = re.findall(json_pattern, response_text, re.DOTALL)

                report = None
                for match in matches:
                    try:
                        potential_report = json.loads(match)
                        if isinstance(potential_report, dict) and "relevance_score" in potential_report:
                            report = potential_report
                            break
                    except json.JSONDecodeError:
                        continue

                if not report:
                    relevance_match = re.search(r'"relevance_score":\s*(\d+\.?\d*)', response_text)
                    relevance_score = float(relevance_match.group(1)) if relevance_match else 5.0

                    report = {
                        "relevance_score": relevance_score,
                        "analysis": "Auto-generated analysis due to parsing issues",
                        "relevant_content": layer_text[:500] + "..." if len(layer_text) > 500 else layer_text
                    }

                report["relevance_score"] = float(report.get("relevance_score", 5.0))
                report["analysis"] = str(report.get("analysis", "No analysis available"))
                report["relevant_content"] = str(report.get("relevant_content", layer_text))

                report["relevance_score"] = min(report["relevance_score"] * layer_weight, 10)
                report["layer"] = layer
                report["original_info"] = layer_info
                analysis_reports.append(report)

            except Exception as e:
                analysis_reports.append({
                    "relevance_score": 3.0 * layer_weight,
                    "analysis": f"Processing fallback due to error: {str(e)[:100]}",
                    "relevant_content": layer_text[:500] + "..." if len(layer_text) > 500 else layer_text,
                    "layer": layer,
                    "original_info": layer_info
                })

        analysis_reports = [r for r in analysis_reports if r.get("relevance_score", 0) >= self.relevance_threshold * 10]
        analysis_reports.sort(key=lambda x: x.get("relevance_score", 0), reverse=True)
        return analysis_reports

    def _format_layer_info(self, layer_info: List[Dict]) -> str:
        """Format layer information for LLM processing"""
        formatted_parts = []
        for item in layer_info:
            if item["type"] == "entity":
                content = item["content"]
                formatted_parts.append(
                    f"Entity: {content.get('name', 'Unknown')}\n"
                    f"Type: {content.get('type', 'Unknown')}\n"
                    f"Description: {content.get('description', 'No description')}\n"
                    f"Attributes: {json.dumps(content.get('attributes', {}))}\n"
                    f"Relationships: {json.dumps(content.get('relationships', []))}\n"
                )
            elif item["type"] == "community":
                content = item["content"]
                formatted_parts.append(
                    f"Community {content.get('id', 'Unknown')} (Layer {content.get('layer', 'Unknown')}):\n"
                    f"Summary: {content.get('summary', 'No summary')}\n"
                    f"Size: {content.get('size', 0)} members\n"
                    f"Members: {', '.join(content.get('members', []))}\n"
                )
        return "\n---\n".join(formatted_parts)

    def generate_response(self, query: str, filtered_info: List[Dict]) -> str:
        """Generate final response using filtered information"""
        if not filtered_info:
            return "I couldn't find relevant information to answer your query."

        context_parts = [report.get("relevant_content", "") for report in filtered_info]
        context = "\n\n".join(context_parts)

        response_prompt = ChatPromptTemplate.from_messages([
            ("system",
             "Answer the query based on the provided context. "
             "Cite specific entities or relationships when relevant. "
             "If the context is insufficient, acknowledge the limitations and provide a general answer."),
            ("human", "Question: {query}\n\nContext:\n{context}")
        ])

        response_chain = LLMChain(llm=self.llm, prompt=response_prompt)

        try:
            result = response_chain.invoke({"query": query, "context": context})
            return result["text"].strip()
        except Exception as e:
            logging.error(f"Error generating response: {e}")
            return f"Error generating response: {e}"

    def query(self, query: str, k: int = 3) -> Dict[str, Any]:
        """Main query method for ArchRAG pipeline"""
        logging.info(f"Processing query: {query}")

        hierarchical_info = self.retrieve_hierarchical_info(query, k)
        filtered_info = self.adaptive_filter_information(query, hierarchical_info)
        response = self.generate_response(query, filtered_info)

        return {
            "query": query,
            "response": response,
            "hierarchical_info": hierarchical_info,
            "filtered_info": filtered_info,
            "layers_searched": list(hierarchical_info.keys())
        }


class ArchRAGPipeline:
    """Complete ArchRAG Pipeline with checkpoint management"""
    
    def __init__(self, pdf_dir: str, neo4j_uri: str, neo4j_username: str,
                 neo4j_password: str, checkpoint_dir: str = './checkpoints/hagragpipeline'):
        
        # Set cache dir
        os.environ["HF_HOME"] = "./workspace/hf_cache"
        os.environ["TRANSFORMERS_CACHE"] = "./workspace/hf_cache"
        
        self.checkpoint_dir = checkpoint_dir
        self.pdf_processor = PDFProcessor(pdf_dir, checkpoint_dir)
        self.graph_store = ArchRAGStore(neo4j_uri, neo4j_username, neo4j_password, checkpoint_dir)
        
        # Initialize Llama 3.1 with 4-bit quantization
        model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
        
        try:
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            )
            
            tokenizer = AutoTokenizer.from_pretrained(
                model_id, 
                token=os.getenv("HF_TOKEN"),
                padding_side="left"
            )
            
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                token=os.getenv("HF_TOKEN"),
                quantization_config=quantization_config,
                low_cpu_mem_usage=True,
                trust_remote_code=True,
            )
            
            raw_pipeline = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.2,
                return_full_text=False,
                device=model.device if hasattr(model, 'device') else 0
            )
            
            print(f"Model loaded successfully on device: {model.device}")
            
        except Exception as e:
            print(f"Error loading quantized model: {e}")
            print("Falling back to non-quantized model...")
            
            tokenizer = AutoTokenizer.from_pretrained(
                model_id, 
                token=os.getenv("HF_TOKEN"),
                padding_side="left"
            )
            
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                token=os.getenv("HF_TOKEN"),
                torch_dtype=torch.float16,
                device_map="auto",
                low_cpu_mem_usage=True,
                trust_remote_code=True
            )
            
            raw_pipeline = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.2,
                return_full_text=False
            )
        
        hf_llm = HuggingFacePipeline(pipeline=raw_pipeline)
        
        self.triplet_extractor = TripletExtractor(raw_pipeline, checkpoint_dir)
        self.query_engine = ArchRAGQueryEngine(self.graph_store, hf_llm)

    def build_knowledge_graph(self, max_docs: Optional[int] = None, 
                             rebuild_from_scratch: bool = False,
                             skip_triplet_extraction: bool = False):
        """Build the complete knowledge graph from PDFs with resume capability"""
        
        if rebuild_from_scratch:
            logging.info("=== REBUILDING FROM SCRATCH ===")
            self.triplet_extractor.clear_cache()
            self.graph_store.clear_neo4j_graph()
        
        logging.info("Starting ArchRAG knowledge graph construction...")

        # Step 1: Extract texts
        texts = self.pdf_processor.extract_texts()
        if max_docs:
            texts = texts[:max_docs]
        logging.info(f"- {len(texts)} documents ready")

        # Step 2: Create chunks
        chunks = self.pdf_processor.create_chunks(texts)
        logging.info(f"- {len(chunks)} chunks ready")

        # Step 3: Triplet extraction
        if not skip_triplet_extraction:
            neo4j_entity_count_before = self.graph_store.get_neo4j_entity_count()
            logging.info(f"Neo4j currently has {neo4j_entity_count_before} entities")
            
            cached_triplets = self.triplet_extractor.load_cached_triplets()
            
            logging.info(f"Processing {len(chunks)} chunks for triplet extraction...")
            logging.info(f"Already processed: {len(cached_triplets)} chunks")
            logging.info(f"Remaining: {len(chunks) - len(cached_triplets)} chunks")
            
            for i, chunk in enumerate(tqdm(chunks, desc="Extracting triplets")):
                doc_id = f"doc_{i // 20}_chunk_{i % 20}"
                
                if i in cached_triplets:
                    entities = cached_triplets[i]['entities']
                    relationships = cached_triplets[i]['relationships']
                    if entities or relationships:
                        self.graph_store.add_triplets(
                            entities, relationships, doc_id, skip_existing=True
                        )
                    continue
                
                entities, relationships = self.triplet_extractor.extract_triplets(
                    chunk, max_entities=5, chunk_idx=i
                )
                
                if entities or relationships:
                    self.graph_store.add_triplets(
                        entities, relationships, doc_id, skip_existing=False
                    )
            
            self.triplet_extractor.print_stats()
            
            neo4j_entity_count_after = self.graph_store.get_neo4j_entity_count()
            logging.info(f"- Neo4j now has {neo4j_entity_count_after} entities (+{neo4j_entity_count_after - neo4j_entity_count_before} new)")
        else:
            logging.info("Skipping triplet extraction (using existing Neo4j data)")

        # Step 4: Build communities
        logging.info("Building hierarchical communities...")
        self.graph_store.build_communities(self.query_engine.llm)

        logging.info("- ArchRAG knowledge graph construction completed!")
        logging.info(f"- All checkpoints saved to: {self.checkpoint_dir}")

    def query(self, query: str, k: int = 10) -> Dict[str, Any]:
        """Query the ArchRAG system"""
        return self.query_engine.query(query, k)

    def close(self):
        """Clean up resources"""
        self.graph_store.close()


print("✓ Cell 4 loaded: ArchRAGQueryEngine + ArchRAGPipeline with checkpoint management")
print("\n" + "="*60)
print("USAGE:")
print("="*60)
print("To load from checkpoints (NS-HAGRAG), use the integration cell")
print("To rebuild from scratch: pipeline.build_knowledge_graph(rebuild_from_scratch=True)")
print("="*60 + "\n")

In [ ]:
# # ============================================================================
# # CELL 0: MASTER LOADER - RUN THIS FIRST AFTER EVERY KERNEL RESTART
# # ============================================================================

# import pickle
# import networkx as nx
# import numpy as np

# class ArchRAGCHNSW:
#     def __init__(self, embedding_dim=384, max_layers=4):
#         self.embedding_dim = embedding_dim
#         self.max_layers = max_layers
#         self.layers = [nx.Graph() for _ in range(max_layers)]
#         self.node_embeddings = {}
#         self.node_data = {}
#         self.inter_layer_links = []
#     def add_node(self, node_id, embedding, layer, node_data=None):
#         if layer < len(self.layers):
#             self.layers[layer].add_node(node_id)
#             self.node_embeddings[node_id] = embedding
#             if node_data: self.node_data[node_id] = node_data
#     def hierarchical_search(self, query_embedding, k=3):
#         return {i: list(g.nodes())[:k] for i, g in enumerate(self.layers) if g.nodes()}
#     def build_intra_layer_links(self): pass
#     def build_inter_layer_links(self): pass

# # Load everything
# with open('./checkpoints/clustering/hagrag_all_algorithms.pkl', 'rb') as f:
#     hagrag_results = pickle.load(f)

# print(f"✓ hagrag_results: {list(hagrag_results.keys())}")
# print("Now run Cell 5 → Cell 10A → then any other cell")

## Cell 1: Imports

Standard Python libraries needed for neurosymbolic processing

In [ ]:
# # Standard library imports
# import json
# import re
# import logging
# from typing import List, Dict, Any, Tuple, Optional
# from dataclasses import dataclass
# from collections import defaultdict
# from langchain_classic.chains import LLMChain  # ← ADD THIS
# from langchain_core.prompts import ChatPromptTemplate  # ← ADD THIS

# # Configure logging to see what's happening
# logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# print("✓ Imports loaded")

## Cell 2: Data Structures

Simple classes to represent claims and verification results

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any, Literal

VerificationStatus = Literal["supported", "contradicted", "unsupported", "novel"]

@dataclass
class Claim:
    """
    Atomic statement to verify.
    Keep it compatible with your current code, but add provenance + future ontology hooks.
    """
    subject: str
    relation: str
    object: str
    polarity: bool = True
    qualifier: Optional[str] = None
    source_text: str = ""

    # =% Needed for open-world + auditability
    doc_id: Optional[str] = None
    chunk_id: Optional[str] = None
    sentence_idx: Optional[int] = None

    # =% Optional (for MRHIER / UMLS alignment later)
    subject_cui: Optional[str] = None
    object_cui: Optional[str] = None

    # =% Useful for hierarchy-aware lifting later
    abstraction_level: int = 0          # 0 = original claim, >0 = lifted claim
    lifted_from: Optional[str] = None   # e.g., claim_id of original (if you later add ids)


@dataclass
class LayerEvidence:
    """
    One piece of evidence attached to a specific retrieval layer (your HAGRAG layer 0..3).
    """
    layer: int
    node_id: str
    node_type: str  # "entity" or "community"
    snippet: str
    score: float = 0.0
    meta: Dict[str, Any] = field(default_factory=dict)


@dataclass
class VerificationResult:
    """
    Open-world verification + hierarchical proof container.
    """
    claim: Claim
    status: VerificationStatus
    reason: str

    # Multi-source scoring (keep yours)
    support_score: float = 0.0
    support_sources: List[str] = field(default_factory=list)

    # Hierarchical proof: layer -> evidence list
    hierarchical_proof: Dict[int, List[LayerEvidence]] = field(default_factory=dict)

    # Conflicts for contradicted cases
    conflicting_facts: List[str] = field(default_factory=list)

print(" Updated data structures defined (open-world + hierarchy-ready)")


In [ ]:
import os

umls_dir = "./data/umls/"  #  Your extracted path

essential_files = {
    "MRCONSO.RRF": "Concepts",
    "MRSTY.RRF": "Semantic Types", 
    "MRREL.RRF": "Relations"
}

print("Verifying UMLS files:")
for filename, desc in essential_files.items():
    path = os.path.join(umls_dir, filename)
    if os.path.exists(path):
        size_gb = os.path.getsize(path) / (1024**3)
        print(f" {filename}: {size_gb:.1f} GB - {desc}")
    else:
        print(f"L {filename}: MISSING!")

In [ ]:
def _load_semantic_relations(self):
    """Load valid semantic type patterns from UMLS SRSTR."""
    filepath = os.path.join(self.umls_dir, "SRSTR")
    self.valid_semtype_pairs = defaultdict(list)
    
    if not os.path.exists(filepath):
        logging.warning("SRSTR not found - using fallback")
        self._build_fallback_semantic_relations()
        return
    
    with open(filepath, "r") as f:
        for line in f:
            fields = line.strip().split("|")
            if len(fields) >= 3:
                sty1, rel, sty2 = fields[0], fields[1], fields[2]
                if rel and not rel.startswith("isa"):  # Skip hierarchy relations
                    self.valid_semtype_pairs[rel.lower()].append((sty1, sty2))
    
    logging.info(f"Loaded {len(self.valid_semtype_pairs)} semantic relations from SRSTR")

def _build_fallback_semantic_relations(self):
    """Fallback if SRSTR not available."""
    self.valid_semtype_pairs = {
        "treats": [
            ("Pharmacologic Substance", "Pathologic Function"),
            ("Pharmacologic Substance", "Sign or Symptom"),
            ("Therapeutic or Preventive Procedure", "Pathologic Function"),
        ],
        "causes": [("Substance", "Pathologic Function")],
        "prevents": [("Pharmacologic Substance", "Pathologic Function")],
        "associated_with": "any",
    }

## Cell 3: Claim Extractor

**What it does:** Converts free-form LLM text into structured claims

**Why it matters:** Symbolic verification needs structured input (subject-relation-object triples) - can this be further extended as next research question?

**How it works:** Uses regex patterns to identify claim structures in text

In [ ]:
import re
import logging
from typing import List, Optional, Dict, Any, Tuple

_REL_MAP = {
    # keep only true biomedical-safe canonicalizations
    "treat": "TREATS",
    "treats": "TREATS",
    "cause": "CAUSES",
    "causes": "CAUSES",
    "associated_with": "ASSOCIATED_WITH",
    "associated": "ASSOCIATED_WITH",
    "linked": "ASSOCIATED_WITH",
    "related": "ASSOCIATED_WITH",
    "correlated": "ASSOCIATED_WITH",
    "increase": "INCREASES",
    "increases": "INCREASES",
    "elevate": "INCREASES",
    "elevates": "INCREASES",
    "decrease": "DECREASES",
    "decreases": "DECREASES",
    "lower": "DECREASES",
    "lowers": "DECREASES",
    "reduce": "DECREASES",     # optional: consider "REDUCES" if you want separate
    "reduces": "DECREASES",
}

def normalize_relation(r: str) -> str:
    """
    Extraction-layer normalizer (biomedical-safe):
    - normalizes surface forms into a small set ONLY when confident
    - otherwise preserves the original relation (normalized formatting)
    """
    r = (r or "").strip()
    if not r:
        return ""

    key = r.lower().replace(" ", "_").replace("-", "_")
    key = re.sub(r"[^a-z0-9_]", "", key)

    # IMPORTANT: do NOT map generic "is" to IS_A (too many false positives)
    # Let your regex pattern tag "IS_A" handle true is-a cases.
    if key in ("is", "are", "was", "were"):
        return key.upper()  # keep as IS/ARE/etc. or return "" if you prefer dropping it

    mapped = _REL_MAP.get(key)
    if mapped:
        return mapped

    # preserve richer relations from LLM triplets (e.g., may_treat, has_finding_site)
    return key.upper()


class BaseClaimExtractor:
    def extract(self, text: str, meta: Optional[Dict[str, Any]] = None) -> List[Claim]:
        raise NotImplementedError


class RegexClaimExtractor(BaseClaimExtractor):
    """
    Baseline extractor (keep for ablations).
    Improvement: can extract MULTIPLE claims per sentence (not first match only).
    """

    def extract(self, text: str, meta: Optional[Dict[str, Any]] = None) -> List[Claim]:
        meta = meta or {}
        claims: List[Claim] = []
        sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]

        for idx, sentence in enumerate(sentences):
            if len(sentence) < 10:
                continue
            claims.extend(self._all_matches(sentence, idx, meta))

        logging.info(f"Extracted {len(claims)} claims (regex baseline)")
        return claims

    def _all_matches(self, sentence: str, sentence_idx: int, meta: Dict[str, Any]) -> List[Claim]:
        out: List[Claim] = []

        patterns = [
            ("IS_A", r'(\w+(?:\s+\w+)?)\s+(?:is|are|was|were)\s+(?:a|an|the)?\s*(\w+(?:\s+\w+)*?)$'),
            ("ASSOCIATED_WITH", r'(\w+(?:\s+\w+)*)\s+(?:is|are|was|were)\s+(?:associated with|linked to|related to|correlated with)\s+(\w+(?:\s+\w+)*)'),
            ("CAUSES", r'(\w+(?:\s+\w+)*)\s+(?:causes?|leads? to|results? in|triggers?)\s+(\w+(?:\s+\w+)*)'),
            ("TREATS", r'(\w+(?:\s+\w+)*)\s+(?:treats?|prevents?|reduces?|alleviates?)\s+(\w+(?:\s+\w+)*)'),
            ("INCDEC", r'(\w+(?:\s+\w+)*)\s+(increases?|decreases?|elevates?|lowers?|reduces?)\s+(\w+(?:\s+\w+)*)'),
        ]

        # Negation
        neg = re.search(r"(\w+(?:\s+\w+)*)\s+(?:does not|do not|doesn't|don't|is not|are not)\s+(\w+)\s+(\w+(?:\s+\w+)*)", sentence, re.I)
        if neg:
            out.append(Claim(
                subject=neg.group(1).strip(),
                relation=normalize_relation(neg.group(2).strip()),
                object=neg.group(3).strip(),
                polarity=False,
                source_text=sentence,
                doc_id=meta.get("doc_id"),
                chunk_id=meta.get("chunk_id"),
                sentence_idx=sentence_idx
            ))

        for tag, pat in patterns:
            for m in re.finditer(pat, sentence, flags=re.I):
                if tag == "INCDEC":
                    verb = m.group(2).lower()
                    rel = "INCREASES" if ("increas" in verb or "elevat" in verb) else "DECREASES"
                    subj, obj = m.group(1).strip(), m.group(3).strip()
                else:
                    rel = tag
                    subj, obj = m.group(1).strip(), m.group(2).strip()

                out.append(Claim(
                    subject=subj,
                    relation=normalize_relation(rel),
                    object=obj,
                    source_text=sentence,
                    doc_id=meta.get("doc_id"),
                    chunk_id=meta.get("chunk_id"),
                    sentence_idx=sentence_idx
                ))

        return out


class TripletClaimExtractor(BaseClaimExtractor):
    """
    Main extractor: reuse YOUR existing TripletExtractor (LLM JSON triplets)
    to extract claims from the draft answer.
    """

    def __init__(self, triplet_extractor, max_paths_per_answer: int = 8):
        self.triplet_extractor = triplet_extractor
        self.max_paths_per_answer = max_paths_per_answer

    def _clean_text(self, text: str) -> str:
        # Remove code fences ```...```
        text = re.sub(r"```.*?```", " ", text, flags=re.DOTALL)
        # Remove markdown table pipes and headers (keeps words)
        text = re.sub(r"\|", " ", text)
        # Collapse whitespace
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def _bad_entity(self, s: str) -> bool:
        if not s:
            return True
        s = s.strip()
        if len(s) < 3:
            return True
        # too long / sentence-like
        if len(s.split()) > 12:
            return True
        # must contain letters
        if not re.search(r"[A-Za-z]", s):
            return True
        # avoid common junk pronouns/fragments
        junk = {"this", "that", "it", "they", "we", "you", "i"}
        if s.lower() in junk:
            return True
        return False

    def extract(self, text: str, meta: Optional[Dict[str, Any]] = None) -> List[Claim]:
        meta = meta or {}
        cleaned = self._clean_text(text)

        # IMPORTANT: request more triplets from the answer
#         entities, relationships = self.triplet_extractor.extract_triplets(
#             cleaned,
#             max_paths_per_chunk=self.max_paths_per_answer
#         )
        entities, relationships = self.triplet_extractor.extract_claim_triplets(cleaned, max_claims=self.max_paths_per_answer)


        claims: List[Claim] = []
        for (src, tgt, rel, rel_desc, rel_attrs) in relationships:
            subj = str(src).strip()
            obj = str(tgt).strip()
            if self._bad_entity(subj) or self._bad_entity(obj):
                continue

            claims.append(Claim(
                subject=subj,
                relation=normalize_relation(str(rel)),
                object=obj,
                polarity=True,
                qualifier=(rel_attrs.get("context") if isinstance(rel_attrs, dict) else None),
                source_text=(rel_desc or ""),
                doc_id=meta.get("doc_id"),
                chunk_id=meta.get("chunk_id"),
                sentence_idx=None
            ))

        logging.info(f"Extracted {len(claims)} claims (TripletExtractor-based)")
        return claims


print(" Updated ClaimExtractor options ready:")
print("  - RegexClaimExtractor (baseline)")
print("  - TripletClaimExtractor (recommended, reuses your TripletExtractor)")


## Cell 4: Symbolic Verifier

**What it does:** Checks claims against ontology rules and KG facts

**Why it matters:** This is the SYMBOLIC reasoning component - catches violations

**How it works:** 
1. Check ontology constraints (type safety)
2. Check KG consistency (does evidence support this?)
3. Return accept/reject/qualify decision

In [ ]:
"""
CELL 4 - FIXED VERSION v2
UMLSOntology + SymbolicVerifier

BUGS FIXED:
1. get_cui() / get_aui(): overlap >= 2  overlap >= 1 (single-word entities now work)
2. get_mrhier_ancestors(): NOW SEARCHES ALL AUIs FOR A CUI (critical fix!)
3. _try_mrhier_lift(): Allow partial ancestors (not both required)
4. _layer_support_evidence(): Allow partial text matches (subject OR object)
5. _decide_open_world(): Better novel detection threshold
6. Safer fake_claim creation in lifting
"""

import os
import re
import logging
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Any, Iterable


# ============================================================
# UMLSOntology - FIXED v2
# ============================================================

class UMLSOntology:
    """
    UMLS loader for:
      - MRCONSO: string  CUI + AUI
      - MRSTY: CUI  semantic types
      - MRREL: CUI  (RELA, CUI)
      - MRHIER (optional): AUI  PTR path (for lifting)
    """

    def __init__(
        self,
        umls_dir: str,
        load_mrhier: bool = False,
        mrhier_sabs: Optional[Iterable[str]] = ("MSH", "RXNORM", "SNOMEDCT_US"),
        mrrel_max_rows: int = 200000,
        mrhier_max_rows: Optional[int] = 5000000,
    ):
        self.umls_dir = umls_dir

        # MRCONSO
        self.cui_to_str: Dict[str, str] = {}
        self.str_to_cui: Dict[str, str] = {}
        self.str_to_aui: Dict[str, str] = {}
        self.aui_to_str: Dict[str, str] = {}
        self.aui_to_cui: Dict[str, str] = {}
        
        # NEW: CUI to all AUIs mapping (for hierarchy lookup)
        self.cui_to_auis: Dict[str, List[str]] = defaultdict(list)

        # MRSTY
        self.cui_to_semtype: Dict[str, List[str]] = {}

        # MRREL
        self.relations = defaultdict(list)

        # MRHIER (optional)
        self.mrhier_loaded = False
        self.aui_to_ptr: Dict[str, str] = {}
        self.aui_to_paui: Dict[str, str] = {}
        self.aui_to_sab: Dict[str, str] = {}
        self.mrhier_sabs = set(mrhier_sabs) if mrhier_sabs else None

        # Semantic network rules
        self.semtype_hierarchy: Dict[str, List[str]] = {}

        logging.info("Loading UMLS...")
        self._load_mrconso()
        self._load_mrsty()
        self._load_mrrel(max_rows=mrrel_max_rows)
        self._build_semantic_network()
        self._load_semantic_relations() 

        if load_mrhier:
            self._load_mrhier(max_rows=mrhier_max_rows)

        logging.info(f"UMLS loaded: {len(self.cui_to_str)} concepts")

    def _load_mrconso(self, max_rows: Optional[int] = None):
        filepath = os.path.join(self.umls_dir, "MRCONSO.RRF")
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"MRCONSO.RRF not found at {filepath}")

        count = 0
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                if max_rows and count >= max_rows:
                    break
                fields = line.rstrip("\n").split("|")
                if len(fields) < 15:
                    continue

                cui = fields[0]
                lat = fields[1]
                ispref = fields[6]
                aui = fields[7]
                sab = fields[11]
                str_name = fields[14]

                if lat == "ENG" and ispref == "Y" and sab in ["MSH", "SNOMEDCT_US", "NCI", "RXNORM"]:
                    name_l = str_name.lower()
                    self.cui_to_str[cui] = str_name
                    self.str_to_cui[name_l] = cui

                    if aui:
                        self.str_to_aui[name_l] = aui
                        self.aui_to_str[aui] = str_name
                        self.aui_to_cui[aui] = cui
                        # NEW: Build reverse mapping CUI -> all AUIs
                        self.cui_to_auis[cui].append(aui)

                    count += 1

        logging.info(f"Loaded {count} preferred EN concepts from MRCONSO")

    def _load_mrsty(self):
        filepath = os.path.join(self.umls_dir, "MRSTY.RRF")
        if not os.path.exists(filepath):
            logging.warning("MRSTY.RRF not found - semantic types unavailable")
            return

        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                fields = line.rstrip("\n").split("|")
                if len(fields) >= 4:
                    cui = fields[0]
                    sty = fields[3]
                    self.cui_to_semtype.setdefault(cui, []).append(sty)

        logging.info(f"Loaded semantic types for {len(self.cui_to_semtype)} CUIs")

    def _load_mrrel(self, max_rows: int = 200000):
        filepath = os.path.join(self.umls_dir, "MRREL.RRF")
        if not os.path.exists(filepath):
            logging.warning("MRREL.RRF not found - relationships unavailable")
            return

        count = 0
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                if max_rows and count >= max_rows:
                    break
                fields = line.rstrip("\n").split("|")
                if len(fields) >= 8:
                    cui1 = fields[0]
                    cui2 = fields[4]
                    rela = fields[7]

                    if rela:
                        self.relations[cui1].append((rela.lower(), cui2))
                        count += 1

        logging.info(f"Loaded {count} MRREL relations (capped)")

    def _load_mrhier(self, max_rows: Optional[int] = 5000000):
        filepath = os.path.join(self.umls_dir, "MRHIER.RRF")
        if not os.path.exists(filepath):
            logging.warning("MRHIER.RRF not found - lifting unavailable")
            return

        count = 0
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                if max_rows and count >= max_rows:
                    break
                fields = line.rstrip("\n").split("|")
                if len(fields) < 7:
                    continue

                cui = fields[0]
                aui = fields[1]
                paui = fields[3]
                sab = fields[4]
                ptr = fields[6]

                if not aui:
                    continue
                if self.mrhier_sabs and sab not in self.mrhier_sabs:
                    continue

                if aui not in self.aui_to_ptr and ptr:
                    self.aui_to_ptr[aui] = ptr
                if aui not in self.aui_to_paui and paui:
                    self.aui_to_paui[aui] = paui
                if aui not in self.aui_to_sab and sab:
                    self.aui_to_sab[aui] = sab
                    
                # NEW: Also add this AUI to cui_to_auis if not already there
                if cui and aui not in self.cui_to_auis.get(cui, []):
                    self.cui_to_auis[cui].append(aui)

                count += 1

        self.mrhier_loaded = True
        logging.info(f"Loaded {count} MRHIER rows (filtered)")

#     def _build_semantic_network(self):
#         self.semtype_hierarchy = {
#             "Pharmacologic Substance": ["Chemical Viewed Functionally"],
#             "Antibiotic": ["Pharmacologic Substance"],
#             "Hormone": ["Pharmacologic Substance"],
#             "Vitamin": ["Pharmacologic Substance"],
#             "Disease or Syndrome": ["Pathologic Function"],
#             "Neoplastic Process": ["Disease or Syndrome"],
#             "Sign or Symptom": ["Finding"],
#             "Laboratory or Test Result": ["Finding"],
#             "Body Part, Organ, or Organ Component": ["Anatomical Structure"],
#             "Cell": ["Anatomical Structure"],
#             "Amino Acid, Peptide, or Protein": ["Chemical Viewed Structurally"],
#             "Enzyme": ["Amino Acid, Peptide, or Protein"],
#         }
        
    def _build_semantic_network(self):
        """
        Disabled manual semantic hierarchy.
        We rely exclusively on UMLS SRSTR for semantic validation.
        Kept for backward compatibility so code does not break.
        """
        self.semtype_hierarchy = {}

    # ============================================================
    # FIX 1: get_cui - Allow single-word matching (overlap >= 1)
    # ============================================================
    def get_cui(self, entity_name: str) -> Optional[str]:
        if not entity_name:
            return None
        name_l = entity_name.lower().strip()
        
        # Exact match first
        cui = self.str_to_cui.get(name_l)
        if cui:
            return cui

        # Partial match - FIX: Changed from >= 2 to >= 1
        entity_words = set(name_l.split())
        best_cui, best_overlap = None, 0
        
        for umls_name, c in self.str_to_cui.items():
            umls_words = set(umls_name.split())
            overlap = len(entity_words & umls_words)
            
            # FIX: Allow single-word match (was >= 2, now >= 1)
            if overlap > best_overlap and overlap >= 1:
                best_overlap = overlap
                best_cui = c
                
        return best_cui

    # ============================================================
    # FIX 1: get_aui - Allow single-word matching (overlap >= 1)
    # ============================================================
    def get_aui(self, entity_name: str) -> Optional[str]:
        if not entity_name:
            return None
        name_l = entity_name.lower().strip()
        
        # Exact match first
        aui = self.str_to_aui.get(name_l)
        if aui:
            return aui

        # Partial match - FIX: Changed from >= 2 to >= 1
        entity_words = set(name_l.split())
        best_aui, best_overlap = None, 0
        
        for umls_name, a in self.str_to_aui.items():
            umls_words = set(umls_name.split())
            overlap = len(entity_words & umls_words)
            
            # FIX: Allow single-word match (was >= 2, now >= 1)
            if overlap > best_overlap and overlap >= 1:
                best_overlap = overlap
                best_aui = a
                
        return best_aui

    def get_semantic_type(self, entity_name: str, cui: Optional[str] = None) -> List[str]:
        cui = cui or self.get_cui(entity_name)
        if not cui:
            return ["Unknown"]
        return self.cui_to_semtype.get(cui, ["Unknown"])


    def _load_semantic_relations(self):
        """Load valid semantic type patterns from UMLS SRSTR."""
        filepath = os.path.join(self.umls_dir, "SRSTR")
        self.valid_semtype_pairs = defaultdict(list)

        if not os.path.exists(filepath):
            logging.warning("SRSTR not found - using permissive mode")
            self.valid_semtype_pairs = None  # Will trigger permissive mode
            return

        with open(filepath, "r") as f:
            for line in f:
                fields = line.strip().split("|")
                if len(fields) >= 3:
                    sty1, rel, sty2 = fields[0], fields[1], fields[2]
                    if rel and rel != "isa":
                        self.valid_semtype_pairs[rel.lower()].append((sty1, sty2))

        logging.info(f"Loaded {len(self.valid_semtype_pairs)} semantic relations from SRSTR")

# Replace is_valid_relation() with:
    def is_valid_relation(self, subject: str, relation: str, object_name: str) -> Tuple[bool, str]:
        """Validate relation using SRSTR semantic network."""
        subject_types = self.get_semantic_type(subject)
        object_types = self.get_semantic_type(object_name)

        if "Unknown" in subject_types or "Unknown" in object_types:
            return True, "Entity not in UMLS (permissive)"

        # If SRSTR not loaded, be permissive
        if not hasattr(self, 'valid_semtype_pairs') or self.valid_semtype_pairs is None:
            return True, "SRSTR not loaded (permissive)"

        rel = relation.strip().lower()

        # Check if relation exists in SRSTR
        if rel not in self.valid_semtype_pairs:
            return True, f"Relation '{rel}' not in SRSTR (permissive)"

        valid_pairs = self.valid_semtype_pairs[rel]

        # Check all subject/object type combinations
        for st in subject_types:
            for ot in object_types:
                # Direct match
                if (st, ot) in valid_pairs:
                    return True, f"Valid: {st} --[{rel}]--> {ot}"

                # Check parent types (Pathologic Function includes Disease or Syndrome)
                for vst, vot in valid_pairs:
                    # Partial/parent match
                    if (vst in st or st in vst) and (vot in ot or ot in vot):
                        return True, f"Valid (partial): {st} --[{rel}]--> {ot}"

        # Be permissive - don't reject, just note
        return True, f"Permissive pass: {subject_types[0]} --[{rel}]--> {object_types[0]}"
        

#     def check_relation_in_umls(self, subject: str, relation: str, object_name: str) -> bool:
#         if not subject or not relation or not object_name:
#             return False

#         subject_cui = self.get_cui(subject)
#         object_cui = self.get_cui(object_name)
#         if not subject_cui or not object_cui:
#             return False

#         rel = relation.strip().lower()

#         rel_synonyms = {
#             "treat": "treats",
#             "treated_by": "treats",
#             "manage": "treats",
#             "manages": "treats",
#             "cause": "causes",
#             "result_in": "causes",
#             "results_in": "causes",
#             "associated": "associated_with",
#             "related": "associated_with",
#             "linked": "associated_with",
#             "correlated": "associated_with",
#             "isa": "is_a",
#             "is-a": "is_a",
#         }
#         rel = rel_synonyms.get(rel, rel)

#         for rela, target_cui in self.relations.get(subject_cui, []):
#             if rela == rel and target_cui == object_cui:
#                 return True

#         return False
    def check_relation_in_umls(self, subject: str, relation: str, object_name: str) -> bool:
        if not subject or not relation or not object_name:
            return False

        subject_cui = self.get_cui(subject)
        object_cui = self.get_cui(object_name)

        if not subject_cui or not object_cui:
            return False

        # Normalize relation safely
        rel = relation.strip().lower().replace(" ", "_").replace("-", "_")

        # Try exact match first
        for rela, target_cui in self.relations.get(subject_cui, []):
            if rela == rel and target_cui == object_cui:
                return True

        # Try relaxed matching (prefix match like treat vs treats)
        for rela, target_cui in self.relations.get(subject_cui, []):
            if target_cui == object_cui:
                if rela.startswith(rel) or rel.startswith(rela):
                    return True

        return False
    # ============================================================
    # CRITICAL FIX: get_mrhier_ancestors - Search ALL AUIs for a CUI
    # ============================================================
    def get_mrhier_ancestors(self, entity_name: str, max_depth: int = 3) -> List[str]:
        """
        FIXED: Find ancestors by checking ALL AUIs for the entity's CUI,
        not just the one AUI stored in str_to_aui.
        
        The problem was:
        - str_to_aui stores ONE AUI per entity (e.g., from NCI)
        - aui_to_ptr only has AUIs from MSH/RXNORM/SNOMEDCT_US
        - Same concept has different AUIs in different vocabularies
        - So we need to find ANY AUI for the CUI that has hierarchy data
        """
        if not self.mrhier_loaded:
            return []

        # Step 1: Get CUI for this entity
        cui = self.get_cui(entity_name)
        if not cui:
            return []
        
        # Step 2: Try all AUIs for this CUI to find one with hierarchy
        candidate_auis = self.cui_to_auis.get(cui, [])
        
        for aui in candidate_auis:
            # Check if this AUI has PTR path
            if aui in self.aui_to_ptr:
                ancestors = self._get_ancestors_from_ptr(aui, max_depth)
                if ancestors:
                    return ancestors
            
            # Check if this AUI has PAUI chain
            if aui in self.aui_to_paui:
                ancestors = self._get_ancestors_from_paui(aui, max_depth)
                if ancestors:
                    return ancestors
        
        # Step 3: Fallback - direct AUI lookup (old behavior)
        aui = self.get_aui(entity_name)
        if aui:
            if aui in self.aui_to_ptr:
                return self._get_ancestors_from_ptr(aui, max_depth)
            if aui in self.aui_to_paui:
                return self._get_ancestors_from_paui(aui, max_depth)
        
        return []
    
    def _get_ancestors_from_ptr(self, aui: str, max_depth: int) -> List[str]:
        """Extract ancestor names from PTR path (dot-separated AUI list)."""
        ptr = self.aui_to_ptr.get(aui)
        if not ptr:
            return []
        
        path = [p for p in ptr.split(".") if p]
        # PTR is root...parent, so reverse to get parent first
        ancestors_aui = list(reversed(path))[:max_depth]
        
        # Convert AUIs to names
        ancestors = []
        for a in ancestors_aui:
            name = self.aui_to_str.get(a)
            if name:
                ancestors.append(name)
        
        return ancestors
    
    def _get_ancestors_from_paui(self, aui: str, max_depth: int) -> List[str]:
        """Walk up PAUI chain to find ancestors."""
        ancestors = []
        current = self.aui_to_paui.get(aui)
        
        while current and len(ancestors) < max_depth:
            ancestor_name = self.aui_to_str.get(current)
            if ancestor_name and ancestor_name != current:
                ancestors.append(ancestor_name)
            current = self.aui_to_paui.get(current)
        
        return ancestors


# ============================================================
# SymbolicVerifier - FIXED
# ============================================================

class SymbolicVerifier:
    """
    Open-world verifier + hierarchy-aware proof builder.
    
    FIXES APPLIED:
    - _layer_support_evidence: Allow partial matches (subject OR object)
    - _try_mrhier_lift: Allow partial ancestors
    - _decide_open_world: Better novel detection
    """

    def __init__(self, graph_store, umls: UMLSOntology, max_layers: int = 4, enable_mrhier_lifting: bool = True):
        self.graph_store = graph_store
        self.umls = umls
        self.max_layers = max_layers
        self.enable_mrhier_lifting = enable_mrhier_lifting and umls.mrhier_loaded

        self.w_neo4j = 1.0
        self.w_umls = 0.8
        self.w_text = 0.5
        self.w_text_partial = 0.25  # NEW: weight for partial text match

        # Build community index
        self._community_index = {}
        try:
            for layer, comms in (self.graph_store.hierarchical_communities or {}).items():
                for c in comms:
                    cid = c.get("id")
                    if cid:
                        self._community_index[cid] = c
        except Exception:
            pass

        # Build entity to L0 community mapping
        self._entity_to_l0 = defaultdict(set)
        try:
            for c in (self.graph_store.hierarchical_communities.get(0, []) if self.graph_store.hierarchical_communities else []):
                cid = c.get("id")
                for m in c.get("members", []):
                    if cid and m and not str(m).startswith("L"):
                        self._entity_to_l0[str(m).lower()].add(cid)
        except Exception:
            pass

    def verify(self, claims: List["Claim"], evidence: Dict) -> List["VerificationResult"]:
        return [self._verify_one(c, evidence) for c in claims]

    def _verify_one(self, claim: "Claim", evidence: Dict) -> "VerificationResult":
        rel_l = (claim.relation or "").lower()

        # 1) Semantic validity check
        is_valid, why = self.umls.is_valid_relation(claim.subject, rel_l, claim.object)
        if not is_valid:
            return VerificationResult(
                claim=claim,
                status="contradicted",
                reason=f"Ontology violation: {why}",
                support_score=0.0,
                support_sources=["UMLS Semantic"],
                hierarchical_proof={}
            )

        proof: Dict[int, List[LayerEvidence]] = defaultdict(list)
        sources: List[str] = []
        score = 0.0

        # 2) Neo4j edge support
        neo4j_ev = self._neo4j_support(claim)
        if neo4j_ev:
            score += self.w_neo4j
            sources.append("Neo4j KG")
            proof[0].append(LayerEvidence(
                layer=0,
                node_id=f"neo4j:{claim.subject}->{claim.relation}->{claim.object}",
                node_type="neo4j_edge",
                snippet=neo4j_ev[:220],
                score=self.w_neo4j,
                meta={"kind": "edge"}
            ))

        # 3) UMLS MRREL support
        umls_ok = self.umls.check_relation_in_umls(claim.subject, claim.relation, claim.object)
        if umls_ok:
            score += self.w_umls
            sources.append("UMLS MRREL")

        # 4) Text + membership evidence across layers (FIXED)
        layer_evs, has_full_match, has_partial_match = self._layer_support_evidence(claim, evidence)
        if layer_evs:
            if has_full_match:
                score += self.w_text
                sources.append("Text evidence (full)")
            elif has_partial_match:
                score += self.w_text_partial
                sources.append("Text evidence (partial)")
            for ev in layer_evs:
                proof[ev.layer].append(ev)

        # 5) Drill-down from high layers to L0
        self._drill_down_attach_L0(claim, evidence, proof)

        # Calculate metrics
        layers_supported = sorted([k for k, v in proof.items() if v])
        vcs = len(layers_supported) / float(self.max_layers) if self.max_layers else 0.0
        msl = min(layers_supported) if layers_supported else None

        # 6) Decide status (FIXED thresholds)
        status, reason = self._decide_open_world(
            sources=sources,
            has_full_text=has_full_match,
            has_partial_text=has_partial_match,
            has_umls=umls_ok,
            has_neo4j=bool(neo4j_ev),
            vcs=vcs,
            msl=msl
        )

        # 7) Optional MRHIER lifting for unsupported claims (FIXED)
        if status == "unsupported" and self.enable_mrhier_lifting:
            lifted = self._try_mrhier_lift(claim, evidence)
            if lifted is not None:
                status = "supported"
                reason = lifted["reason"]
                sources = list(set(sources + lifted["sources"]))
                score = max(score, lifted["score"])
                for ev in lifted["evidence"]:
                    proof[ev.layer].append(ev)

        proof = {int(k): v for k, v in proof.items()}

        return VerificationResult(
            claim=claim,
            status=status,
            reason=reason,
            support_score=score,
            support_sources=sources,
            hierarchical_proof=proof
        )

    # ============================================================
    # FIX 4: Better open-world decision with partial text support
    # ============================================================
    def _decide_open_world(
        self, 
        sources: List[str], 
        has_full_text: bool, 
        has_partial_text: bool,
        has_umls: bool, 
        has_neo4j: bool, 
        vcs: float, 
        msl: Optional[int]
    ) -> Tuple[str, str]:
        
        # Strong support: Neo4j edge OR (UMLS + full text) OR (full text + good VCS)
        if has_neo4j or (has_umls and has_full_text) or (has_full_text and vcs >= 0.5):
            return "supported", f"Supported by: {', '.join(sources)} (VCS={vcs:.2f}, MSL={msl})"

        # Medium support: UMLS alone OR (partial text + UMLS)
        if has_umls or (has_partial_text and has_umls):
            return "supported", f"Supported by: {', '.join(sources)} (VCS={vcs:.2f}, MSL={msl})"

        # FIX: Novel if ANY text evidence (full or partial) but no KG/UMLS
        if (has_full_text or has_partial_text) and not has_umls and not has_neo4j:
            return "novel", f"Text-supported but missing in KG/UMLS (VCS={vcs:.2f}, MSL={msl})"

        return "unsupported", "No supporting evidence found (open-world: not treated as false)"

    def _neo4j_support(self, claim: "Claim") -> Optional[str]:
        try:
            with self.graph_store.driver.session() as session:
                q = """
                MATCH (s:Entity {name: $subject})-[r:RELATION]->(o:Entity {name: $object})
                WHERE r.type = $relation
                RETURN r.description as desc, r.doc_id as doc_id
                LIMIT 1
                """
                rec = session.run(
                    q,
                    subject=claim.subject,
                    object=claim.object,
                    relation=str(claim.relation).upper()
                ).single()
                if rec:
                    return f"{rec.get('desc','')} (doc_id={rec.get('doc_id','')})"
        except Exception as e:
            logging.debug(f"Neo4j check failed: {e}")
        return None

    # ============================================================
    # FIX 3: Layer support evidence - Allow partial matches
    # ============================================================
    def _layer_support_evidence(self, claim: "Claim", evidence: Dict) -> Tuple[List["LayerEvidence"], bool, bool]:
        """
        Returns: (evidence_list, has_full_match, has_partial_match)
        
        FIX: Now accepts partial matches (subject OR object) with lower score
        """
        out: List[LayerEvidence] = []
        has_full_match = False
        has_partial_match = False
        
        subj = (claim.subject or "").lower()
        obj = (claim.object or "").lower()
        if not subj or not obj:
            return out, False, False

        for layer_key, items in evidence.items():
            try:
                layer = int(layer_key)
            except Exception:
                continue

            for it in items:
                node_id = it.get("id", it.get("node_id", "unknown"))
                node_type = it.get("type", "unknown")
                content = it.get("content", {}) or {}

                # 1) MEMBERSHIP MATCH
                members = content.get("members", []) if isinstance(content, dict) else []
                if isinstance(members, list) and members:
                    mem_l = {str(m).lower() for m in members}

                    # Case A: members contain entity names
                    subj_in_mem = subj in mem_l
                    obj_in_mem = obj in mem_l
                    
                    if subj_in_mem or obj_in_mem:
                        if subj_in_mem and obj_in_mem:
                            match_type = "members_both"
                            match_score = self.w_text
                            has_full_match = True
                        else:
                            match_type = "members_partial"
                            match_score = self.w_text_partial
                            has_partial_match = True
                            
                        snippet = "members match: " + ", ".join(list(members)[:12])
                        out.append(LayerEvidence(
                            layer=layer, 
                            node_id=str(node_id), 
                            node_type=str(node_type),
                            snippet=snippet[:240], 
                            score=match_score, 
                            meta={"match": match_type}
                        ))
                        continue

                    # Case B: members contain community IDs
                    subj_l0 = self._entity_to_l0.get(subj, set())
                    obj_l0 = self._entity_to_l0.get(obj, set())
                    
                    subj_comm_match = any(c.lower() in mem_l for c in subj_l0)
                    obj_comm_match = any(c.lower() in mem_l for c in obj_l0)
                    
                    if subj_comm_match or obj_comm_match:
                        if subj_comm_match and obj_comm_match:
                            match_type = "members_comm_both"
                            match_score = self.w_text
                            has_full_match = True
                        else:
                            match_type = "members_comm_partial"
                            match_score = self.w_text_partial
                            has_partial_match = True
                            
                        snippet = f"L0 community match: subj={list(subj_l0)[:3]}, obj={list(obj_l0)[:3]}"
                        out.append(LayerEvidence(
                            layer=layer, 
                            node_id=str(node_id), 
                            node_type=str(node_type),
                            snippet=snippet[:240], 
                            score=match_score, 
                            meta={"match": match_type}
                        ))
                        continue

                # 2) TEXT MATCH (FIXED: allow partial)
                txt = ""
                if isinstance(content, dict):
                    txt = (content.get("description") or content.get("summary") or content.get("name") or "")
                else:
                    txt = str(content)

                tl = txt.lower()
                subj_in_text = subj in tl
                obj_in_text = obj in tl
                
                # FIX: Accept partial matches with lower score
                if subj_in_text or obj_in_text:
                    if subj_in_text and obj_in_text:
                        match_type = "text_both"
                        match_score = self.w_text
                        has_full_match = True
                    else:
                        match_type = "text_partial"
                        match_score = self.w_text_partial
                        has_partial_match = True
                    
                    out.append(LayerEvidence(
                        layer=layer,
                        node_id=str(node_id),
                        node_type=str(node_type),
                        snippet=self._make_snippet(txt, subj, obj),
                        score=match_score,
                        meta={"match": match_type, "subj_found": subj_in_text, "obj_found": obj_in_text}
                    ))

        return out, has_full_match, has_partial_match

    def _make_snippet(self, text: str, subj_l: str, obj_l: str, window: int = 120) -> str:
        tl = text.lower()
        i = tl.find(subj_l)
        j = tl.find(obj_l)
        positions = [x for x in [i, j] if x != -1]
        k = min(positions) if positions else 0
        start = max(0, k - window)
        end = min(len(text), k + window)
        s = text[start:end].replace("\n", " ")
        return (s[:240] + "...") if len(s) > 240 else s

    def _get_comm(self, node_id: str) -> Optional[Dict[str, Any]]:
        if not node_id:
            return None
        return self._community_index.get(node_id)

    def _drill_down_attach_L0(self, claim: "Claim", evidence: Dict, proof: Dict[int, List[LayerEvidence]], max_entities: int = 10):
        start = None
        for layer in (3, 2, 1):
            if layer in proof and proof[layer]:
                start = proof[layer][0].node_id
                break
        if not start:
            return

        subj = claim.subject.lower()
        obj = claim.object.lower()

        queue = [start]
        visited = set()
        l0_hits = 0

        while queue and l0_hits < 3:
            nid = queue.pop(0)
            if nid in visited:
                continue
            visited.add(nid)

            comm = self._get_comm(nid)
            if not comm:
                continue

            members = comm.get("members", [])
            if not isinstance(members, list):
                continue

            if str(comm.get("layer")) == "0" or comm.get("id", "").startswith("L0_"):
                mem_l = {str(m).lower() for m in members}
                if (subj in mem_l) or (obj in mem_l):
                    snippet = f"Drill-down: {nid} members include claim entity(ies)"
                    proof[0].append(LayerEvidence(
                        layer=0,
                        node_id=nid,
                        node_type="community",
                        snippet=snippet,
                        score=0.2,
                        meta={"match": "drill_down_L0", "from": start}
                    ))
                    l0_hits += 1

                for e in members[:max_entities]:
                    proof[0].append(LayerEvidence(
                        layer=0,
                        node_id=str(e),
                        node_type="entity",
                        snippet=f"Entity from {nid}: {e}",
                        score=0.1,
                        meta={"match": "drill_down_entity", "from": nid}
                    ))
                continue

            for m in members:
                m = str(m)
                if m.startswith("L"):
                    queue.append(m)

    # ============================================================
    # FIX 2: MRHIER lift - Allow partial ancestors
    # ============================================================
    def _try_mrhier_lift(self, claim: "Claim", evidence: Dict) -> Optional[Dict[str, Any]]:
        """
        FIX: No longer requires BOTH subject AND object to have ancestors.
        If one has ancestors, use original for the other.
        """
        subj_anc = self.umls.get_mrhier_ancestors(claim.subject, max_depth=2)
        obj_anc = self.umls.get_mrhier_ancestors(claim.object, max_depth=2)
        
        # FIX: Only fail if NEITHER has ancestors
        if not subj_anc and not obj_anc:
            return None
        
        # FIX: If one is empty, use original entity
        if not subj_anc:
            subj_anc = [claim.subject]
        if not obj_anc:
            obj_anc = [claim.object]

        best_score = 0.0
        best_sources: List[str] = []
        best_evidence: List[LayerEvidence] = []
        best_lift_info = ""

        for sA in subj_anc:
            for oA in obj_anc:
                # Skip if both are original (no actual lifting)
                if sA == claim.subject and oA == claim.object:
                    continue
                
                # FIX: Safer fake claim creation using simple object
                class FakeClaim:
                    def __init__(self, subject, relation, obj):
                        self.subject = subject
                        self.relation = relation
                        self.object = obj
                
                fake_claim = FakeClaim(sA, claim.relation, oA)

                # Check text evidence for lifted claim
                text_evs, has_full, has_partial = self._layer_support_evidence(fake_claim, evidence)
                
                # Check UMLS for lifted claim
                umls_ok = self.umls.check_relation_in_umls(sA, claim.relation, oA)

                sc = 0.0
                srcs = []
                
                if umls_ok:
                    sc += self.w_umls
                    srcs.append("UMLS MRREL (lifted)")
                if has_full:
                    sc += self.w_text
                    srcs.append("Text evidence (lifted, full)")
                elif has_partial:
                    sc += self.w_text_partial
                    srcs.append("Text evidence (lifted, partial)")

                if sc > best_score and (umls_ok or has_full or has_partial):
                    best_score = sc
                    best_sources = srcs
                    best_evidence = text_evs
                    best_lift_info = f"Lifted: {claim.subject}->{sA}, {claim.object}->{oA}"


        if best_score > 0.0:
            return {
                "score": best_score,
                "sources": best_sources + ["MRHIER Lifting"],
                "evidence": best_evidence,
                "reason": f"Supported via MRHIER lifting: {best_lift_info}. Sources: {', '.join(best_sources)}"
            }
        
        # NEW v3 FIX: MRHIER hierarchy alone = weak support
        subj_info = f"{claim.subject}{subj_anc[0]}" if subj_anc and subj_anc[0] != claim.subject else claim.subject
        obj_info = f"{claim.object}{obj_anc[0]}" if obj_anc and obj_anc[0] != claim.object else claim.object
        
        return {
            "score": 0.3,
            "sources": ["MRHIER Lifting"],
            "evidence": [],
            "reason": f"Supported via MRHIER inheritance: {subj_info}, {obj_info}"
        }
        

print(" Cell 4 FIXED v2: UMLSOntology + SymbolicVerifier")
print("  Fixes applied:")
print("  1. get_cui/get_aui: Single-word entities now match (overlap >= 1)")
print("  2. get_mrhier_ancestors: NOW SEARCHES ALL AUIs FOR A CUI (critical!)")
print("  3. _try_mrhier_lift: Partial ancestors allowed")
print("  4. _layer_support_evidence: Partial text matches accepted")
print("  5. _decide_open_world: Better novel detection thresholds")

## Cell 5: Evidence Weighter

**What it does:** Reorders evidence based on verification results

**Why it matters:** Prioritizes evidence that supports accepted claims

**How it works:** Scores evidence nodes, sorts by score

In [ ]:
import logging
from typing import Dict, List, Any
from collections import defaultdict

class EvidenceWeighter:
    """
    Proof-guided reweighting:
      - boost nodes that appear in proofs of supported/novel claims
      - penalize nodes that appear in proofs of contradicted claims
      - do NOTHING for unsupported (open-world)
    """

    def reweight(self, evidence: Dict, verification: List["VerificationResult"]) -> Dict:
        # node_boost[(layer, node_id)] -> delta
        node_boost = defaultdict(float)

        for vr in verification:
            if not vr.hierarchical_proof:
                continue

            if vr.status in ("supported", "novel"):
                delta = 0.6
            elif vr.status == "contradicted":
                delta = -0.6
            else:
                delta = 0.0

            if delta == 0.0:
                continue

            for layer, ev_list in vr.hierarchical_proof.items():
                for ev in ev_list:
                    node_boost[(int(layer), str(ev.node_id))] += delta

        reweighted = {}

        for layer_key, items in evidence.items():
            try:
                layer = int(layer_key)
            except Exception:
                layer = layer_key

            scored = []
            for it in items:
                node_id = str(it.get("id", it.get("node_id", "unknown")))
                base = 1.0
                base += node_boost.get((layer if isinstance(layer, int) else int(layer), node_id), 0.0)
                scored.append((base, it))

            scored.sort(key=lambda x: x[0], reverse=True)
            reweighted[layer_key] = [it for _, it in scored]

        logging.info(f"Reweighted evidence using proof nodes across {len(reweighted)} layers")
        return reweighted

print(" Cell 5 updated: EvidenceWeighter now uses proof-based node boosts")


## Cell 6: Constrained Generator

**What it does:** Regenerates answer with symbolic constraints

**Why it matters:** Forces LLM to respect verification results

**How it works:** Builds prompt with must-include/must-not-include instructions

In [ ]:
"""
CELL 6 - FIXED VERSION
ConstrainedGenerator + AuditLogger

FIX APPLIED:
- AuditLogger.proof_stats(): MSL now defaults to -1 instead of None for cleaner aggregation
"""

import json
from typing import Dict, Any, List


class ConstrainedGenerator:
    """
    Optional: regeneration with open-world constraints.
    Use only if you later re-enable Step-6 regeneration.
    
    NO CHANGES - This class was working correctly.
    """

    def __init__(self, llm):
        self.llm = llm

    def generate(self, query: str, evidence: Dict, verification: List["VerificationResult"]) -> str:
        supported = [v.claim for v in verification if v.status == "supported"]
        novel = [v.claim for v in verification if v.status == "novel"]
        contradicted = [v.claim for v in verification if v.status == "contradicted"]
        unsupported = [v.claim for v in verification if v.status == "unsupported"]

        prompt_text = self._build_prompt(query, evidence, supported, novel, contradicted, unsupported)

        from langchain_core.prompts import PromptTemplate
        from langchain_classic.chains import LLMChain

        template = "You answer using evidence and follow constraints.\n\n{full_prompt}\n\nAnswer:"
        p = PromptTemplate(input_variables=["full_prompt"], template=template)
        chain = LLMChain(llm=self.llm, prompt=p)
        out = chain.invoke({"full_prompt": prompt_text})
        return out["text"].strip() if isinstance(out, dict) and "text" in out else str(out).strip()

    def _build_prompt(self, query: str, evidence: Dict,
                      supported: List["Claim"], novel: List["Claim"],
                      contradicted: List["Claim"], unsupported: List["Claim"]) -> str:
        ev = self._format_evidence(evidence)

        lines = []
        lines.append("Question: " + query)
        lines.append("\nEvidence:\n" + ev)

        if supported:
            lines.append("\nSUPPORTED (state confidently):")
            for c in supported[:4]:
                lines.append(" - " + f"{c.subject} {c.relation} {c.object}")

        if novel:
            lines.append("\nNOVEL (state cautiously, mention evidence limits):")
            for c in novel[:3]:
                lines.append(" - " + f"{c.subject} {c.relation} {c.object}")

        if unsupported:
            lines.append("\nUNSUPPORTED (do not assert as fact):")
            for c in unsupported[:2]:
                lines.append(" - " + f"{c.subject} {c.relation} {c.object}")

        if contradicted:
            lines.append("\nCONTRADICTED (do NOT claim):")
            for c in contradicted[:3]:
                lines.append(" - " + f"{c.subject} {c.relation} {c.object}")

        lines.append("\nWrite a short medical answer. Use cautious language for NOVEL/UNSUPPORTED.")
        return "\n".join(lines)

    def _format_evidence(self, evidence: Dict, max_items: int = 5) -> str:
        out = []
        for layer, items in evidence.items():
            out.append(f"\nLayer {layer}:")
            for it in items[:max_items]:
                content = it.get("content", "")
                out.append(" - " + str(content)[:200])
        return "\n".join(out)


class AuditLogger:
    """
    Audit trail aligned to open-world labels + proof objects.
    
    FIX: proof_stats now handles MSL more robustly
    """

    def build(self, query: str, evidence: Dict, draft: str,
              verification: List["VerificationResult"], final_answer: str) -> Dict[str, Any]:

        def proof_stats(vr: "VerificationResult") -> Dict[str, Any]:
            """
            Calculate proof statistics from hierarchical_proof.
            
            FIX: More robust handling of empty proofs and None values
            """
            # Safely get layers that have evidence
            layers = []
            if vr.hierarchical_proof:
                for k, v in vr.hierarchical_proof.items():
                    if v:  # Has evidence
                        try:
                            layers.append(int(k))
                        except (ValueError, TypeError):
                            pass
            
            layers = sorted(layers)
            
            # Calculate VCS (Verification Confidence Score)
            # Normalized by max_layers (default 4)
            vcs = len(layers) / 4.0 if layers else 0.0
            
            # Calculate MSL (Minimum Support Layer)
            # FIX: Use explicit None check, not -1, for cleaner downstream handling
            # But ensure it's properly handled in aggregation
            msl = min(layers) if layers else None
            
            return {
                "layers_supported": layers, 
                "VCS": round(vcs, 3),  # Round for cleaner output
                "MSL": msl
            }

        # Count status distribution
        summary = {"supported": 0, "novel": 0, "unsupported": 0, "contradicted": 0}
        details = []

        for vr in verification:
            # Safely increment status count
            status = vr.status if vr.status in summary else "unsupported"
            summary[status] = summary.get(status, 0) + 1
            
            stats = proof_stats(vr)

            # Build detail entry
            detail_entry = {
                "claim": f"{vr.claim.subject} {vr.claim.relation} {vr.claim.object}",
                "status": vr.status,
                "reason": vr.reason,
                "support_score": round(vr.support_score, 3) if vr.support_score else 0.0,
                "support_sources": vr.support_sources or [],
                "proof_stats": stats,
                "hierarchical_proof": {}
            }
            
            # Safely serialize hierarchical_proof
            if vr.hierarchical_proof:
                for layer, evs in vr.hierarchical_proof.items():
                    layer_key = str(layer)
                    detail_entry["hierarchical_proof"][layer_key] = []
                    
                    if evs:
                        for ev in evs:
                            detail_entry["hierarchical_proof"][layer_key].append({
                                "node_id": getattr(ev, 'node_id', 'unknown'),
                                "node_type": getattr(ev, 'node_type', 'unknown'),
                                "snippet": getattr(ev, 'snippet', '')[:200],  # Truncate for readability
                                "score": round(getattr(ev, 'score', 0.0), 3),
                                "meta": getattr(ev, 'meta', {})
                            })

            details.append(detail_entry)

        # Build final audit trail
        return {
            "query": query,
            "retrieval": {
                "layers_used": list(evidence.keys()),
                "total_items": sum(len(v) for v in evidence.values()),
                "sample_evidence": {
                    k: [str(i)[:120] + "..." for i in v[:2]] 
                    for k, v in evidence.items()
                }
            },
            "draft_answer": draft,
            "claims_extracted": len(verification),
            "verification_summary": summary,
            "verification_details": details,
            "final_answer": final_answer,
        }


print("✓ Cell 6 FIXED: ConstrainedGenerator + AuditLogger")
print("  Fixes applied:")
print("  1. proof_stats: More robust layer extraction")
print("  2. proof_stats: Proper rounding for cleaner output")
print("  3. Safer serialization of hierarchical_proof")

## Cell 7: Audit Logger

**What it does:** Builds explainability trace

**Why it matters:** Every decision is logged for transparency

**How it works:** Collects all intermediate steps into structured JSON

## Cell 8: Main NS-HAGRAG Runner

**What it does:** Orchestrates the complete neurosymbolic pipeline

**Why it matters:** Single wrapper that integrates with your existing code

**How it works:** Calls your existing retrieval, adds verification layer, regenerates

In [ ]:
# ============================================================
# NEUROSYMBOLIC HAGRAG RUNNER - FIXED VERSION v2
# ============================================================
import re

class NeuroSymbolicHAGRAGRunner:
    
    def __init__(self, query_engine, umls_ontology):
        self.query_engine = query_engine
        self.llm = query_engine.llm
        self.graph_store = query_engine.graph_store
        
        self.claim_extractor = TripletClaimExtractor(pipeline.triplet_extractor, max_paths_per_answer=8)
        self.verifier = SymbolicVerifier(self.graph_store, umls_ontology, enable_mrhier_lifting=True)
        self.weighter = EvidenceWeighter()
        self.audit_logger = AuditLogger()
        
        logging.info("NS-HAGRAG initialized with UMLS ontology")
        logging.info("  - Claim extraction: TripletExtractor-based")
        logging.info("  - Verification: Multi-source scoring + hierarchical proofs")
    
    def answer(self, query: str, k: int = 5) -> Dict[str, Any]:
        logging.info(f"Processing query: {query}")
        
        # STEP 1: Hierarchical Retrieval
        logging.info("Step 1/7: Hierarchical retrieval...")
        evidence = self.query_engine.retrieve_hierarchical_info(query, k=k)
        
        # STEP 2: Draft Generation  fixed prompt construction
        logging.info("Step 2/7: Generating draft answer...")

        def _format_evidence(ev: dict) -> str:
            parts = []
            # L0 entities first (most specific)
            for item in ev.get(0, []):
                c = item.get("content", {})
                name = c.get("name", "")
                desc = c.get("description", "")
                rels = c.get("relationships", [])
                rel_text = "; ".join(
                    f"{r.get('relation','')} {r.get('connected_entity','')}"
                    for r in rels if r.get("relation")
                )
                if name:
                    line = f"Entity: {name}"
                    if desc and desc != "No description":
                        line += f". {desc}"
                    if rel_text:
                        line += f". {rel_text}"
                    parts.append(line)
            # L1/L2 community summaries
            for layer in [1, 2]:
                for item in ev.get(layer, []):
                    summary = item.get("content", {}).get("summary", "")
                    if summary and len(summary.strip()) > 50:
                        parts.append(f"Background: {summary.strip()[:400]}")
            return "\n".join(parts) if parts else "No relevant evidence found."

        context = _format_evidence(evidence)

        prompt = (
            "You are a biomedical question answering assistant.\n"
            "Answer the following clinical question based on the evidence provided.\n"
            "Be direct and specific. Start your answer immediately.\n\n"
            f"Evidence:\n{context}\n\n"
            f"Question: {query}\n\n"
            "Answer:"
        )

        raw = self.llm.invoke(prompt)

        # Strip echoed prompt if model returns full text
        if "Answer:" in raw:
            draft = raw.split("Answer:")[-1].strip()
        else:
            draft = raw.strip()

        #  FIX: strip leading JSON artifacts from community summary leakage
        #draft = re.sub(r'^[\s\}\]\{\["\',:]+', '', draft).strip()
        # Replace the current artifact strip line with this stronger version
        draft = re.sub(r'^[\s\}\]\{\["\',:\-]+', '', draft).strip()
        # Also strip lines that are just separators like "---"
        draft = re.sub(r'^[-=\s]+\n', '', draft).strip()
        # STEP 3: Claim Extraction
        logging.info("Step 3/7: Extracting claims...")
        claims = self.claim_extractor.extract(draft)
        
        # STEP 4: Symbolic Verification
        logging.info("Step 4/7: Verifying claims...")
        verification = self.verifier.verify(claims, evidence)
        
        # STEP 5: Evidence Reweighting
        logging.info("Step 5/7: Reweighting evidence...")
        evidence_reweighted = self.weighter.reweight(evidence, verification)
        
        # STEP 6: Final Answer  no regeneration, preserves draft
        logging.info("Step 6/7: Preparing final answer...")
        final_answer = draft
        
        # STEP 7: Build Audit Trail
        logging.info("Step 7/7: Building audit trail...")
        audit = self.audit_logger.build(
            query=query,
            evidence=evidence,
            draft=draft,
            verification=verification,
            final_answer=final_answer
        )
        
        logging.info(" Processing complete")
        
        return {
            "response": final_answer,
            "draft": draft,
            "conflict_report": audit
        }

print(" NeuroSymbolicHAGRAGRunner defined")
print("  - 7-step pipeline")
print("  - Fixed prompt construction (no community template leakage)")
print("  - JSON artifact stripping")
print("  - Complete audit trail with hierarchical proofs")

## Cell 9: Integration Example

**This shows how to use NS-HAGRAG with your existing code**


In [ ]:
# ============================================================
# COMPLETE INTEGRATION - LOAD FROM CHECKPOINTS VERSION
# ============================================================

import os
import json
import logging

logging.basicConfig(level=logging.INFO)

# STEP 1: Load UMLS (do this ONCE)
umls_dir = "./data/umls/"
print("="*60)
print("LOADING UMLS ONTOLOGY")
print("="*60)

for f in ["MRCONSO.RRF", "MRSTY.RRF", "MRREL.RRF"]:
    exists = "✓" if os.path.exists(os.path.join(umls_dir, f)) else "✗"
    print(f"{exists} {f}")

print("\nLoading UMLS (takes ~2 minutes)...")
umls = UMLSOntology(umls_dir=umls_dir, load_mrhier=True, mrhier_sabs=("MSH", "RXNORM", "SNOMEDCT_US"), 
mrhier_max_rows=5000000)

test_cui = umls.get_cui("metformin")
test_semtype = umls.get_semantic_type("metformin")
print(f"\n✓ UMLS loaded: {len(umls.cui_to_str)} concepts")
print(f"Test: Metformin CUI={test_cui}, Type={test_semtype}")

# STEP 2: Initialize pipeline and LOAD FROM CHECKPOINTS
print("\n" + "="*60)
print("INITIALIZING PIPELINE FROM CHECKPOINTS")
print("="*60)

config = {
    "pdf_dir": "./data/pubmed_papers",
    "neo4j_uri": os.getenv("NEO4J_URI", "bolt://localhost:7687"),
    "neo4j_username": os.getenv("NEO4J_USERNAME", "neo4j"),
    "neo4j_password": os.getenv("NEO4J_PASSWORD"),
    "checkpoint_dir": "./checkpoints/hagragpipeline"  # CRITICAL: Same as baseline
}

pipeline = ArchRAGPipeline(**config)

# ============================================================
# 🆕 CRITICAL: LOAD FROM CHECKPOINTS INSTEAD OF REBUILDING
# ============================================================

print("\nLoading from existing checkpoints...")

# Step 2a: Load local checkpoints (embeddings, communities, CHNSW index)
loaded = pipeline.graph_store.load_local_checkpoints()

if loaded:
    print("✓ Loaded embeddings, communities, and CHNSW index from checkpoints")
else:
    print("⚠ Checkpoints not found - checking Neo4j...")

# Step 2b: Verify Neo4j has data, restore from backup if needed
neo4j_count = pipeline.graph_store.get_neo4j_entity_count()
print(f"Neo4j entity count: {neo4j_count}")

if neo4j_count == 0:
    print("⚠ Neo4j is empty - restoring from local backup...")
    restored = pipeline.graph_store.restore_neo4j_from_local()
    if restored:
        print("✓ Neo4j restored from backup")
        neo4j_count = pipeline.graph_store.get_neo4j_entity_count()
        print(f"Neo4j entity count after restore: {neo4j_count}")
    else:
        print("✗ ERROR: Could not restore Neo4j - need to rebuild pipeline")
        raise RuntimeError("No checkpoints or backups available")

# Step 2c: If checkpoints weren't loaded but Neo4j has data, rebuild communities only
if not loaded and neo4j_count > 0:
    print("\n⚠ Checkpoints missing but Neo4j has data")
    print("Regenerating embeddings and communities from Neo4j...")
    
    # Regenerate embeddings from Neo4j entities
    with pipeline.graph_store.driver.session() as session:
        result = session.run(
            "MATCH (e:Entity) RETURN e.name, e.type, e.description, e.attributes"
        )
        for record in result:
            entity_name = record["e.name"]
            entity_type = record["e.type"] or "Unknown"
            entity_desc = record["e.description"] or ""
            entity_attrs = record["e.attributes"] or "{}"
            
            entity_text = f"{entity_name} {entity_type} {entity_desc} {entity_attrs}"
            embedding = pipeline.graph_store.embedding_model.encode(entity_text)
            pipeline.graph_store.entity_embeddings[entity_name] = embedding
    
    print(f"✓ Generated {len(pipeline.graph_store.entity_embeddings)} embeddings")
    
    # Build communities
    print("Building hierarchical communities...")
    pipeline.graph_store.build_communities(pipeline.query_engine.llm)
    print("✓ Communities rebuilt and saved")

# Print final status
print("\n" + "-"*60)
print("CHECKPOINT LOAD SUMMARY:")
print("-"*60)
print(f"  Entities in Neo4j: {pipeline.graph_store.get_neo4j_entity_count()}")
print(f"  Embeddings loaded: {len(pipeline.graph_store.entity_embeddings)}")
print(f"  Community layers: {len(pipeline.graph_store.hierarchical_communities)}")
print(f"  CHNSW layers: {len(pipeline.graph_store.chnsw_index.layers)}")
print("-"*60)
print("✓ Pipeline ready (loaded from checkpoints)")

# STEP 3: Initialize NS-HAGRAG with UMLS
print("\n" + "="*60)
print("INITIALIZING NS-HAGRAG")
print("="*60)

ns_engine = NeuroSymbolicHAGRAGRunner(
    query_engine=pipeline.query_engine,
    umls_ontology=umls
)
print("✓ NS-HAGRAG ready")

# STEP 4: Test query
print("\n" + "="*60)
print("TESTING QUERY")
print("="*60)

result = ns_engine.answer("What causes diabetes?")

# STEP 5: Display results
print("\n📄 FINAL ANSWER:")
print("-"*60)
print(result["response"])

print("\n📝 DRAFT ANSWER:")
print("-"*60)
print(result["draft"][:300] + "...")

print("\n📊 VERIFICATION STATS:")
print("-"*60)
report = result["conflict_report"]
print(f"  Claims extracted: {report.get('claims_extracted', 'N/A')}")
summary = report.get('verification_summary', {})
print(f"  Supported: {summary.get('supported', 0)}")
print(f"  Novel: {summary.get('novel', 0)}")
print(f"  Unsupported: {summary.get('unsupported', 0)}")
print(f"  Contradicted: {summary.get('contradicted', 0)}")

print("\n" + "="*60)
print("✓ NS-HAGRAG TEST COMPLETE!")
print("="*60)

In [ ]:
# ============================================================
# CELL: RUN NS-HAGRAG EVALUATION WITH CHECKPOINTING
# ============================================================
# This cell:
#   ✅ Auto-resumes if interrupted
#   ✅ Saves after EACH question
#   ✅ Creates periodic backups
#   ✅ Exports to Excel when done
# ============================================================

import json
import logging
import os
import pickle
import shutil
from datetime import datetime
from pathlib import Path
from typing import Dict, Any, List, Optional
import time

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


# ============================================================
# EVALUATOR CLASS (with checkpointing)
# ============================================================

class NSHAGRAGEvaluator:
    """NS-HAGRAG Evaluator with checkpoint and resume capability"""
    
    def __init__(self, ns_engine, dataset_path: str, 
                 checkpoint_dir: str = './checkpoints/ns_hagrag_eval'):
        self.ns_engine = ns_engine
        self.dataset_path = dataset_path
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        
        # Checkpoint files
        self.progress_file = self.checkpoint_dir / 'eval_progress.json'
        self.results_file = self.checkpoint_dir / 'eval_results.jsonl'
        self.backup_dir = self.checkpoint_dir / 'backups'
        self.backup_dir.mkdir(parents=True, exist_ok=True)
        
        # Load dataset
        self.dataset = self._load_jsonl(dataset_path)
        logging.info(f"Loaded {len(self.dataset)} questions from {dataset_path}")
        
        # Load existing progress
        self.processed_indices = self._load_progress()
        self.results = self._load_results()
        
        if self.processed_indices:
            logging.info(f"✓ Resuming: {len(self.processed_indices)}/{len(self.dataset)} questions already processed")
    
    def _load_jsonl(self, path: str) -> List[Dict[str, Any]]:
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rows.append(json.loads(line))
        return rows
    
    def _load_progress(self) -> set:
        if self.progress_file.exists():
            with open(self.progress_file, 'r') as f:
                data = json.load(f)
                return set(data.get('processed_indices', []))
        return set()
    
    def _load_results(self) -> List[Dict[str, Any]]:
        results = []
        if self.results_file.exists():
            with open(self.results_file, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip():
                        results.append(json.loads(line))
        return results
    
    def _save_progress(self, idx: int, result: Dict[str, Any]):
        self.processed_indices.add(idx)
        
        # Append to results file
        with open(self.results_file, 'a', encoding='utf-8') as f:
            f.write(json.dumps(result, ensure_ascii=False, default=str) + '\n')
        
        # Update progress file
        with open(self.progress_file, 'w') as f:
            json.dump({
                'processed_indices': sorted(list(self.processed_indices)),
                'total_processed': len(self.processed_indices),
                'total_questions': len(self.dataset),
                'last_updated': datetime.utcnow().isoformat() + 'Z'
            }, f, indent=2)
    
    def _create_backup(self):
        if not self.results_file.exists():
            return
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        backup_path = self.backup_dir / f'backup_{timestamp}.jsonl'
        shutil.copy(self.results_file, backup_path)
        # Keep last 5 backups
        backups = sorted(self.backup_dir.glob('backup_*.jsonl'))
        for old in backups[:-5]:
            old.unlink()
        logging.info(f"✓ Backup saved: {backup_path.name}")
    
    def clear_progress(self):
        """Clear all progress and start fresh"""
        self._create_backup()  # Backup before clearing
        if self.progress_file.exists():
            self.progress_file.unlink()
        if self.results_file.exists():
            self.results_file.unlink()
        self.processed_indices = set()
        self.results = []
        logging.info("✓ Progress cleared")
    
    def get_status(self) -> Dict[str, Any]:
        """Get current status"""
        total = len(self.dataset)
        processed = len(self.processed_indices)
        return {
            "total": total,
            "processed": processed,
            "remaining": total - processed,
            "progress": f"{processed}/{total} ({processed/total*100:.1f}%)" if total > 0 else "0/0",
            "checkpoint_dir": str(self.checkpoint_dir)
        }
    
    def run(self, limit: Optional[int] = None, k: int = 5, backup_every: int = 10) -> Dict[str, Any]:
        """
        Run evaluation with checkpointing.
        
        Args:
            limit: Max questions (None = all)
            k: Retrieval depth
            backup_every: Create backup every N questions
        """
        data = self.dataset[:limit] if limit else self.dataset
        remaining = [i for i in range(len(data)) if i not in self.processed_indices]
        
        print(f"\n{'='*60}")
        print("NS-HAGRAG EVALUATION")
        print(f"{'='*60}")
        print(f"Total questions: {len(data)}")
        print(f"Already processed: {len(self.processed_indices)}")
        print(f"Remaining: {len(remaining)}")
        print(f"Checkpoint dir: {self.checkpoint_dir}")
        print(f"{'='*60}\n")
        
        if not remaining:
            print("✓ All questions already processed!")
            return self._generate_report(limit, k)
        
        start_time = time.time()
        session_count = 0
        
        for i in remaining:
            row = data[i]
            q = row.get("question") or row.get("query")
            gt = row.get("answer") or row.get("ground_truth") or ""
            
            if not q:
                continue
            
            q_start = time.time()
            
            try:
                out = self.ns_engine.answer(q, k=k)
                
                report = out.get("conflict_report", {})
                summary = report.get("verification_summary", {})
                vdetails = report.get("verification_details", [])
                
                # Calculate metrics
                claims = len(vdetails)
                supported = int(summary.get("supported", 0))
                val_rate = (supported / claims * 100) if claims > 0 else 0
                
                # VCS/MSL
                vcs_vals = [float(d.get("proof_stats", {}).get("VCS", 0)) for d in vdetails 
                           if isinstance(d.get("proof_stats", {}).get("VCS"), (int, float))]
                msl_vals = [d.get("proof_stats", {}).get("MSL") for d in vdetails 
                           if isinstance(d.get("proof_stats", {}).get("MSL"), int)]
                
                avg_vcs = sum(vcs_vals) / len(vcs_vals) if vcs_vals else 0
                avg_msl = sum(msl_vals) / len(msl_vals) if msl_vals else None
                
                result = {
                    "idx": i,
                    "question": q,
                    "gold_answer": gt,
                    "final_answer": out.get("response", ""),
                    "draft_answer": out.get("draft", ""),
                    "claims_extracted": claims,
                    "verification_summary": summary,
                    "validation_rate": val_rate,
                    "avg_VCS": avg_vcs,
                    "avg_MSL": avg_msl,
                    "latency_seconds": time.time() - q_start,
                    "timestamp": datetime.utcnow().isoformat() + 'Z'
                }
                
            except Exception as e:
                logging.error(f"Error on question {i}: {e}")
                result = {
                    "idx": i,
                    "question": q,
                    "error": str(e),
                    "timestamp": datetime.utcnow().isoformat() + 'Z'
                }
            
            # Save immediately
            self._save_progress(i, result)
            self.results.append(result)
            session_count += 1
            
            # Backup periodically
            if session_count % backup_every == 0:
                self._create_backup()
            
            # Progress logging
            elapsed = time.time() - start_time
            avg_time = elapsed / session_count
            eta = avg_time * (len(remaining) - session_count)
            
            claims_str = result.get("claims_extracted", 0)
            supported_str = result.get("verification_summary", {}).get("supported", 0)
            val_str = f"{result.get('validation_rate', 0):.1f}%"
            time_str = f"{result.get('latency_seconds', 0):.1f}s"
            eta_str = f"{eta/60:.1f}m" if eta > 60 else f"{eta:.0f}s"
            
            print(f"[{len(self.processed_indices)}/{len(data)}] Q{i}: "
                  f"claims={claims_str}, supported={supported_str}, "
                  f"val={val_str}, time={time_str}, ETA={eta_str}")
        
        # Final backup
        self._create_backup()
        
        return self._generate_report(limit, k)
    
    def _generate_report(self, limit, k) -> Dict[str, Any]:
        """Generate final report"""
        all_results = self._load_results()
        valid = [r for r in all_results if "error" not in r]
        errors = [r for r in all_results if "error" in r]
        
        # Aggregate
        agg = {"supported": 0, "novel": 0, "unsupported": 0, "contradicted": 0,
               "total_claims": 0, "total_val": 0, "total_vcs": 0, "total_latency": 0}
        
        for r in valid:
            s = r.get("verification_summary", {})
            agg["supported"] += int(s.get("supported", 0))
            agg["novel"] += int(s.get("novel", 0))
            agg["unsupported"] += int(s.get("unsupported", 0))
            agg["contradicted"] += int(s.get("contradicted", 0))
            agg["total_claims"] += r.get("claims_extracted", 0)
            agg["total_val"] += r.get("validation_rate", 0)
            agg["total_vcs"] += r.get("avg_VCS", 0)
            agg["total_latency"] += r.get("latency_seconds", 0)
        
        n = len(valid)
        stats = {
            "total_questions": len(all_results),
            "successful": n,
            "failed": len(errors),
            "total_claims": agg["total_claims"],
            "supported": agg["supported"],
            "novel": agg["novel"],
            "unsupported": agg["unsupported"],
            "contradicted": agg["contradicted"],
            "avg_validation_rate": agg["total_val"] / n if n > 0 else 0,
            "avg_VCS": agg["total_vcs"] / n if n > 0 else 0,
            "avg_latency": agg["total_latency"] / n if n > 0 else 0,
            "total_time_minutes": agg["total_latency"] / 60
        }
        
        report = {"statistics": stats, "results": valid, "errors": errors}
        
        # Save report
        with open(self.checkpoint_dir / 'final_report.json', 'w') as f:
            json.dump(report, f, indent=2, default=str)
        
        print(f"\n{'='*60}")
        print("EVALUATION COMPLETE!")
        print(f"{'='*60}")
        print(f"Questions: {stats['total_questions']} (✓{stats['successful']}, ✗{stats['failed']})")
        print(f"Claims: {stats['total_claims']} (supported={stats['supported']}, novel={stats['novel']})")
        print(f"Avg validation: {stats['avg_validation_rate']:.1f}%")
        print(f"Avg VCS: {stats['avg_VCS']:.3f}")
        print(f"Avg latency: {stats['avg_latency']:.1f}s")
        print(f"Total time: {stats['total_time_minutes']:.1f} minutes")
        print(f"{'='*60}\n")
        
        return report
    
    def export_to_excel(self, path: Optional[str] = None) -> str:
        """Export to Excel"""
        import pandas as pd
        
        results = self._load_results()
        rows = []
        for r in results:
            if "error" in r:
                rows.append({"idx": r["idx"], "question": r["question"], "error": r["error"]})
            else:
                s = r.get("verification_summary", {})
                rows.append({
                    "idx": r["idx"],
                    "question": r["question"][:200],
                    "final_answer": r.get("final_answer", "")[:500],
                    "claims": r.get("claims_extracted", 0),
                    "supported": s.get("supported", 0),
                    "novel": s.get("novel", 0),
                    "unsupported": s.get("unsupported", 0),
                    "contradicted": s.get("contradicted", 0),
                    "validation_rate": r.get("validation_rate", 0),
                    "avg_VCS": r.get("avg_VCS", 0),
                    "latency": r.get("latency_seconds", 0)
                })
        
        df = pd.DataFrame(rows)
        if not path:
            path = str(self.checkpoint_dir / f'results_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx')
        df.to_excel(path, index=False)
        print(f"✓ Exported: {path}")
        return path


# ============================================================
# RUN EVALUATION
# ============================================================

print("="*60)
print("INITIALIZING EVALUATOR")
print("="*60)

# Initialize evaluator
evaluator = NSHAGRAGEvaluator(
    ns_engine=ns_engine,  # Your NS-HAGRAG instance from previous cell
    dataset_path="./data/100diabetes_qa_dataset.jsonl",
    checkpoint_dir="./checkpoints/ns_hagrag_eval"
)

# Check current status
status = evaluator.get_status()
print(f"\nStatus: {status['progress']}")
print(f"Checkpoint dir: {status['checkpoint_dir']}\n")

# ============================================================
# OPTION 1: Run ALL remaining questions (resumes automatically)
# ============================================================
# print("Running evaluation (will auto-resume if interrupted)...")
# report = evaluator.run(limit=None, k=5, backup_every=10)

# ============================================================
# OPTION 2: Run just a few to test
# ============================================================
# evaluator.clear_progress()
# report = evaluator.run(limit=5, k=5)

# ============================================================
# OPTION 3: Clear progress and start fresh
# ============================================================
#evaluator.clear_progress()
report = evaluator.run(limit=100, k=5)

# ============================================================
# EXPORT TO EXCEL
# ============================================================
print("\nExporting to Excel...")
#excel_path = evaluator.export_to_excel()excel_path = evaluator.export_to_excel(f"./results/ns_hagrag_PMA_100_{datetime.now().strftime('%Y%m%d')}.xlsx")
excel_path = evaluator.export_to_excel(
    f"./results/ns_hagrag_PMA_100_{datetime.now().strftime('%Y%m%d')}.xlsx"
)
print("\n" + "="*60)
print("DONE!")
print(f"Results: {evaluator.checkpoint_dir / 'eval_results.jsonl'}")
print(f"Report: {evaluator.checkpoint_dir / 'final_report.json'}")
print(f"Excel: {excel_path}")
print("="*60)

## NS-HAGRAG System - Complete Integration

**What you have now:**
1.  **ClaimExtractor** - Extracts structured claims from text (regex + LLM hybrid)
2.  **SymbolicVerifier** - Validates claims using UMLS ontology + Neo4j KG
3.  **EvidenceWeighter** - Reweights evidence based on verification results
4.  **ConstrainedGenerator** - Regenerates answers with symbolic constraints
5.  **AuditLogger** - Builds complete explainability traces
6.  **NeuroSymbolicHAGRAGRunner** - Main orchestrator (neurosymbolic integration)
7.  **UMLSOntology** - NLM gold standard semantic validation

**How to use:**
```python
# Load UMLS ontology (once)
umls = UMLSOntology(umls_dir="./data/umls/")

# Initialize your pipeline (automatically loads saved state)
pipeline = ArchRAGPipeline(
    pdf_dir="./data/pubmed_papers",
    neo4j_uri=os.getenv("NEO4J_URI", "bolt://localhost:7687"),
    neo4j_username=os.getenv("NEO4J_USERNAME", "neo4j"),
    neo4j_password=os.getenv("NEO4J_PASSWORD")
)

# Add neurosymbolic layer with UMLS validation
ns_engine = NeuroSymbolicHAGRAGRunner(
    query_engine=pipeline.query_engine,
    umls_ontology=umls  #  UMLS integration
)

# Use it
result = ns_engine.answer("What causes diabetes?")
print(result["response"])
print(json.dumps(result["conflict_report"], indent=2))
```

**Contributions:**
-  **Neurosymbolic Integration** - Symbolic verification controls neural generation
-  **UMLS Validation** - NLM gold standard (1.2M+ medical concepts)
-  **Hierarchical Graph RAG** - Multi-layer evidence retrieval (L0 entities  L3 meta-communities)
-  **Explainability** - Complete audit trail with verification reasoning
-  **Conflict Resolution** - Claims validated against KG + ontology constraints
-  **Production Ready** - Persistent state, auto-loading, minimal complexity

**System automatically:**
- Loads 280 embeddings from cache
- Loads 4-layer hierarchical community structure
- Loads CHNSW index for fast retrieval
- No manual rebuilding needed!
## NS-HAGRAG System - Complete Integration

**What you have now:**
1.  **ClaimExtractor** - Extracts structured claims from text (regex + LLM hybrid)
2.  **SymbolicVerifier** - Validates claims using UMLS ontology + Neo4j KG
3.  **EvidenceWeighter** - Reweights evidence based on verification results
4.  **ConstrainedGenerator** - Regenerates answers with symbolic constraints
5.  **AuditLogger** - Builds complete explainability traces
6.  **NeuroSymbolicHAGRAGRunner** - Main orchestrator (neurosymbolic integration)
7.  **UMLSOntology** - NLM gold standard semantic validation

**How to use:**
```python
# Load UMLS ontology (once)
umls = UMLSOntology(umls_dir="./data/umls/")

# Initialize your pipeline (automatically loads saved state)
pipeline = ArchRAGPipeline(
    pdf_dir="./data/pubmed_papers",
    neo4j_uri=os.getenv("NEO4J_URI", "bolt://localhost:7687"),
    neo4j_username=os.getenv("NEO4J_USERNAME", "neo4j"),
    neo4j_password=os.getenv("NEO4J_PASSWORD")
)

# Add neurosymbolic layer with UMLS validation
ns_engine = NeuroSymbolicHAGRAGRunner(
    query_engine=pipeline.query_engine,
    umls_ontology=umls  #  UMLS integration
)

# Use it
result = ns_engine.answer("What causes diabetes?")
print(result["response"])
print(json.dumps(result["conflict_report"], indent=2))
```

**PhD contributions:**
-  **Neurosymbolic Integration** - Symbolic verification controls neural generation
-  **UMLS Validation** - NLM gold standard (1.2M+ medical concepts)
-  **Hierarchical Graph RAG** - Multi-layer evidence retrieval (L0 entities  L3 meta-communities)
-  **Explainability** - Complete audit trail with verification reasoning
-  **Conflict Resolution** - Claims validated against KG + ontology constraints
-  **Production Ready** - Persistent state, auto-loading, minimal complexity

**System automatically:**
- Loads 280 embeddings from cache
- Loads 4-layer hierarchical community structure
- Loads CHNSW index for fast retrieval
- No manual rebuilding needed!

In [ ]:
"""
NS-HAGRAG Evaluation on Diabetes QA Dataset
============================================

Evaluates neurosymbolic HAGRAG on 100 diabetes questions with:
- Quantitative metrics (ROUGE, BERTScore, claim validation, etc.)
- Qualitative analysis ready for human evaluation
- Excel export with complete results

"""

import json
import pandas as pd
from typing import Dict, List, Any
import time
from datetime import datetime
import numpy as np
from tqdm import tqdm

# For text similarity metrics
try:
    from rouge_score import rouge_scorer
    ROUGE_AVAILABLE = True
except:
    ROUGE_AVAILABLE = False
    print("   rouge-score not installed. Run: pip install rouge-score --break-system-packages")

try:
    from bert_score import score as bert_score
    BERTSCORE_AVAILABLE = True
except:
    BERTSCORE_AVAILABLE = False
    print("   bert-score not installed. Run: pip install bert-score --break-system-packages")


class NSHAGRAGEvaluator:
    """
    Comprehensive evaluator for NS-HAGRAG system.
    """
    
    def __init__(self, ns_engine, dataset_path: str):
        """
        Args:
            ns_engine: NeuroSymbolicHAGRAGRunner instance
            dataset_path: Path to JSONL dataset file
        """
        self.ns_engine = ns_engine
        self.dataset_path = dataset_path
        self.results = []
        
        # Initialize metrics
        if ROUGE_AVAILABLE:
            self.rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        
    def load_dataset(self) -> List[Dict]:
        """Load JSONL dataset."""
        data = []
        with open(self.dataset_path, 'r') as f:
            for line in f:
                data.append(json.loads(line))
        print(f" Loaded {len(data)} questions from dataset")
        return data
    
    def evaluate_single_query(self, question: str, ground_truth: str, context: str, 
                             source: str) -> Dict[str, Any]:
        """
        Evaluate NS-HAGRAG on a single question.
        
        Returns comprehensive metrics and outputs.
        """
        print(f"\n{'='*60}")
        print(f"Q: {question[:80]}...")
        print(f"{'='*60}")
        
        # Run NS-HAGRAG
        start_time = time.time()
        try:
            result = self.ns_engine.answer(question, k=5)
            latency = time.time() - start_time
            success = True
            error = None
        except Exception as e:
            latency = time.time() - start_time
            success = False
            error = str(e)
            result = {
                "response": "[ERROR]",
                "draft": "[ERROR]",
                "conflict_report": {
                    "claims_extracted": 0,
                    "verification_summary": {"accepted": 0, "rejected": 0, "qualified": 0}
                }
            }
        
        # Extract outputs
        final_answer = result["response"]
        draft_answer = result["draft"]
        report = result["conflict_report"]
        
        # Compute metrics
        metrics = self._compute_metrics(
            generated=final_answer,
            ground_truth=ground_truth,
            draft=draft_answer,
            report=report,
            latency=latency
        )
        
        # Build result record
        record = {
            # Input
            "question": question,
            "ground_truth": ground_truth,
            "context": context,
            "source": source,
            
            # Outputs
            "final_answer": final_answer,
            "draft_answer": draft_answer,
            
            # Metrics
            **metrics,
            
            # Neurosymbolic details
            "claims_extracted": report.get("claims_extracted", 0),
            "claims_accepted": report.get("verification_summary", {}).get("accepted", 0),
            "claims_rejected": report.get("verification_summary", {}).get("rejected", 0),
            "claims_qualified": report.get("verification_summary", {}).get("qualified", 0),
            "evidence_layers": len(report.get("retrieval", {}).get("layers_used", [])),
            "total_evidence_items": report.get("retrieval", {}).get("total_items", 0),
            
            # System
            "latency_seconds": latency,
            "success": success,
            "error": error,
            
            # Full report (as JSON string)
            "full_report": json.dumps(report, indent=2)
        }
        
        print(f" Claims: {record['claims_extracted']} | "
              f"Accepted: {record['claims_accepted']} | "
              f"Rejected: {record['claims_rejected']} | "
              f"Time: {latency:.2f}s")
        
        return record
    
    def _compute_metrics(self, generated: str, ground_truth: str, draft: str,
                         report: Dict, latency: float) -> Dict[str, float]:
        """Compute all quantitative metrics."""
        metrics = {}
        
        # 1. ROUGE scores (if available)
        if ROUGE_AVAILABLE:
            try:
                rouge_scores = self.rouge.score(ground_truth, generated)
                metrics["rouge1_f"] = rouge_scores['rouge1'].fmeasure
                metrics["rouge2_f"] = rouge_scores['rouge2'].fmeasure
                metrics["rougeL_f"] = rouge_scores['rougeL'].fmeasure
            except:
                metrics["rouge1_f"] = 0.0
                metrics["rouge2_f"] = 0.0
                metrics["rougeL_f"] = 0.0
        else:
            metrics["rouge1_f"] = None
            metrics["rouge2_f"] = None
            metrics["rougeL_f"] = None
        
        # 2. Length metrics
        metrics["answer_length"] = len(generated.split())
        metrics["draft_length"] = len(draft.split())
        metrics["gt_length"] = len(ground_truth.split())
        
        # 3. Claim validation rate
        claims_total = report.get("claims_extracted", 0)
        if claims_total > 0:
            verification = report.get("verification_summary", {})
            metrics["validation_rate"] = verification.get("accepted", 0) / claims_total
            metrics["rejection_rate"] = verification.get("rejected", 0) / claims_total
            metrics["qualification_rate"] = verification.get("qualified", 0) / claims_total
        else:
            metrics["validation_rate"] = 0.0
            metrics["rejection_rate"] = 0.0
            metrics["qualification_rate"] = 0.0
        
        # 4. Evidence utilization
        metrics["evidence_density"] = (
            report.get("retrieval", {}).get("total_items", 0) / max(1, metrics["answer_length"])
        )
        
        return metrics
    
    def evaluate_all(self, limit: int = None) -> pd.DataFrame:
        """
        Evaluate on entire dataset.
        
        Args:
            limit: Optional limit on number of questions (for testing)
            
        Returns:
            DataFrame with all results
        """
        dataset = self.load_dataset()
        
        if limit:
            dataset = dataset[:limit]
            print(f"   Limited to first {limit} questions for testing")
        
        print(f"\n= Starting evaluation on {len(dataset)} questions...")
        print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        
        # Process each question
        for i, item in enumerate(tqdm(dataset, desc="Evaluating")):
            record = self.evaluate_single_query(
                question=item["question"],
                ground_truth=item["answer"],
                context=item["context"],
                source=item["source"]
            )
            record["question_id"] = i + 1
            self.results.append(record)
        
        # Convert to DataFrame
        df = pd.DataFrame(self.results)
        
        # Reorder columns for readability
        col_order = [
            "question_id", "question", "ground_truth", "final_answer", "draft_answer",
            "rouge1_f", "rouge2_f", "rougeL_f",
            "claims_extracted", "claims_accepted", "claims_rejected", "claims_qualified",
            "validation_rate", "rejection_rate",
            "answer_length", "latency_seconds", "success", "source"
        ]
        df = df[[c for c in col_order if c in df.columns] + 
                [c for c in df.columns if c not in col_order]]
        
        print(f"\n Evaluation complete!")
        print(f"Finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        
        return df
    
    def compute_aggregate_metrics(self, df: pd.DataFrame) -> Dict[str, float]:
        """Compute aggregate statistics across all queries."""
        agg = {}
        
        # Success rate
        agg["success_rate"] = df["success"].mean()
        
        # ROUGE averages
        if "rouge1_f" in df.columns:
            agg["avg_rouge1"] = df["rouge1_f"].mean()
            agg["avg_rouge2"] = df["rouge2_f"].mean()
            agg["avg_rougeL"] = df["rougeL_f"].mean()
        
        # Claim statistics
        agg["avg_claims_extracted"] = df["claims_extracted"].mean()
        agg["avg_validation_rate"] = df["validation_rate"].mean()
        agg["avg_rejection_rate"] = df["rejection_rate"].mean()
        agg["total_claims_extracted"] = df["claims_extracted"].sum()
        agg["total_claims_accepted"] = df["claims_accepted"].sum()
        agg["total_claims_rejected"] = df["claims_rejected"].sum()
        
        # Performance
        agg["avg_latency"] = df["latency_seconds"].mean()
        agg["median_latency"] = df["latency_seconds"].median()
        agg["total_time_minutes"] = df["latency_seconds"].sum() / 60
        
        return agg
    
    def export_to_excel(self, df: pd.DataFrame, output_path: str):
        """
        Export results to Excel with multiple sheets.
        
        Sheets:
        1. Summary - Aggregate metrics
        2. All Results - Complete data
        3. Top Performing - Best ROUGE scores
        4. Failed Queries - Errors
        """
        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            # Sheet 1: Summary
            agg = self.compute_aggregate_metrics(df)
            summary_df = pd.DataFrame([agg]).T
            summary_df.columns = ["Value"]
            summary_df.to_excel(writer, sheet_name='Summary')
            
            # Sheet 2: All Results
            df.to_excel(writer, sheet_name='All Results', index=False)
            
            # Sheet 3: Top Performing (by ROUGE-L)
            if "rougeL_f" in df.columns:
                top_df = df.nlargest(20, "rougeL_f")[[
                    "question_id", "question", "final_answer", "ground_truth",
                    "rougeL_f", "claims_extracted", "validation_rate"
                ]]
                top_df.to_excel(writer, sheet_name='Top 20', index=False)
            
            # Sheet 4: Failed Queries
            failed_df = df[~df["success"]]
            if len(failed_df) > 0:
                failed_df.to_excel(writer, sheet_name='Failed', index=False)
        
        print(f" Results exported to: {output_path}")


# ============================================================
# HELPER: Print Summary Statistics
# ============================================================

def print_summary(df: pd.DataFrame):
    """Print human-readable summary."""
    print("\n" + "="*60)
    print("EVALUATION SUMMARY")
    print("="*60)
    
    print(f"\n=Ê Dataset:")
    print(f"  Total questions: {len(df)}")
    print(f"  Successful: {df['success'].sum()} ({df['success'].mean()*100:.1f}%)")
    
    if "rouge1_f" in df.columns and df["rouge1_f"].notna().any():
        print(f"\n=È Answer Quality (ROUGE):")
        print(f"  ROUGE-1: {df['rouge1_f'].mean():.3f}")
        print(f"  ROUGE-2: {df['rouge2_f'].mean():.3f}")
        print(f"  ROUGE-L: {df['rougeL_f'].mean():.3f}")
    
    print(f"\n=, Neurosymbolic Validation:")
    print(f"  Total claims extracted: {df['claims_extracted'].sum()}")
    print(f"  Avg claims per question: {df['claims_extracted'].mean():.1f}")
    print(f"  Validation rate: {df['validation_rate'].mean()*100:.1f}%")
    print(f"  Rejection rate: {df['rejection_rate'].mean()*100:.1f}%")
    print(f"  Claims accepted: {df['claims_accepted'].sum()}")
    print(f"  Claims rejected: {df['claims_rejected'].sum()}")
    
    print(f"\n¡ Performance:")
    print(f"  Avg latency: {df['latency_seconds'].mean():.2f}s")
    print(f"  Total time: {df['latency_seconds'].sum()/60:.1f} minutes")
    
    print("\n" + "="*60)


print(" NS-HAGRAG Evaluator loaded")

In [ ]:
# ============================================================
# PATCHES  run immediately after NSHAGRAGEvaluator class
# ============================================================
import types, json as _json, time as _time

# Initialize evaluator instance
evaluator = NSHAGRAGEvaluator(
    ns_engine=ns_engine,
    dataset_path="./data/100diabetes_qa_dataset.jsonl"
)

original_evaluate_all = evaluator.evaluate_all.__func__

def fixed_evaluate_all(self, limit=None):
    dataset = self.load_dataset()
    if limit:
        dataset = dataset[:limit]
    
    for i, item in enumerate(dataset):
        record = self.evaluate_single_query(
            question = item.get("question") or item.get("query", ""),
            ground_truth = item.get("answer") or item.get("ground_truth") or item.get("gold_answer", ""),
            context  = item.get("context", ""),
            source   = item.get("source", f"item_{i}")
        )
        record["question_id"] = i + 1
        self.results.append(record)
    
    import pandas as pd
    df = pd.DataFrame(self.results)
    return df

import types
evaluator.evaluate_all = types.MethodType(fixed_evaluate_all, evaluator)


original_compute = evaluator._compute_metrics.__func__

def fixed_compute_metrics(self, generated, ground_truth, draft, report, latency):
    metrics = {}

    if ROUGE_AVAILABLE:
        try:
            rs = self.rouge.score(ground_truth, generated)
            metrics["rouge1_f"] = rs['rouge1'].fmeasure
            metrics["rouge2_f"] = rs['rouge2'].fmeasure
            metrics["rougeL_f"] = rs['rougeL'].fmeasure
        except:
            metrics["rouge1_f"] = metrics["rouge2_f"] = metrics["rougeL_f"] = 0.0
    else:
        metrics["rouge1_f"] = metrics["rouge2_f"] = metrics["rougeL_f"] = None

    metrics["answer_length"] = len(generated.split())
    metrics["draft_length"]  = len(draft.split())
    metrics["gt_length"]     = len(ground_truth.split())

    claims_total = report.get("claims_extracted", 0)
    verification = report.get("verification_summary", {})
    if claims_total > 0:
        #  FIXED: use "supported"/"unsupported"/"novel" not "accepted"/"rejected"/"qualified"
        metrics["validation_rate"]    = verification.get("supported",   0) / claims_total
        metrics["rejection_rate"]     = verification.get("unsupported", 0) / claims_total
        metrics["qualification_rate"] = verification.get("novel",       0) / claims_total
    else:
        metrics["validation_rate"] = metrics["rejection_rate"] = metrics["qualification_rate"] = 0.0

    metrics["evidence_density"] = (
        report.get("retrieval", {}).get("total_items", 0) / max(1, metrics["answer_length"])
    )
    return metrics

evaluator._compute_metrics = types.MethodType(fixed_compute_metrics, evaluator)



original_single = evaluator.evaluate_single_query.__func__

def fixed_single_query(self, question, ground_truth, context, source):
    import time, json
    start_time = time.time()
    try:
        result  = self.ns_engine.answer(question, k=5)
        latency = time.time() - start_time
        success = True
        error   = None
    except Exception as e:
        latency = time.time() - start_time
        success = False
        error   = str(e)
        result  = {"response": "[ERROR]", "draft": "[ERROR]",
                   "conflict_report": {"claims_extracted": 0,
                                       "verification_summary": {}}}

    final_answer = result["response"]
    draft_answer = result["draft"]
    report       = result["conflict_report"]
    verification = report.get("verification_summary", {})

    metrics = self._compute_metrics(
        generated=final_answer, ground_truth=ground_truth,
        draft=draft_answer, report=report, latency=latency
    )

    record = {
        "question": question, "ground_truth": ground_truth,
        "context": context,   "source": source,
        "final_answer": final_answer, "draft_answer": draft_answer,
        **metrics,
        "claims_extracted": report.get("claims_extracted", 0),
        #  FIXED keys
        "claims_accepted":  verification.get("supported",   0),
        "claims_rejected":  verification.get("unsupported", 0),
        "claims_qualified": verification.get("novel",       0),
        "claims_contradicted": verification.get("contradicted", 0),
        "evidence_layers":      len(report.get("retrieval", {}).get("layers_used", [])),
        "total_evidence_items": report.get("retrieval", {}).get("total_items", 0),
        "latency_seconds": latency, "success": success, "error": error,
        "full_report": json.dumps(report, indent=2)
    }

    print(f"  Claims={record['claims_extracted']} | "
          f"Supported={record['claims_accepted']} | "
          f"Novel={record['claims_qualified']} | "
          f"Unsupported={record['claims_rejected']} | "
          f"Time={latency:.1f}s")
    return record

evaluator.evaluate_single_query = types.MethodType(fixed_single_query, evaluator)
evaluator.results = []  # reset so old results don't bleed in

print(" All 3 patches applied  ready to run")


In [ ]:
# ============================================================
# CELL: RUN NS-HAGRAG EVALUATION WITH CHECKPOINTING
# ============================================================
# This cell:
#   ✅ Auto-resumes if interrupted
#   ✅ Saves after EACH question
#   ✅ Creates periodic backups
#   ✅ Exports to Excel when done
# ============================================================

import json
import logging
import os
import pickle
import shutil
from datetime import datetime
from pathlib import Path
from typing import Dict, Any, List, Optional
import time

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


# ============================================================
# EVALUATOR CLASS (with checkpointing)
# ============================================================

class NSHAGRAGEvaluator:
    """NS-HAGRAG Evaluator with checkpoint and resume capability"""
    
    def __init__(self, ns_engine, dataset_path: str, 
                 checkpoint_dir: str = './checkpoints/ns_hagrag_eval'):
        self.ns_engine = ns_engine
        self.dataset_path = dataset_path
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        
        # Checkpoint files
        self.progress_file = self.checkpoint_dir / 'eval_progress.json'
        self.results_file = self.checkpoint_dir / 'eval_results.jsonl'
        self.backup_dir = self.checkpoint_dir / 'backups'
        self.backup_dir.mkdir(parents=True, exist_ok=True)
        
        # Load dataset
        self.dataset = self._load_jsonl(dataset_path)
        logging.info(f"Loaded {len(self.dataset)} questions from {dataset_path}")
        
        # Load existing progress
        self.processed_indices = self._load_progress()
        self.results = self._load_results()
        
        if self.processed_indices:
            logging.info(f"✓ Resuming: {len(self.processed_indices)}/{len(self.dataset)} questions already processed")
    
    def _load_jsonl(self, path: str) -> List[Dict[str, Any]]:
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rows.append(json.loads(line))
        return rows
    
    def _load_progress(self) -> set:
        if self.progress_file.exists():
            with open(self.progress_file, 'r') as f:
                data = json.load(f)
                return set(data.get('processed_indices', []))
        return set()
    
    def _load_results(self) -> List[Dict[str, Any]]:
        results = []
        if self.results_file.exists():
            with open(self.results_file, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip():
                        results.append(json.loads(line))
        return results
    
    def _save_progress(self, idx: int, result: Dict[str, Any]):
        self.processed_indices.add(idx)
        
        # Append to results file
        with open(self.results_file, 'a', encoding='utf-8') as f:
            f.write(json.dumps(result, ensure_ascii=False, default=str) + '\n')
        
        # Update progress file
        with open(self.progress_file, 'w') as f:
            json.dump({
                'processed_indices': sorted(list(self.processed_indices)),
                'total_processed': len(self.processed_indices),
                'total_questions': len(self.dataset),
                'last_updated': datetime.utcnow().isoformat() + 'Z'
            }, f, indent=2)
    
    def _create_backup(self):
        if not self.results_file.exists():
            return
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        backup_path = self.backup_dir / f'backup_{timestamp}.jsonl'
        shutil.copy(self.results_file, backup_path)
        # Keep last 5 backups
        backups = sorted(self.backup_dir.glob('backup_*.jsonl'))
        for old in backups[:-5]:
            old.unlink()
        logging.info(f"✓ Backup saved: {backup_path.name}")
    
    def clear_progress(self):
        """Clear all progress and start fresh"""
        self._create_backup()  # Backup before clearing
        if self.progress_file.exists():
            self.progress_file.unlink()
        if self.results_file.exists():
            self.results_file.unlink()
        self.processed_indices = set()
        self.results = []
        logging.info("✓ Progress cleared")
    
    def get_status(self) -> Dict[str, Any]:
        """Get current status"""
        total = len(self.dataset)
        processed = len(self.processed_indices)
        return {
            "total": total,
            "processed": processed,
            "remaining": total - processed,
            "progress": f"{processed}/{total} ({processed/total*100:.1f}%)" if total > 0 else "0/0",
            "checkpoint_dir": str(self.checkpoint_dir)
        }
    
    def run(self, limit: Optional[int] = None, k: int = 5, backup_every: int = 10) -> Dict[str, Any]:
        """
        Run evaluation with checkpointing.
        
        Args:
            limit: Max questions (None = all)
            k: Retrieval depth
            backup_every: Create backup every N questions
        """
        data = self.dataset[:limit] if limit else self.dataset
        remaining = [i for i in range(len(data)) if i not in self.processed_indices]
        
        print(f"\n{'='*60}")
        print("NS-HAGRAG EVALUATION")
        print(f"{'='*60}")
        print(f"Total questions: {len(data)}")
        print(f"Already processed: {len(self.processed_indices)}")
        print(f"Remaining: {len(remaining)}")
        print(f"Checkpoint dir: {self.checkpoint_dir}")
        print(f"{'='*60}\n")
        
        if not remaining:
            print("✓ All questions already processed!")
            return self._generate_report(limit, k)
        
        start_time = time.time()
        session_count = 0
        
        for i in remaining:
            row = data[i]
            q = row.get("question") or row.get("query")
            gt = row.get("answer") or row.get("ground_truth") or ""
            
            if not q:
                continue
            
            q_start = time.time()
            
            try:
                out = self.ns_engine.answer(q, k=k)
                
                report = out.get("conflict_report", {})
                summary = report.get("verification_summary", {})
                vdetails = report.get("verification_details", [])
                
                # Calculate metrics
                claims = len(vdetails)
                supported = int(summary.get("supported", 0))
                val_rate = (supported / claims * 100) if claims > 0 else 0
                
                # VCS/MSL
                vcs_vals = [float(d.get("proof_stats", {}).get("VCS", 0)) for d in vdetails 
                           if isinstance(d.get("proof_stats", {}).get("VCS"), (int, float))]
                msl_vals = [d.get("proof_stats", {}).get("MSL") for d in vdetails 
                           if isinstance(d.get("proof_stats", {}).get("MSL"), int)]
                
                avg_vcs = sum(vcs_vals) / len(vcs_vals) if vcs_vals else 0
                avg_msl = sum(msl_vals) / len(msl_vals) if msl_vals else None
                
                result = {
                    "idx": i,
                    "question": q,
                    "gold_answer": gt,
                    "final_answer": out.get("response", ""),
                    "draft_answer": out.get("draft", ""),
                    "claims_extracted": claims,
                    "verification_summary": summary,
                    "validation_rate": val_rate,
                    "avg_VCS": avg_vcs,
                    "avg_MSL": avg_msl,
                    "latency_seconds": time.time() - q_start,
                    "timestamp": datetime.utcnow().isoformat() + 'Z'
                }
                
            except Exception as e:
                logging.error(f"Error on question {i}: {e}")
                result = {
                    "idx": i,
                    "question": q,
                    "error": str(e),
                    "timestamp": datetime.utcnow().isoformat() + 'Z'
                }
            
            # Save immediately
            self._save_progress(i, result)
            self.results.append(result)
            session_count += 1
            
            # Backup periodically
            if session_count % backup_every == 0:
                self._create_backup()
            
            # Progress logging
            elapsed = time.time() - start_time
            avg_time = elapsed / session_count
            eta = avg_time * (len(remaining) - session_count)
            
            claims_str = result.get("claims_extracted", 0)
            supported_str = result.get("verification_summary", {}).get("supported", 0)
            val_str = f"{result.get('validation_rate', 0):.1f}%"
            time_str = f"{result.get('latency_seconds', 0):.1f}s"
            eta_str = f"{eta/60:.1f}m" if eta > 60 else f"{eta:.0f}s"
            
            print(f"[{len(self.processed_indices)}/{len(data)}] Q{i}: "
                  f"claims={claims_str}, supported={supported_str}, "
                  f"val={val_str}, time={time_str}, ETA={eta_str}")
        
        # Final backup
        self._create_backup()
        
        return self._generate_report(limit, k)
    
    def _generate_report(self, limit, k) -> Dict[str, Any]:
        """Generate final report"""
        all_results = self._load_results()
        valid = [r for r in all_results if "error" not in r]
        errors = [r for r in all_results if "error" in r]
        
        # Aggregate
        agg = {"supported": 0, "novel": 0, "unsupported": 0, "contradicted": 0,
               "total_claims": 0, "total_val": 0, "total_vcs": 0, "total_latency": 0}
        
        for r in valid:
            s = r.get("verification_summary", {})
            agg["supported"] += int(s.get("supported", 0))
            agg["novel"] += int(s.get("novel", 0))
            agg["unsupported"] += int(s.get("unsupported", 0))
            agg["contradicted"] += int(s.get("contradicted", 0))
            agg["total_claims"] += r.get("claims_extracted", 0)
            agg["total_val"] += r.get("validation_rate", 0)
            agg["total_vcs"] += r.get("avg_VCS", 0)
            agg["total_latency"] += r.get("latency_seconds", 0)
        
        n = len(valid)
        stats = {
            "total_questions": len(all_results),
            "successful": n,
            "failed": len(errors),
            "total_claims": agg["total_claims"],
            "supported": agg["supported"],
            "novel": agg["novel"],
            "unsupported": agg["unsupported"],
            "contradicted": agg["contradicted"],
            "avg_validation_rate": agg["total_val"] / n if n > 0 else 0,
            "avg_VCS": agg["total_vcs"] / n if n > 0 else 0,
            "avg_latency": agg["total_latency"] / n if n > 0 else 0,
            "total_time_minutes": agg["total_latency"] / 60
        }
        
        report = {"statistics": stats, "results": valid, "errors": errors}
        
        # Save report
        with open(self.checkpoint_dir / 'final_report.json', 'w') as f:
            json.dump(report, f, indent=2, default=str)
        
        print(f"\n{'='*60}")
        print("EVALUATION COMPLETE!")
        print(f"{'='*60}")
        print(f"Questions: {stats['total_questions']} (✓{stats['successful']}, ✗{stats['failed']})")
        print(f"Claims: {stats['total_claims']} (supported={stats['supported']}, novel={stats['novel']})")
        print(f"Avg validation: {stats['avg_validation_rate']:.1f}%")
        print(f"Avg VCS: {stats['avg_VCS']:.3f}")
        print(f"Avg latency: {stats['avg_latency']:.1f}s")
        print(f"Total time: {stats['total_time_minutes']:.1f} minutes")
        print(f"{'='*60}\n")
        
        return report
    
    def export_to_excel(self, path: Optional[str] = None) -> str:
        """Export to Excel"""
        import pandas as pd
        
        results = self._load_results()
        rows = []
        for r in results:
            if "error" in r:
                rows.append({"idx": r["idx"], "question": r["question"], "error": r["error"]})
            else:
                s = r.get("verification_summary", {})
                rows.append({
                    "idx": r["idx"],
                    "question": r["question"][:200],
                    "final_answer": r.get("final_answer", "")[:500],
                    "claims": r.get("claims_extracted", 0),
                    "supported": s.get("supported", 0),
                    "novel": s.get("novel", 0),
                    "unsupported": s.get("unsupported", 0),
                    "contradicted": s.get("contradicted", 0),
                    "validation_rate": r.get("validation_rate", 0),
                    "avg_VCS": r.get("avg_VCS", 0),
                    "latency": r.get("latency_seconds", 0)
                })
        
        df = pd.DataFrame(rows)
        if not path:
            path = str(self.checkpoint_dir / f'results_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx')
        df.to_excel(path, index=False)
        print(f"✓ Exported: {path}")
        return path


# ============================================================
# RUN EVALUATION
# ============================================================

print("="*60)
print("INITIALIZING EVALUATOR")
print("="*60)

# Initialize evaluator
evaluator = NSHAGRAGEvaluator(
    ns_engine=ns_engine,  # Your NS-HAGRAG instance from previous cell
    dataset_path="./data/100diabetes_qa_dataset.jsonl",
    checkpoint_dir="./checkpoints/ns_hagrag_eval"
)

# Check current status
status = evaluator.get_status()
print(f"\nStatus: {status['progress']}")
print(f"Checkpoint dir: {status['checkpoint_dir']}\n")

# ============================================================
# OPTION 1: Run ALL remaining questions (resumes automatically)
# ============================================================
# print("Running evaluation (will auto-resume if interrupted)...")
# report = evaluator.run(limit=None, k=5, backup_every=10)

# ============================================================
# OPTION 2: Run just a few to test
# ============================================================
# evaluator.clear_progress()
# report = evaluator.run(limit=5, k=5)

# ============================================================
# OPTION 3: Clear progress and start fresh
# ============================================================
#evaluator.clear_progress()
report = evaluator.run(limit=100, k=5)

# ============================================================
# EXPORT TO EXCEL
# ============================================================
print("\nExporting to Excel...")
#excel_path = evaluator.export_to_excel()excel_path = evaluator.export_to_excel(f"./results/ns_hagrag_PMA_100_{datetime.now().strftime('%Y%m%d')}.xlsx")
excel_path = evaluator.export_to_excel(
    f"./results/ns_hagrag_PMA_100_{datetime.now().strftime('%Y%m%d')}.xlsx"
)
print("\n" + "="*60)
print("DONE!")
print(f"Results: {evaluator.checkpoint_dir / 'eval_results.jsonl'}")
print(f"Report: {evaluator.checkpoint_dir / 'final_report.json'}")
print(f"Excel: {excel_path}")
print("="*60)

In [ ]:
# ============================================================
# RUN COMPLETE NS-HAGRAG EVALUATION  100 DIABETES QUESTIONS
# ============================================================
import sys
import subprocess
from datetime import datetime

# STEP 1: Dependencies
print("Checking dependencies...")
try:
    import rouge_score
    print("rouge-score installed")
except:
    subprocess.run(
        ["pip", "install", "rouge-score",
         "--break-system-packages"],
        capture_output=True
    )
    print("rouge-score installed")

# STEP 2: Checkpoint evaluator
# NOTE: patches already applied above  do NOT re-create
# the comprehensive evaluator here, use checkpoint version
ck_evaluator = NSHAGRAGEvaluator(
    ns_engine=ns_engine,
    dataset_path=(
        "./"
        "100diabetes_qa_dataset.jsonl"
    ),
    checkpoint_dir=(
        "./"
        "checkpoints/ns_hagrag_eval_fixed"
    )
)

status = ck_evaluator.get_status()
print(f"\nStatus: {status['progress']}")

# STEP 3: Run evaluation
# If starting fresh: uncomment clear_progress()
# If resuming interrupted run: leave it commented
# ck_evaluator.clear_progress()
print("\nRunning evaluation (auto-resumes if interrupted)...")
report = ck_evaluator.run(
    limit=100, k=5, backup_every=10
)

# STEP 4: Export to Excel
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = (
    "./results/"
    f"ns_hagrag_evaluation_PMA100_{ts}.xlsx"
)
ck_evaluator.export_to_excel(output_path)
print(f"\nResults saved to: {output_path}")

# STEP 5: Display sample results
print("\nSample Results (First 3 Questions):")
print("="*60)
stats = report['statistics']
print(f"Questions: {stats['total_questions']}")
print(f"Supported: {stats['supported']}")
print(f"Novel:     {stats['novel']}")
print(f"Avg validation: {stats['avg_validation_rate']:.1f}%")
print(f"Avg VCS:        {stats['avg_VCS']:.3f}")
print(f"Avg latency:    {stats['avg_latency']:.1f}s")
print("="*60)
for r in report['results'][:3]:
    print(f"\nQ: {r['question'][:70]}")
    print(f"A: {r['final_answer'][:200]}")
    vs = r['verification_summary']
    print(f"Supported={vs.get('supported',0)} "
          f"Novel={vs.get('novel',0)} "
          f"Val={r.get('validation_rate',0):.1f}%")
print("\n" + "="*60)
print("EVALUATION COMPLETE!")
print("="*60)

For answering research questions

In [ ]:
# ============================================================
# RQ2: PRODUCTION SYMBOLIC ATTRIBUTION WITH CHECKPOINTS
# ============================================================

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import json
import logging
import pickle
import os
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, asdict
from collections import defaultdict
from pathlib import Path
from datetime import datetime

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


# ============================================================
# 1. CHECKPOINT MANAGER
# ============================================================

class RQ2Checkpoint:
    """Manage checkpoints for resumable RQ2 experiments."""
    
    def __init__(self, checkpoint_dir='./rq2_checkpoints'):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(exist_ok=True)
        self.items_file = self.checkpoint_dir / 'items.pkl'
        self.progress_file = self.checkpoint_dir / 'progress.json'
    
    def save(self, items: List, query_idx: int, total_queries: int):
        """Save progress checkpoint."""
        # Save items (pickle for RetrievalItem objects)
        with open(self.items_file, 'wb') as f:
            pickle.dump(items, f)
        
        # Save metadata
        progress = {
            'query_idx': query_idx,
            'total_queries': total_queries,
            'n_items': len(items),
            'timestamp': str(datetime.now())
        }
        with open(self.progress_file, 'w') as f:
            json.dump(progress, f, indent=2)
        
        logger.info(f"=¾ Checkpoint saved: {query_idx}/{total_queries} queries, {len(items)} items")
    
    def load(self) -> Tuple[List, int]:
        """Load checkpoint if exists."""
        if not self.exists():
            return [], 0
        
        # Load items
        with open(self.items_file, 'rb') as f:
            items = pickle.load(f)
        
        # Load metadata
        with open(self.progress_file, 'r') as f:
            progress = json.load(f)
        
        logger.info(f"=Â Resuming from checkpoint:")
        logger.info(f"   Query: {progress['query_idx']}/{progress['total_queries']}")
        logger.info(f"   Items: {len(items)}")
        logger.info(f"   Saved: {progress['timestamp']}")
        
        return items, progress['query_idx']
    
    def exists(self) -> bool:
        """Check if checkpoint exists."""
        return self.items_file.exists() and self.progress_file.exists()
    
    def clear(self):
        """Delete checkpoints (start fresh)."""
        if self.items_file.exists():
            self.items_file.unlink()
        if self.progress_file.exists():
            self.progress_file.unlink()
        logger.info("=Ñ  Checkpoints cleared - starting fresh")


# ============================================================
# 2. DATA STRUCTURE
# ============================================================

@dataclass
class RetrievalItem:
    """Stores one retrieved node with all features."""
    node_id: str
    layer: int
    neural_score: float
    rank: int
    query: str
    has_umls_cui: float = 0.0
    n_semantic_types: float = 0.0
    is_drug: float = 0.0
    is_disease: float = 0.0
    is_procedure: float = 0.0
    kg_edge_count: float = 0.0
    has_kg_edge: float = 0.0
    layer_priority: float = 0.0
    is_entity: float = 0.0
    is_community: float = 0.0
    verification_count: float = 0.0


# ============================================================
# 3. FEATURE EXTRACTOR WITH CHECKPOINTING
# ============================================================

class RealFeatureExtractor:
    """Extract features with checkpoint support."""
    
    def __init__(self, ns_engine):
        self.ns_engine = ns_engine
        self.graph_store = ns_engine.graph_store
        self.umls = ns_engine.verifier.umls
        self.embedding_model = ns_engine.query_engine.embedding_model
        self.verification_history = defaultdict(int)
        
        logger.info("RealFeatureExtractor initialized")
        logger.info(f"  Graph store: {type(self.graph_store).__name__}")
        logger.info(f"  UMLS loaded: {self.umls is not None}")
        logger.info(f"  Embedding model: {type(self.embedding_model).__name__}")
    
    def get_real_neural_scores(self, query: str, k: int = 5) -> Dict[int, List[Tuple[str, float]]]:
        """Get ACTUAL neural scores from C-HNSW."""
        query_embedding = self.embedding_model.encode(query, convert_to_numpy=True)
        search_results = self.graph_store.chnsw_index.hierarchical_search(query_embedding, k=k)
        
        scores_by_layer = {}
        for layer, node_ids in search_results.items():
            layer_scores = []
            for node_id in node_ids:
                if node_id in self.graph_store.chnsw_index.node_embeddings:
                    node_emb = self.graph_store.chnsw_index.node_embeddings[node_id]
                    cosine_sim = np.dot(query_embedding, node_emb) / (
                        np.linalg.norm(query_embedding) * np.linalg.norm(node_emb)
                    )
                    layer_scores.append((node_id, float(cosine_sim)))
                else:
                    rank = node_ids.index(node_id) + 1
                    layer_scores.append((node_id, 1.0 / rank))
            
            layer_scores.sort(key=lambda x: x[1], reverse=True)
            scores_by_layer[layer] = layer_scores[:k]
            logger.info(f"Layer {layer}: {len(layer_scores)} items, "
                       f"score range [{min(s for _, s in layer_scores):.3f}, "
                       f"{max(s for _, s in layer_scores):.3f}]")
        
        return scores_by_layer
    
    def extract_symbolic_features(self, node_id: str, layer: int, query_entities: List[str]) -> Dict[str, float]:
        """Extract symbolic features from UMLS and Neo4j."""
        features = {}
        node_data = self.graph_store.chnsw_index.node_data.get(node_id, {})
        
        # UMLS Features
        members = node_data.get('members', []) or [node_id]
        all_semantic_types = []
        found_cui = False
        
        for member in members[:10]:
            cui = self.umls.get_cui(member)
            if cui:
                found_cui = True
                sem_types = self.umls.get_semantic_type(member, cui)
                if sem_types:
                    all_semantic_types.extend(sem_types)
        
        features['has_umls_cui'] = 1.0 if found_cui else 0.0
        features['n_semantic_types'] = float(len(set(all_semantic_types)))
        
        type_str = ' '.join(all_semantic_types).lower() if all_semantic_types else ''
        features['is_drug'] = 1.0 if any(t in type_str for t in ['pharmacologic', 'substance', 'antibiotic']) else 0.0
        features['is_disease'] = 1.0 if any(t in type_str for t in ['disease', 'disorder', 'syndrome']) else 0.0
        features['is_procedure'] = 1.0 if any(t in type_str for t in ['procedure', 'therapeutic']) else 0.0
        
        # Neo4j Features (skip to speed up - it's timing out anyway)
        features['kg_edge_count'] = 0.0
        features['has_kg_edge'] = 0.0
        
        # Hierarchical Features
        features['layer_priority'] = 3.0 - float(layer)
        features['is_entity'] = 1.0 if layer == 0 else 0.0
        features['is_community'] = 1.0 if layer >= 1 else 0.0
        features['verification_count'] = float(self.verification_history.get(node_id, 0))
        
        return features
    
    def collect_data_from_dataset(
        self,
        dataset_path: str,
        k: int = 5,
        limit: Optional[int] = None,
        resume: bool = True,
        checkpoint_every: int = 5
    ) -> List[RetrievalItem]:
        """
        Collect data with checkpoint support.
        
        Args:
            dataset_path: Path to dataset
            k: Items per layer
            limit: Max queries
            resume: If True, resume from checkpoint
            checkpoint_every: Save checkpoint every N queries
        """
        checkpoint = RQ2Checkpoint()
        
        # Load checkpoint if resuming
        if resume and checkpoint.exists():
            all_items, start_idx = checkpoint.load()
        else:
            all_items, start_idx = [], 0
        
        # Load queries
        queries = []
        with open(dataset_path, 'r') as f:
            for line in f:
                data = json.loads(line)
                queries.append(data['question'])
                if limit and len(queries) >= limit:
                    break
        
        total_queries = len(queries)
        logger.info(f"Loaded {total_queries} queries (starting from {start_idx})")
        
        # Process queries
        for q_idx in range(start_idx, total_queries):
            query = queries[q_idx]
            logger.info(f"[{q_idx+1}/{total_queries}] Processing: {query[:60]}...")
            
            try:
                query_entities = [w for w in query.split() 
                                 if len(w) > 3 and w.lower() not in 
                                 ['what', 'when', 'where', 'which', 'does', 'the']]
                
                neural_scores = self.get_real_neural_scores(query, k=k)
                
                for layer, node_score_pairs in neural_scores.items():
                    for rank, (node_id, neural_score) in enumerate(node_score_pairs, 1):
                        features = self.extract_symbolic_features(node_id, layer, query_entities)
                        features.pop('layer', None)
                        
                        item = RetrievalItem(
                            node_id=node_id,
                            layer=layer,
                            neural_score=neural_score,
                            rank=rank,
                            query=query,
                            **features
                        )
                        all_items.append(item)
                
                # Save checkpoint periodically
                if (q_idx + 1) % checkpoint_every == 0:
                    checkpoint.save(all_items, q_idx + 1, total_queries)
            
            except Exception as e:
                logger.error(f"Error processing query: {e}")
                continue
        
        # Final save
        checkpoint.save(all_items, total_queries, total_queries)
        logger.info(f" Collected {len(all_items)} items from YOUR dataset")
        
        return all_items


# ============================================================
# 4. ATTRIBUTION MODEL
# ============================================================

class AttributionModel:
    """Train interpretable model."""
    
    def __init__(self, model_type='decision_tree'):
        if model_type == 'decision_tree':
            self.model = DecisionTreeRegressor(max_depth=6, min_samples_leaf=10, random_state=42)
        elif model_type == 'linear':
            self.model = LinearRegression()
        elif model_type == 'random_forest':
            self.model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
        
        self.model_type = model_type
        self.feature_names = None
        self.fitted = False
    
    def prepare_data(self, items: List[RetrievalItem]):
        exclude = {'node_id', 'neural_score', 'rank', 'query'}
        sample_dict = asdict(items[0])
        self.feature_names = [k for k in sample_dict.keys() if k not in exclude]
        
        X, y = [], []
        for item in items:
            item_dict = asdict(item)
            X.append([item_dict[f] for f in self.feature_names])
            y.append(item.neural_score)
        
        return np.array(X), np.array(y)
    
    def train_and_evaluate(self, items: List[RetrievalItem], test_size=0.2):
        X, y = self.prepare_data(items)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
        
        self.model.fit(X_train, y_train)
        self.fitted = True
        
        y_pred = self.model.predict(X_test)
        test_r2 = r2_score(y_test, y_pred)
        test_spearman = spearmanr(y_test, y_pred)[0]
        
        if hasattr(self.model, 'feature_importances_'):
            importances = self.model.feature_importances_
        elif hasattr(self.model, 'coef_'):
            importances = np.abs(self.model.coef_)
        else:
            importances = np.zeros(len(self.feature_names))
        
        feature_importance = sorted(zip(self.feature_names, importances), key=lambda x: x[1], reverse=True)
        
        logger.info(f"\n{'='*60}")
        logger.info(f"Model: {self.model_type}")
        logger.info(f"Test R²: {test_r2:.4f} (Target: e0.60)")
        logger.info(f"Test Spearman: {test_spearman:.4f}")
        logger.info(f"\nTop 5 Features:")
        for fname, imp in feature_importance[:5]:
            logger.info(f"  {fname}: {imp:.4f}")
        logger.info(f"{'='*60}")
        
        return {
            'test_r2': test_r2,
            'test_spearman': test_spearman,
            'feature_importance': feature_importance,
            'n_test': len(X_test)
        }


# ============================================================
# 5. MAIN RUNNER WITH RESUME
# ============================================================

def run_rq2_on_your_data(
    ns_engine,
    dataset_path: str = "./data/100diabetes_qa_dataset.jsonl",
    k: int = 5,
    limit: Optional[int] = None,
    test_size: float = 0.2,
    resume: bool = True  # Auto-resume by default
) -> Dict:
    """
    Run RQ2 with checkpoint support.
    
    Args:
        ns_engine: Your NeuroSymbolicHAGRAGRunner
        dataset_path: Dataset path
        k: Items per layer
        limit: Max queries (None = all)
        test_size: Test split
        resume: If True, resume from checkpoint; if False, start fresh
    """
    logger.info("="*80)
    logger.info("RQ2: SYMBOLIC ATTRIBUTION WITH CHECKPOINTS")
    logger.info("="*80)
    
    # Phase 1: Collect Data (with resume)
    logger.info(f"\n=Ê Phase 1: Collecting data (resume={resume})...")
    
    extractor = RealFeatureExtractor(ns_engine)
    items = extractor.collect_data_from_dataset(
        dataset_path, k=k, limit=limit, resume=resume, checkpoint_every=5
    )
    
    if len(items) < 50:
        logger.warning(f"  Only {len(items)} items. Need 50+ for robust evaluation.")
    
    # Save raw data
    with open('rq2_your_data.json', 'w') as f:
        json.dump([asdict(item) for item in items], f, indent=2)
    logger.info(f" Saved: rq2_your_data.json ({len(items)} items)")
    
    # Phase 2: Train Models
    logger.info("\n> Phase 2: Training models...")
    
    results = {}
    for model_type in ['decision_tree', 'linear', 'random_forest']:
        logger.info(f"\nTraining {model_type}...")
        model = AttributionModel(model_type=model_type)
        result = model.train_and_evaluate(items, test_size=test_size)
        results[model_type] = result
    
    # Phase 3: Select Best
    best_type = max(results.keys(), key=lambda k: results[k]['test_r2'])
    best_result = results[best_type]
    
    logger.info(f"\n<Æ Best Model: {best_type} (R²={best_result['test_r2']:.4f})")
    
    # Phase 4: Save Results
    final_results = {
        'dataset': dataset_path,
        'n_queries': limit or 102,
        'n_items': len(items),
        'best_model': {
            'type': best_type,
            'test_r2': best_result['test_r2'],
            'test_spearman': best_result['test_spearman'],
            'top_10_features': best_result['feature_importance'][:10]
        },
        'all_models': results
    }
    
    with open('rq2_results.json', 'w') as f:
        json.dump(final_results, f, indent=2)
    logger.info(f" Saved: rq2_results.json")
    
    # Phase 5: Visualize
    logger.info("\n=Ê Phase 5: Creating plots...")
    create_plots(items, best_result)
    
    logger.info("\n" + "="*80)
    logger.info("RQ2 COMPLETE")
    logger.info("="*80)
    logger.info(f"Test R²: {best_result['test_r2']:.4f} {'' if best_result['test_r2'] >= 0.6 else ' '}")
    logger.info("\nFiles:")
    logger.info(" rq2_your_data.json")
    logger.info(" rq2_results.json")
    logger.info("  rq2_features.png")
    logger.info("="*80)
    
    return final_results


def create_plots(items: List[RetrievalItem], best_result: Dict):
    """Create plots."""
    fig, ax = plt.subplots(figsize=(10, 8))
    top_features = best_result['feature_importance'][:15]
    names, importances = zip(*top_features)
    colors = ['#27ae60' if imp > 0.1 else '#3498db' if imp > 0.05 else '#95a5a6' for imp in importances]
    
    ax.barh(names, importances, color=colors)
    ax.set_xlabel('Feature Importance', fontsize=12)
    ax.set_title(f'RQ2: Feature Importance (R²={best_result["test_r2"]:.3f})', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('rq2_features.png', dpi=300, bbox_inches='tight')
    logger.info(" Saved: rq2_features.png")
    plt.close()


# ============================================================
# 6. HELPER FUNCTIONS
# ============================================================

def clear_rq2_checkpoints():
    """Clear all checkpoints - start fresh."""
    checkpoint = RQ2Checkpoint()
    checkpoint.clear()
    print(" Checkpoints cleared. Next run will start from scratch.")


def load_rq2_results():
    """Load completed results if they exist."""
    if os.path.exists('rq2_results.json'):
        with open('rq2_results.json', 'r') as f:
            results = json.load(f)
        print(f" Loaded results:")
        print(f"   Test R²: {results['best_model']['test_r2']:.3f}")
        print(f"   Items: {results['n_items']}")
        print(f"   Top 3 features: {results['best_model']['top_10_features'][:3]}")
        return results
    else:
        print("L No results found. Run experiment first.")
        return None


# ============================================================
# 7. USAGE EXAMPLES
# ============================================================

print("="*60)
print("RQ2 CHECKPOINT SYSTEM LOADED")
print("="*60)
print("\nUSAGE:")
print("\n1. Run experiment (auto-resumes if interrupted):")
print("   rq2_results = run_rq2_on_your_data(ns_engine, limit=100)")
print("\n2. Start fresh (ignore checkpoints):")
print("   clear_rq2_checkpoints()")
print("   rq2_results = run_rq2_on_your_data(ns_engine, limit=100, resume=False)")
print("\n3. Load completed results:")
print("   results = load_rq2_results()")
print("="*60)

# Run experiment (paste this):
rq2_results = run_rq2_on_your_data(
    ns_engine=ns_engine,
    dataset_path="./data/100diabetes_qa_dataset.jsonl",
    k=5,
    limit=100,  # 100 queries
    resume=True  # Auto-resume if stopped
)
#clear_rq2_checkpoints()
#rq2_results = run_rq2_on_your_data(ns_engine, limit=100, resume=False)

In [ ]:
#rq2_results = run_rq2_on_your_data(ns_engine, limit=100)

In [ ]:
# ============================================================
# RQ1 + RQ3 COMPLETE - DUAL LIFTING (KG + UMLS)
# ============================================================
# 
# IMPROVEMENTS:
#  - Dual lifting: KG-based + UMLS text-based
#  - Uses verifier's lifting results first
#  - Falls back to KG lifting if still unsupported
#  - Proper tracking of which lifting method succeeded
#  - Better ancestor lookup using fixed UMLS
# ============================================================

import json
import logging
import numpy as np
import pickle
import re
import shutil
import time
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Set, Any
from dataclasses import dataclass, asdict, field
from collections import defaultdict

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


# ============================================================
# CHECKPOINT MANAGERS
# ============================================================

class RQ1Checkpoint:
    """Checkpoint manager for RQ1 with resume capability."""
    
    def __init__(self, checkpoint_dir: str = './rq1_checkpoints'):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(exist_ok=True)
        self.claims_file = self.checkpoint_dir / 'all_claims.pkl'
        self.progress_file = self.checkpoint_dir / 'progress.json'
    
    def save(self, all_claims: List[Dict], query_idx: int, total_queries: int):
        """Save checkpoint after processing queries."""
        self.checkpoint_dir.mkdir(exist_ok=True) 
        with open(self.claims_file, 'wb') as f:
            pickle.dump(all_claims, f)
        
        progress = {
            'query_idx': query_idx,
            'total_queries': total_queries,
            'n_claims': len(all_claims),
            'timestamp': str(datetime.now())
        }
        with open(self.progress_file, 'w') as f:
            json.dump(progress, f, indent=2)
        
        logger.info(f" RQ1 Checkpoint saved: {query_idx}/{total_queries} queries, {len(all_claims)} claims")
    
    def load(self) -> Tuple[List[Dict], int]:
        """Load checkpoint. Returns (all_claims, start_query_idx)."""
        if not self.exists():
            return [], 0
        
        with open(self.claims_file, 'rb') as f:
            all_claims = pickle.load(f)
        
        with open(self.progress_file, 'r') as f:
            progress = json.load(f)
        
        logger.info(f" RQ1 Resuming from checkpoint:")
        logger.info(f"   Query: {progress['query_idx']}/{progress['total_queries']}")
        logger.info(f"   Claims collected: {len(all_claims)}")
        
        return all_claims, progress['query_idx']
    
    def exists(self) -> bool:
        return self.claims_file.exists() and self.progress_file.exists()
    
    def clear(self):
        if self.checkpoint_dir.exists():
            shutil.rmtree(self.checkpoint_dir)
            logger.info(" RQ1 checkpoints cleared")


class RQ3Checkpoint:
    """Checkpoint manager for RQ3 with resume capability."""
    
    def __init__(self, checkpoint_dir: str = './rq3_checkpoints'):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(exist_ok=True)
        self.iterations_file = self.checkpoint_dir / 'iteration_results.pkl'
        self.kg_additions_file = self.checkpoint_dir / 'kg_additions.json'
        self.progress_file = self.checkpoint_dir / 'progress.json'
    
    def save_iteration(self, iteration_results: List, kg_additions: List, 
                       completed_iteration: int, total_iterations: int):
        """Save checkpoint after completing an iteration."""
        self.checkpoint_dir.mkdir(exist_ok=True)
        with open(self.iterations_file, 'wb') as f:
            pickle.dump(iteration_results, f)
        
        with open(self.kg_additions_file, 'w') as f:
            json.dump(kg_additions, f, indent=2)
        
        progress = {
            'completed_iteration': completed_iteration,
            'total_iterations': total_iterations,
            'n_iteration_results': len(iteration_results),
            'n_kg_additions': len(kg_additions),
            'timestamp': str(datetime.now())
        }
        with open(self.progress_file, 'w') as f:
            json.dump(progress, f, indent=2)
        
        logger.info(f" RQ3 Checkpoint saved: iteration {completed_iteration}/{total_iterations}")
    
    def load(self) -> Tuple[List, List, int]:
        """Load checkpoint. Returns (iteration_results, kg_additions, start_iteration)."""
        if not self.exists():
            return [], [], 0
        
        with open(self.iterations_file, 'rb') as f:
            iteration_results = pickle.load(f)
        
        kg_additions = []
        if self.kg_additions_file.exists():
            with open(self.kg_additions_file, 'r') as f:
                kg_additions = json.load(f)
        
        with open(self.progress_file, 'r') as f:
            progress = json.load(f)
        
        logger.info(f" RQ3 Resuming from checkpoint:")
        logger.info(f"   Completed iterations: {progress['completed_iteration']}/{progress['total_iterations']}")
        logger.info(f"   KG additions: {len(kg_additions)}")
        
        return iteration_results, kg_additions, progress['completed_iteration']
    
    def exists(self) -> bool:
        return self.iterations_file.exists() and self.progress_file.exists()
    
    def clear(self):
        if self.checkpoint_dir.exists():
            shutil.rmtree(self.checkpoint_dir)
            logger.info(" RQ3 checkpoints cleared")


def clear_all_rq_checkpoints():
    """Clear ALL checkpoints and result files for RQ1 and RQ3."""
    logger.info("="*60)
    logger.info("CLEARING ALL RQ CHECKPOINTS AND RESULTS")
    logger.info("="*60)
    
    RQ1Checkpoint().clear()
    RQ3Checkpoint().clear()
    
    result_files = [
        'rq1_fixed_results.json',
        'rq3_fixed_results.json',
        'rq3_fixed_kg_additions.json',
    ]
    
    for f in result_files:
        if Path(f).exists():
            Path(f).unlink()
            logger.info(f" Removed {f}")
    
    logger.info("="*60)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def safe_mean(values: List) -> float:
    """Calculate mean handling None values safely."""
    if not values:
        return 0.0
    clean = [v if v is not None else 0 for v in values]
    return float(np.mean(clean)) if clean else 0.0


def load_queries(dataset_path: str) -> List[str]:
    """Load queries from JSONL, filtering out empty ones."""
    queries = []
    with open(dataset_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            query = data.get('question') or data.get('query', '')
            if query and query.strip():
                queries.append(query.strip())
    return queries


# ============================================================
# CLAIM PARSER
# ============================================================

class ClaimParser:
    """Parse claim strings like "Metformin TREATS Diabetes" into triplets."""
    
    RELATIONS = [
        'TREATS', 'CAUSES', 'ASSOCIATED_WITH', 'PREVENTS', 'DIAGNOSES',
        'INHIBITS', 'ACTIVATES', 'REGULATES', 'INCREASES', 'DECREASES',
        'INDUCES', 'REDUCES', 'AFFECTS', 'IMPROVES', 'WORSENS',
        'IS_A', 'PART_OF', 'HAS', 'USED_FOR', 'INDICATED_FOR',
        'CONTRAINDICATES', 'INTERACTS_WITH', 'METABOLIZES', 'BINDS_TO'
    ]
    
    @classmethod
    def parse(cls, claim_text: str) -> Tuple[str, str, str]:
        """Parse "Subject RELATION Object" into (subject, relation, object)."""
        if not claim_text:
            return "", "", ""
        
        for rel in cls.RELATIONS:
            if f' {rel} ' in claim_text:
                parts = claim_text.split(f' {rel} ', 1)
                if len(parts) == 2:
                    return parts[0].strip(), rel, parts[1].strip()
        
        words = claim_text.split()
        for i, word in enumerate(words):
            if word.isupper() and len(word) > 2 and i > 0:
                subject = ' '.join(words[:i])
                relation = word
                obj = ' '.join(words[i+1:])
                if subject and obj:
                    return subject, relation, obj
        
        lower_relations = ['treats', 'causes', 'is associated with', 'prevents']
        for rel in lower_relations:
            if f' {rel} ' in claim_text.lower():
                idx = claim_text.lower().find(f' {rel} ')
                subject = claim_text[:idx].strip()
                obj = claim_text[idx + len(rel) + 2:].strip()
                return subject, rel.upper().replace(' ', '_'), obj
        
        return "", "", ""


# ============================================================
# RQ1: HIERARCHICAL EXPLANATION FIDELITY
# ============================================================

@dataclass
class RQ1Result:
    """Results for RQ1 experiment."""
    n_queries: int = 0
    n_claims: int = 0
    claims_with_l0_evidence: int = 0
    claims_with_l1_evidence: int = 0
    claims_with_l2_evidence: int = 0
    l0_to_l1_consistent: int = 0
    l0_evidence_rate: float = 0.0
    cross_layer_consistency: float = 0.0
    avg_proof_depth: float = 0.0
    avg_vcs: float = 0.0
    avg_msl: float = 0.0


class RQ1Analyzer:
    """Analyze hierarchical explanation fidelity using actual proof data."""
    
    def __init__(self, ns_engine):
        self.ns_engine = ns_engine
    
    def analyze_query(self, query: str, k: int = 5) -> Dict:
        """Analyze one query's hierarchical proof structure."""
        output = self.ns_engine.answer(query, k=k)
        
        verification_details = output.get('conflict_report', {}).get('verification_details', [])
        
        query_result = {
            'query': query,
            'n_claims': len(verification_details),
            'claims': []
        }
        
        for vd in verification_details:
            claim_text = vd.get('claim', '')
            proof = vd.get('hierarchical_proof', {})
            proof_stats = vd.get('proof_stats', {})
            
            l0_evidence = proof.get('0', []) or proof.get(0, [])
            l1_evidence = proof.get('1', []) or proof.get(1, [])
            l2_evidence = proof.get('2', []) or proof.get(2, [])
            
            claim_analysis = {
                'claim': claim_text,
                'status': vd.get('status', ''),
                'has_l0': len(l0_evidence) > 0,
                'has_l1': len(l1_evidence) > 0,
                'has_l2': len(l2_evidence) > 0,
                'n_l0_nodes': len(l0_evidence),
                'n_l1_nodes': len(l1_evidence),
                'n_l2_nodes': len(l2_evidence),
                'vcs': proof_stats.get('VCS') or 0,
                'msl': proof_stats.get('MSL') or 0,
                'layers_supported': proof_stats.get('layers_supported', [])
            }
            
            if l0_evidence and l1_evidence:
                l0_communities = set()
                for item in l0_evidence:
                    meta = item.get('meta', {})
                    if 'from' in meta:
                        l0_communities.add(meta['from'])
                
                l1_nodes = {item.get('node_id', '') for item in l1_evidence}
                claim_analysis['l0_l1_consistent'] = bool(l0_communities & l1_nodes)
            else:
                claim_analysis['l0_l1_consistent'] = False
            
            query_result['claims'].append(claim_analysis)
        
        return query_result
    
    def run_experiment(self, queries: List[str], max_queries: int = 50, 
                       checkpoint_every: int = 5, resume: bool = True) -> RQ1Result:
        """Run RQ1 experiment with checkpointing."""
        checkpoint = RQ1Checkpoint()
        
        if resume and checkpoint.exists():
            all_claims, start_idx = checkpoint.load()
        else:
            all_claims, start_idx = [], 0
            if checkpoint.exists():
                checkpoint.clear()
        
        total_queries = min(len(queries), max_queries)
        
        for i in range(start_idx, total_queries):
            query = queries[i]
            logger.info(f"[{i+1}/{total_queries}] {query[:50]}...")
            
            try:
                query_result = self.analyze_query(query)
                all_claims.extend(query_result['claims'])
                
                if (i + 1) % checkpoint_every == 0:
                    checkpoint.save(all_claims, i + 1, total_queries)
                    
            except Exception as e:
                logger.error(f"Error on query {i}: {e}")
                checkpoint.save(all_claims, i + 1, total_queries)
                raise
        
        checkpoint.save(all_claims, total_queries, total_queries)
        
        result = RQ1Result()
        result.n_queries = total_queries
        result.n_claims = len(all_claims)
        
        if all_claims:
            result.claims_with_l0_evidence = sum(1 for c in all_claims if c['has_l0'])
            result.claims_with_l1_evidence = sum(1 for c in all_claims if c['has_l1'])
            result.claims_with_l2_evidence = sum(1 for c in all_claims if c['has_l2'])
            result.l0_to_l1_consistent = sum(1 for c in all_claims if c.get('l0_l1_consistent', False))
            
            result.l0_evidence_rate = result.claims_with_l0_evidence / len(all_claims)
            result.cross_layer_consistency = result.l0_to_l1_consistent / len(all_claims)
            result.avg_vcs = safe_mean([c['vcs'] for c in all_claims])
            result.avg_msl = safe_mean([c['msl'] for c in all_claims])
            
            depths = [sum([c['has_l0'], c['has_l1'], c['has_l2']]) for c in all_claims]
            result.avg_proof_depth = safe_mean(depths)
        
        return result


# ============================================================
# RQ3: VERIFICATION-GUIDED REFINEMENT (DUAL LIFTING)
# ============================================================

@dataclass
class RQ3ClaimAnalysis:
    """Analysis of a single claim with dual lifting support."""
    claim_text: str = ""
    subject: str = ""
    relation: str = ""
    obj: str = ""
    
    # Status tracking
    verifier_status: str = ""      # Status from verifier (already includes UMLS lifting)
    final_status: str = ""         # Final status after all lifting attempts
    support_score: float = 0.0
    
    # UMLS-based lifting (done by verifier)
    umls_lift_attempted: bool = False
    umls_lift_succeeded: bool = False
    umls_lift_evidence: str = ""
    
    # KG-based lifting (our additional attempt)
    kg_lift_attempted: bool = False
    kg_lift_succeeded: bool = False
    kg_lift_path: List[str] = field(default_factory=list)
    kg_lift_evidence: str = ""


@dataclass
class RQ3IterationResult:
    """Results from one iteration with dual lifting stats."""
    iteration: int = 0
    n_queries: int = 0
    n_claims: int = 0
    
    # Final status counts
    supported: int = 0
    supported_via_umls_lift: int = 0
    supported_via_kg_lift: int = 0
    novel: int = 0
    unsupported: int = 0
    contradicted: int = 0
    
    # Lifting stats
    umls_lift_attempted: int = 0
    umls_lift_succeeded: int = 0
    kg_lift_attempted: int = 0
    kg_lift_succeeded: int = 0
    
    # Rates
    support_rate: float = 0.0
    original_support_rate: float = 0.0
    umls_lift_success_rate: float = 0.0
    kg_lift_success_rate: float = 0.0
    
    # Parsing
    parsed_triplets: int = 0
    unparsed_claims: int = 0
    duration_seconds: float = 0.0


class RQ3RefinerDualLifting:
    """
    RQ3 Refiner with DUAL LIFTING:
    1. UMLS-based lifting (text evidence) - done by verifier
    2. KG-based lifting (graph edges) - additional attempt here
    """
    
    RELATION_MAPPING = {
        'TREATS': ['TREATS', 'treats', 'THERAPY', 'therapy', 'RELATION'],
        'CAUSES': ['CAUSES', 'causes', 'INDUCES', 'induces', 'RELATION'],
        'ASSOCIATED_WITH': ['ASSOCIATED_WITH', 'associated_with', 'RELATED_TO', 'RELATION'],
        'PREVENTS': ['PREVENTS', 'prevents', 'RELATION'],
        'DIAGNOSES': ['DIAGNOSES', 'diagnoses', 'RELATION'],
        'INHIBITS': ['INHIBITS', 'inhibits', 'RELATION'],
        'ACTIVATES': ['ACTIVATES', 'activates', 'RELATION'],
        'INCREASES': ['INCREASES', 'increases', 'RELATION'],
        'DECREASES': ['DECREASES', 'decreases', 'RELATION'],
        'AFFECTS': ['AFFECTS', 'affects', 'RELATION'],
    }
    
    def __init__(self, ns_engine, enable_kg_lifting: bool = True):
        """
        Initialize refiner with dual lifting.
        
        Args:
            ns_engine: NeuroSymbolicHAGRAGRunner instance
            enable_kg_lifting: Whether to attempt additional KG-based lifting
        """
        self.ns_engine = ns_engine
        self.graph_store = ns_engine.graph_store
        self.umls = ns_engine.verifier.umls
        self.enable_kg_lifting = enable_kg_lifting
        self.kg_additions = []
        
        logger.info("RQ3RefinerDualLifting initialized")
        logger.info(f"  UMLS lifting: Always ON (via verifier)")
        logger.info(f"  KG lifting: {enable_kg_lifting}")
        logger.info(f"  MRHIER loaded: {self.umls.mrhier_loaded if self.umls else False}")
    
    def get_umls_ancestors(self, entity: str, max_depth: int = 3) -> List[str]:
        """Get UMLS MRHIER ancestors for an entity."""
        if not entity or not self.umls:
            return []
        
        try:
            # Use the fixed get_mrhier_ancestors that checks all AUIs
            ancestors = self.umls.get_mrhier_ancestors(entity, max_depth=max_depth)
            return ancestors if ancestors else []
        except Exception as e:
            logger.debug(f"Ancestor lookup failed for {entity}: {e}")
            return []
    
    def check_kg_support(self, subject: str, relation: str, obj: str) -> Tuple[bool, str]:
        """
        Check if relation exists in KG (with flexible matching).
        Returns (found, evidence_description).
        """
        if not subject or not obj:
            return False, ""
        
        try:
            with self.graph_store.driver.session() as session:
                # Get relation variants
                rel_variants = self.RELATION_MAPPING.get(relation, [relation, relation.lower(), 'RELATION'])
                
                # Try with relation type first
                query = """
                    MATCH (s:Entity)-[r]->(o:Entity)
                    WHERE (toLower(s.name) CONTAINS toLower($subj) OR toLower($subj) CONTAINS toLower(s.name))
                    AND (toLower(o.name) CONTAINS toLower($obj) OR toLower($obj) CONTAINS toLower(o.name))
                    AND (type(r) IN $rel_types OR r.type IN $rel_types)
                    RETURN s.name as src, type(r) as rel, o.name as tgt
                    LIMIT 1
                """
                
                result = session.run(
                    query,
                    subj=subject[:50],
                    obj=obj[:50],
                    rel_types=rel_variants
                )
                
                record = result.single()
                if record:
                    return True, f"Found: {record['src']} --[{record['rel']}]--> {record['tgt']}"
                
                # Fallback: any relation between entities
                query_any = """
                    MATCH (s:Entity)-[r]->(o:Entity)
                    WHERE (toLower(s.name) CONTAINS toLower($subj) OR toLower($subj) CONTAINS toLower(s.name))
                    AND (toLower(o.name) CONTAINS toLower($obj) OR toLower($obj) CONTAINS toLower(o.name))
                    RETURN s.name as src, type(r) as rel, o.name as tgt
                    LIMIT 1
                """
                
                result = session.run(query_any, subj=subject[:50], obj=obj[:50])
                record = result.single()
                if record:
                    return True, f"Found (any rel): {record['src']} --[{record['rel']}]--> {record['tgt']}"
                
                return False, ""
                
        except Exception as e:
            logger.debug(f"KG check failed: {e}")
            return False, ""
    
    def try_kg_lifting(self, subject: str, relation: str, obj: str) -> Tuple[bool, List[str], str]:
        """
        Try to verify claim by lifting subject to ancestors and checking KG.
        
        Returns: (success, lift_path, evidence)
        """
        if not subject or not relation or not obj:
            return False, [], ""
        
        # Get ancestors for subject
        subject_ancestors = self.get_umls_ancestors(subject, max_depth=3)
        
        # Get ancestors for object too
        object_ancestors = self.get_umls_ancestors(obj, max_depth=3)
        
        lift_path = [f"Original: {subject} -> {obj}"]
        
        # Try lifting subject
        for ancestor in subject_ancestors:
            lift_path.append(f"Subject lifted to: {ancestor}")
            
            found, evidence = self.check_kg_support(ancestor, relation, obj)
            if found:
                return True, lift_path, f"KG lift (subject): {evidence}"
            
            # Also try with lifted object
            for obj_ancestor in object_ancestors:
                found, evidence = self.check_kg_support(ancestor, relation, obj_ancestor)
                if found:
                    lift_path.append(f"Object lifted to: {obj_ancestor}")
                    return True, lift_path, f"KG lift (both): {evidence}"
        
        # Try lifting only object
        for obj_ancestor in object_ancestors:
            lift_path.append(f"Object lifted to: {obj_ancestor}")
            found, evidence = self.check_kg_support(subject, relation, obj_ancestor)
            if found:
                return True, lift_path, f"KG lift (object): {evidence}"
        
        return False, lift_path, ""
    
    def process_query(self, query: str, k: int = 5) -> List[RQ3ClaimAnalysis]:
        """Process a query and analyze all claims with dual lifting."""
        output = self.ns_engine.answer(query, k=k)
        
        verification_details = output.get('conflict_report', {}).get('verification_details', [])
        analyses = []
        
        for vd in verification_details:
            claim_text = vd.get('claim', '')
            subject, relation, obj = ClaimParser.parse(claim_text)
            
            verifier_status = vd.get('status', '')
            reason = vd.get('reason', '')
            
            analysis = RQ3ClaimAnalysis(
                claim_text=claim_text,
                subject=subject,
                relation=relation,
                obj=obj,
                verifier_status=verifier_status,
                final_status=verifier_status,
                support_score=vd.get('support_score', 0.0)
            )
            
            # Check if verifier already did UMLS lifting
            if 'lifting' in reason.lower() or 'lifted' in reason.lower():
                analysis.umls_lift_attempted = True
                if verifier_status in ['supported', 'novel']:
                    analysis.umls_lift_succeeded = True
                    analysis.umls_lift_evidence = reason
                    analysis.final_status = 'supported_via_umls_lift'
            
            # If still unsupported after verifier, try KG lifting
            if verifier_status in ['unsupported', 'contradicted'] and self.enable_kg_lifting:
                if subject and relation and obj:
                    analysis.kg_lift_attempted = True
                    
                    success, path, evidence = self.try_kg_lifting(subject, relation, obj)
                    analysis.kg_lift_path = path
                    
                    if success:
                        analysis.kg_lift_succeeded = True
                        analysis.kg_lift_evidence = evidence
                        analysis.final_status = 'supported_via_kg_lift'
            
            # Track novel claims for KG expansion
            if analysis.final_status == 'novel' and subject and relation and obj:
                self.kg_additions.append({
                    'subject': subject,
                    'relation': relation,
                    'object': obj,
                    'claim': claim_text,
                    'source_query': query,
                    'timestamp': str(datetime.now())
                })
            
            analyses.append(analysis)
        
        return analyses
    
    def run_iteration(self, queries: List[str], k: int = 5) -> RQ3IterationResult:
        """Run one iteration with dual lifting tracking."""
        start_time = time.time()
        
        result = RQ3IterationResult()
        result.n_queries = len(queries)
        
        all_analyses = []
        
        for i, query in enumerate(queries):
            logger.info(f"Processing query: {query[:80]}...")
            
            try:
                analyses = self.process_query(query, k=k)
                all_analyses.extend(analyses)
            except Exception as e:
                logger.warning(f"Error on query {i}: {e}")
        
        result.n_claims = len(all_analyses)
        
        # Count by final status
        original_supported = 0
        
        for a in all_analyses:
            # Track original support
            if a.verifier_status == 'supported':
                original_supported += 1
            
            # Count by final status
            if a.final_status == 'supported':
                result.supported += 1
            elif a.final_status == 'supported_via_umls_lift':
                result.supported_via_umls_lift += 1
            elif a.final_status == 'supported_via_kg_lift':
                result.supported_via_kg_lift += 1
            elif a.final_status == 'novel':
                result.novel += 1
            elif a.final_status == 'unsupported':
                result.unsupported += 1
            elif a.final_status == 'contradicted':
                result.contradicted += 1
            
            # Lifting stats
            if a.umls_lift_attempted:
                result.umls_lift_attempted += 1
                if a.umls_lift_succeeded:
                    result.umls_lift_succeeded += 1
            
            if a.kg_lift_attempted:
                result.kg_lift_attempted += 1
                if a.kg_lift_succeeded:
                    result.kg_lift_succeeded += 1
            
            # Parsing stats
            if a.subject and a.relation and a.obj:
                result.parsed_triplets += 1
            else:
                result.unparsed_claims += 1
        
        # Calculate rates
        if result.n_claims > 0:
            total_supported = result.supported + result.supported_via_umls_lift + result.supported_via_kg_lift
            result.support_rate = total_supported / result.n_claims
            result.original_support_rate = original_supported / result.n_claims
        
        if result.umls_lift_attempted > 0:
            result.umls_lift_success_rate = result.umls_lift_succeeded / result.umls_lift_attempted
        
        if result.kg_lift_attempted > 0:
            result.kg_lift_success_rate = result.kg_lift_succeeded / result.kg_lift_attempted
        
        result.duration_seconds = time.time() - start_time
        
        return result


# ============================================================
# MAIN RUNNERS
# ============================================================

def run_rq1_fixed(ns_engine, dataset_path: str, max_queries: int = 50, 
                  output_dir: str = '.', resume: bool = True,
                  checkpoint_every: int = 5) -> Dict:
    """Run RQ1 experiment with checkpointing."""
    logger.info("="*70)
    logger.info("RQ1: HIERARCHICAL EXPLANATION FIDELITY")
    logger.info(f"Resume from checkpoint: {resume}")
    logger.info("="*70)
    
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    queries = load_queries(dataset_path)
    logger.info(f"Loaded {len(queries)} valid queries")
    
    analyzer = RQ1Analyzer(ns_engine)
    result = analyzer.run_experiment(
        queries, 
        max_queries=max_queries, 
        checkpoint_every=checkpoint_every,
        resume=resume
    )
    
    final_results = {
        'experiment_info': {
            'dataset': dataset_path,
            'max_queries': max_queries,
            'timestamp': str(datetime.now())
        },
        'metrics': {
            'n_queries': result.n_queries,
            'n_claims': result.n_claims,
            'l0_evidence_rate': result.l0_evidence_rate,
            'cross_layer_consistency': result.cross_layer_consistency,
            'avg_proof_depth': result.avg_proof_depth,
            'avg_vcs': result.avg_vcs,
            'avg_msl': result.avg_msl
        },
        'evidence_distribution': {
            'claims_with_l0': result.claims_with_l0_evidence,
            'claims_with_l1': result.claims_with_l1_evidence,
            'claims_with_l2': result.claims_with_l2_evidence
        }
    }
    
    with open(output_path / 'rq1_fixed_results.json', 'w') as f:
        json.dump(final_results, f, indent=2)
    
    logger.info("\n" + "="*70)
    logger.info("RQ1 RESULTS")
    logger.info("="*70)
    logger.info(f"Queries: {result.n_queries}, Claims: {result.n_claims}")
    logger.info(f"L0 Evidence Rate: {result.l0_evidence_rate:.2%}")
    logger.info(f"Cross-Layer Consistency: {result.cross_layer_consistency:.2%}")
    logger.info(f"Avg Proof Depth: {result.avg_proof_depth:.2f} layers")
    logger.info(f"Avg VCS: {result.avg_vcs:.3f}")
    logger.info(f"Avg MSL: {result.avg_msl:.3f}")
    logger.info(f"\nSaved: {output_path / 'rq1_fixed_results.json'}")
    logger.info("="*70)
    
    return final_results


def run_rq3_fixed(ns_engine, dataset_path: str, n_iterations: int = 3, 
                  queries_per_iteration: int = 30, output_dir: str = '.',
                  resume: bool = True, enable_kg_lifting: bool = True) -> Dict:
    """
    Run RQ3 experiment with DUAL LIFTING.
    
    Args:
        enable_kg_lifting: If True, try KG-based lifting in addition to UMLS lifting
    """
    logger.info("="*70)
    logger.info("RQ3: VERIFICATION-GUIDED REFINEMENT (DUAL LIFTING)")
    logger.info(f"Resume from checkpoint: {resume}")
    logger.info(f"KG lifting enabled: {enable_kg_lifting}")
    logger.info("="*70)
    
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    checkpoint = RQ3Checkpoint()
    
    queries = load_queries(dataset_path)
    logger.info(f"Loaded {len(queries)} valid queries")
    
    if resume and checkpoint.exists():
        iteration_results, kg_additions, start_iteration = checkpoint.load()
    else:
        iteration_results, kg_additions, start_iteration = [], [], 0
        if checkpoint.exists():
            checkpoint.clear()
    
    refiner = RQ3RefinerDualLifting(ns_engine, enable_kg_lifting=enable_kg_lifting)
    refiner.kg_additions = kg_additions
    
    for iteration in range(start_iteration, n_iterations):
        logger.info(f"\n{'='*60}")
        logger.info(f"ITERATION {iteration + 1}/{n_iterations}")
        logger.info(f"{'='*60}")
        
        start = (iteration * queries_per_iteration) % len(queries)
        end = start + queries_per_iteration
        if end > len(queries):
            iter_queries = queries[start:] + queries[:end - len(queries)]
        else:
            iter_queries = queries[start:end]
        
        try:
            result = refiner.run_iteration(iter_queries)
            result.iteration = iteration + 1
            iteration_results.append(result)
            
            logger.info(f"\n--- Iteration {iteration + 1} Summary ---")
            logger.info(f"  Claims: {result.n_claims}")
            logger.info(f"  Parsed triplets: {result.parsed_triplets} ({result.parsed_triplets/max(result.n_claims,1):.1%})")
            logger.info(f"  Original support rate: {result.original_support_rate:.2%}")
            logger.info(f"  Final support rate: {result.support_rate:.2%}")
            logger.info(f"  ")
            logger.info(f"  UMLS Lifting: {result.umls_lift_attempted} attempted, {result.umls_lift_succeeded} succeeded")
            logger.info(f"  KG Lifting: {result.kg_lift_attempted} attempted, {result.kg_lift_succeeded} succeeded")
            logger.info(f"  ")
            logger.info(f"  Supported (original): {result.supported}")
            logger.info(f"  Supported via UMLS lift: {result.supported_via_umls_lift}")
            logger.info(f"  Supported via KG lift: {result.supported_via_kg_lift}")
            logger.info(f"  Novel: {result.novel}")
            logger.info(f"  Unsupported: {result.unsupported}")
            
            checkpoint.save_iteration(
                iteration_results, 
                refiner.kg_additions, 
                iteration + 1, 
                n_iterations
            )
            
        except Exception as e:
            logger.error(f"Error in iteration {iteration + 1}: {e}")
            checkpoint.save_iteration(
                iteration_results, 
                refiner.kg_additions, 
                iteration, 
                n_iterations
            )
            raise
    
    # Compile final results
    first = iteration_results[0]
    last = iteration_results[-1]
    
    total_umls_lift = sum(r.umls_lift_succeeded for r in iteration_results)
    total_kg_lift = sum(r.kg_lift_succeeded for r in iteration_results)
    
    final_results = {
        'experiment_info': {
            'dataset': dataset_path,
            'n_iterations': n_iterations,
            'queries_per_iteration': queries_per_iteration,
            'kg_lifting_enabled': enable_kg_lifting,
            'timestamp': str(datetime.now())
        },
        'improvement': {
            'initial_support_rate': first.support_rate,
            'final_support_rate': last.support_rate,
            'change': last.support_rate - first.support_rate,
            'initial_original_support_rate': first.original_support_rate,
            'final_original_support_rate': last.original_support_rate
        },
        'parsing': {
            'total_claims': sum(r.n_claims for r in iteration_results),
            'parsed_triplets': sum(r.parsed_triplets for r in iteration_results),
            'unparsed_claims': sum(r.unparsed_claims for r in iteration_results),
            'parse_rate': sum(r.parsed_triplets for r in iteration_results) / max(sum(r.n_claims for r in iteration_results), 1)
        },
        'lifting': {
            'umls': {
                'total_attempted': sum(r.umls_lift_attempted for r in iteration_results),
                'total_succeeded': total_umls_lift,
                'success_rate': total_umls_lift / max(sum(r.umls_lift_attempted for r in iteration_results), 1)
            },
            'kg': {
                'total_attempted': sum(r.kg_lift_attempted for r in iteration_results),
                'total_succeeded': total_kg_lift,
                'success_rate': total_kg_lift / max(sum(r.kg_lift_attempted for r in iteration_results), 1)
            },
            'combined': {
                'total_lifted': total_umls_lift + total_kg_lift,
                'total_supported_via_lift': sum(r.supported_via_umls_lift + r.supported_via_kg_lift for r in iteration_results)
            }
        },
        'kg_expansion': {
            'novel_claims': len(refiner.kg_additions),
            'candidates': refiner.kg_additions[:20]
        },
        'per_iteration': [asdict(r) for r in iteration_results]
    }
    
    with open(output_path / 'rq3_fixed_results.json', 'w') as f:
        json.dump(final_results, f, indent=2)
    
    with open(output_path / 'rq3_fixed_kg_additions.json', 'w') as f:
        json.dump(refiner.kg_additions, f, indent=2)
    
    logger.info("\n" + "="*70)
    logger.info("RQ3 FINAL RESULTS (DUAL LIFTING)")
    logger.info("="*70)
    logger.info(f"Total claims: {final_results['parsing']['total_claims']}")
    logger.info(f"Parse rate: {final_results['parsing']['parse_rate']:.2%}")
    logger.info(f"")
    logger.info(f"Support rates:")
    logger.info(f"  Original: {first.original_support_rate:.2%}  {last.original_support_rate:.2%}")
    logger.info(f"  Final:    {first.support_rate:.2%}  {last.support_rate:.2%} ({last.support_rate - first.support_rate:+.2%})")
    logger.info(f"")
    logger.info(f"Lifting performance:")
    logger.info(f"  UMLS: {final_results['lifting']['umls']['total_succeeded']}/{final_results['lifting']['umls']['total_attempted']} succeeded")
    logger.info(f"  KG:   {final_results['lifting']['kg']['total_succeeded']}/{final_results['lifting']['kg']['total_attempted']} succeeded")
    logger.info(f"  Total lifted: {final_results['lifting']['combined']['total_lifted']}")
    logger.info(f"")
    logger.info(f"Novel claims for KG: {len(refiner.kg_additions)}")
    logger.info(f"\nSaved: {output_path / 'rq3_fixed_results.json'}")
    logger.info("="*70)
    
    return final_results


# ============================================================
# USAGE
# ============================================================

print("="*60)
print("RQ1 + RQ3 WITH DUAL LIFTING (KG + UMLS)")
print("="*60)
print("")
print("FEATURES:")
print("   UMLS-based lifting (text evidence via verifier)")
print("   KG-based lifting (graph edges with ancestors)")
print("   Proper tracking of which method succeeded")
print("   Bidirectional entity matching in KG queries")
print("")
print("USAGE:")
print("")
print("  # Run with both lifting methods")
print("  rq3_results = run_rq3_fixed(ns_engine, dataset_path)")
print("")
print("  # Run with only UMLS lifting (disable KG lifting)")
print("  rq3_results = run_rq3_fixed(ns_engine, dataset_path, enable_kg_lifting=False)")
print("")
print("="*60)


# ============================================================
# UNCOMMENT TO RUN
# ============================================================

# rq1_results = run_rq1_fixed(
#     ns_engine=ns_engine,
#     dataset_path="./data/100diabetes_qa_dataset.jsonl",
#     max_queries=10,
#     resume=False 
# )

# rq3_results = run_rq3_fixed(
#     ns_engine=ns_engine,
#     dataset_path="./data/100diabetes_qa_dataset.jsonl",
#     n_iterations=3,
#     queries_per_iteration=10,
#     enable_kg_lifting=True,
#     resume=False
# )


# Clear and run
#clear_all_rq_checkpoints()

rq1_results = run_rq1_fixed(
    ns_engine=ns_engine,
    dataset_path="./data/100diabetes_qa_dataset.jsonl",
    max_queries=100,
    resume=True 
)

rq3_results = run_rq3_fixed(
    ns_engine=ns_engine,
    dataset_path="./data/100diabetes_qa_dataset.jsonl",
    n_iterations=3,
    queries_per_iteration=100,
    enable_kg_lifting=True,
    resume=True
)

In [ ]:
# ============================================================
# CELL: DATASET EXTRACTION - PubmedQA Original + BioASQ
# ============================================================
# Extracts diabetes-related QA pairs from:
#   1. PubmedQA Original (GitHub)
#   2. BioASQ (HuggingFace)
# Outputs: JSONL files matching your existing dataset format
#   {"question": ..., "answer": ..., "context": ..., "source": ...}
# ============================================================

import json
import os
import re
import urllib.request
import logging
from pathlib import Path
from typing import List, Dict, Optional

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# ============================================================
# CONFIG
# ============================================================

DATASET_OUTPUT_DIR = "./"
PUBMEDQA_ORIG_URL = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqal.json"
PUBMEDQA_ORIG_OUTPUT = os.path.join(DATASET_OUTPUT_DIR, "pubmedqa_original_diabetes.jsonl")
BIOASQ_OUTPUT = os.path.join(DATASET_OUTPUT_DIR, "bioasq_diabetes.jsonl")

DIABETES_KEYWORDS = [
    "diabetes", "diabetic", "insulin", "glucose", "glycemic", "glycaemic",
    "hyperglycemia", "hyperglycaemia", "hypoglycemia", "hypoglycaemia",
    "metformin", "hba1c", "a1c", "type 1", "type 2", "t1d", "t2d",
    "t1dm", "t2dm", "dka", "diabetic ketoacidosis", "pancreas", "beta cell",
    "glucagon", "glycosylated", "insulin resistance", "prediabetes",
    "gestational diabetes", "gdm", "islet", "glipizide", "gliclazide",
    "sitagliptin", "empagliflozin", "liraglutide", "pioglitazone",
    "sulfonylurea", "sglt2", "glp-1", "fasting glucose", "blood glucose",
    "postprandial glucose", "hemoglobin a1c"
]


def is_diabetes_related(text: str) -> bool:
    """Check if text contains diabetes-related keywords."""
    text_l = text.lower()
    return any(kw in text_l for kw in DIABETES_KEYWORDS)


# ============================================================
# EXTRACTOR 1: PubmedQA Original
# ============================================================

class PubmedQAOriginalExtractor:
    """
    Downloads and filters PubmedQA original labeled dataset from GitHub.
    
    Source format:
      {PMID: {QUESTION, CONTEXTS, LABELS, LONG_ANSWER, MESHES}}
    
    Output format (matching your existing JSONL):
      {"question": ..., "answer": ..., "context": ..., "source": ...}
    """

    def __init__(self, url: str = PUBMEDQA_ORIG_URL, output_path: str = PUBMEDQA_ORIG_OUTPUT):
        self.url = url
        self.output_path = output_path

    def _download(self) -> Dict:
        """Download JSON from GitHub."""
        logging.info(f"Downloading PubmedQA original from GitHub...")
        try:
            with urllib.request.urlopen(self.url, timeout=30) as response:
                data = json.loads(response.read().decode("utf-8"))
            logging.info(f"Downloaded {len(data)} entries")
            return data
        except Exception as e:
            raise RuntimeError(f"Failed to download PubmedQA original: {e}")

    def extract(self, force: bool = False) -> str:
        """
        Extract diabetes-related QA pairs and save to JSONL.

        Args:
            force: If True, re-extract even if output file exists.

        Returns:
            Path to output JSONL file.
        """
        if not force and Path(self.output_path).exists():
            existing = sum(1 for _ in open(self.output_path))
            logging.info(f"PubmedQA original already extracted: {existing} diabetes QAs at {self.output_path}")
            return self.output_path

        raw = self._download()
        rows = []

        for pmid, entry in raw.items():
            question = entry.get("QUESTION", "").strip()
            long_answer = entry.get("LONG_ANSWER", "").strip()
            contexts = entry.get("CONTEXTS", [])
            context_text = " ".join(contexts).strip()

            # Filter to diabetes-related
            combined = question + " " + long_answer + " " + context_text
            if not is_diabetes_related(combined):
                continue

            if not question or not long_answer:
                continue

            rows.append({
                "question": question,
                "answer": long_answer,
                "context": context_text[:3000],   # truncate to match pipeline limits
                "source": f"pubmedqa_original_{pmid}.pdf"
            })

        Path(self.output_path).parent.mkdir(parents=True, exist_ok=True)
        with open(self.output_path, "w", encoding="utf-8") as f:
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

        logging.info(f"PubmedQA original: {len(rows)} diabetes QAs saved to {self.output_path}")
        return self.output_path


# ============================================================
# EXTRACTOR 2: BioASQ
# ============================================================

class BioASQExtractor:
    """
    Loads BioASQ Task B from HuggingFace and filters for diabetes.
    
    Falls back gracefully with instructions if access fails.
    
    Output format (matching your existing JSONL):
      {"question": ..., "answer": ..., "context": ..., "source": ...}
    """

    def __init__(self, output_path: str = BIOASQ_OUTPUT):
        self.output_path = output_path

    def _load_from_huggingface(self) -> List[Dict]:
        """Attempt to load BioASQ from HuggingFace datasets."""
        from datasets import load_dataset

        # Try multiple known BioASQ HuggingFace configurations
        configs_to_try = [
            ("bigbio/bioasq", "bioasq_2021_task_b_source"),
            ("bigbio/bioasq", "bioasq10b_source"),
            ("kroshan/BioASQ",  None),
        ]

        for dataset_name, config in configs_to_try:
            try:
                logging.info(f"Trying BioASQ config: {dataset_name} / {config}")
                if config:
                    ds = load_dataset(dataset_name, config, trust_remote_code=True)
                else:
                    ds = load_dataset(dataset_name, trust_remote_code=True)
                logging.info(f"Loaded BioASQ from: {dataset_name} / {config}")
                return ds, dataset_name
            except Exception as e:
                logging.warning(f"Could not load {dataset_name}/{config}: {e}")
                continue

        raise RuntimeError(
            "Could not load BioASQ from any known HuggingFace config.\n"
            "Please manually download from http://bioasq.org and place at:\n"
            f"{self.output_path.replace('.jsonl', '_raw.json')}\n"
            "Then re-run with force=True."
        )

    def _parse_entry(self, entry: Dict, source_name: str) -> Optional[Dict]:
        """Parse a BioASQ entry into your JSONL format."""
        # Field names vary across BioASQ versions
        question = (
            entry.get("body") or
            entry.get("question") or
            entry.get("QUESTION") or ""
        ).strip()

        # ideal_answer is the long-form answer in BioASQ
        answer_raw = (
            entry.get("ideal_answer") or
            entry.get("answer") or
            entry.get("LONG_ANSWER") or ""
        )
        if isinstance(answer_raw, list):
            answer = " ".join(answer_raw).strip()
        else:
            answer = str(answer_raw).strip()

        # context from snippets or documents
        snippets = entry.get("snippets", [])
        if snippets:
            if isinstance(snippets[0], dict):
                context = " ".join(s.get("text", "") for s in snippets)
            else:
                context = " ".join(str(s) for s in snippets)
        else:
            context = entry.get("context", "")

        if not question or not answer:
            return None

        combined = question + " " + answer + " " + context
        if not is_diabetes_related(combined):
            return None

        pmid = str(entry.get("id", entry.get("pmid", "unknown")))

        return {
            "question": question,
            "answer": answer,
            "context": context[:3000],
            "source": f"bioasq_{pmid}.pdf"
        }

    def extract(self, force: bool = False) -> str:
        """
        Extract diabetes-related QA pairs from BioASQ and save to JSONL.

        Args:
            force: If True, re-extract even if output file exists.

        Returns:
            Path to output JSONL file.
        """
        if not force and Path(self.output_path).exists():
            existing = sum(1 for _ in open(self.output_path))
            logging.info(f"BioASQ already extracted: {existing} diabetes QAs at {self.output_path}")
            return self.output_path

        try:
            ds, source_name = self._load_from_huggingface()
        except RuntimeError as e:
            logging.error(str(e))
            return None

        rows = []

        # Iterate over all splits
        for split_name, split_data in ds.items():
            logging.info(f"Processing BioASQ split: {split_name} ({len(split_data)} entries)")
            for entry in split_data:
                parsed = self._parse_entry(dict(entry), source_name)
                if parsed:
                    rows.append(parsed)

        Path(self.output_path).parent.mkdir(parents=True, exist_ok=True)
        with open(self.output_path, "w", encoding="utf-8") as f:
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

        logging.info(f"BioASQ: {len(rows)} diabetes QAs saved to {self.output_path}")
        return self.output_path


# ============================================================
# RUN EXTRACTION
# ============================================================

print("=" * 60)
print("STEP 1: EXTRACTING PUBMEDQA ORIGINAL")
print("=" * 60)

pubmedqa_extractor = PubmedQAOriginalExtractor()
pubmedqa_orig_path = pubmedqa_extractor.extract(force=False)

if pubmedqa_orig_path:
    count = sum(1 for _ in open(pubmedqa_orig_path))
    print(f"PubmedQA Original: {count} diabetes QA pairs -> {pubmedqa_orig_path}")

print()
print("=" * 60)
print("STEP 2: EXTRACTING BIOASQ")
print("=" * 60)

bioasq_extractor = BioASQExtractor()
bioasq_path = bioasq_extractor.extract(force=False)

if bioasq_path:
    count = sum(1 for _ in open(bioasq_path))
    print(f"BioASQ: {count} diabetes QA pairs -> {bioasq_path}")

print()
print("=" * 60)
print("EXTRACTION SUMMARY")
print("=" * 60)
print(f"  PubmedQA Original : {pubmedqa_orig_path}")
print(f"  BioASQ            : {bioasq_path}")
print()
print("To re-extract from scratch, run with force=True:")
print("  pubmedqa_extractor.extract(force=True)")
print("  bioasq_extractor.extract(force=True)")
print("=" * 60)

In [ ]:
# ============================================================
# CELL: MULTI-DATASET EVALUATION - RQ1 + RQ2 + RQ3
# ============================================================
# Runs all 3 research questions on:
#   - PubmedQA Original (diabetes subset)
#   - BioASQ (diabetes subset)
#
# Each dataset gets independent checkpoints.
# Options:
#   resume=True  -> picks up from last saved checkpoint
#   resume=False -> starts from scratch (clears checkpoint first)
# ============================================================

import json
import logging
import os
import pickle
import shutil
import time
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# ============================================================
# DATASET REGISTRY
# ============================================================

MULTI_DATASET_REGISTRY = {
    "pubmedqa_original": {
        "path": "./data/pubmedqa_original_diabetes.jsonl",
        "label": "PubmedQA Original",
        "checkpoint_prefix": "pubmedqa_orig"
    },
    "bioasq": {
        "path": "./data/bioasq_diabetes.jsonl",
        "label": "BioASQ",
        "checkpoint_prefix": "bioasq"
    },
}

MULTI_EVAL_CHECKPOINT_ROOT = "./checkpoints/multi_dataset_eval"


# ============================================================
# GENERIC CHECKPOINT MANAGER (dataset-aware)
# ============================================================

class DatasetCheckpoint:
    """
    Generic checkpoint manager scoped to a single dataset + RQ combination.
    Mirrors the pattern of RQ1Checkpoint / RQ3Checkpoint in your existing code.
    """

    def __init__(self, base_dir: str, dataset_key: str, rq: str):
        """
        Args:
            base_dir: Root checkpoint directory.
            dataset_key: e.g. "pubmedqa_original", "bioasq".
            rq: e.g. "rq1", "rq2", "rq3".
        """
        self.checkpoint_dir = Path(base_dir) / dataset_key / rq
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.data_file = self.checkpoint_dir / "data.pkl"
        self.progress_file = self.checkpoint_dir / "progress.json"
        self.backup_dir = self.checkpoint_dir / "backups"
        self.backup_dir.mkdir(parents=True, exist_ok=True)

    def save(self, data: Any, step: int, total: int, extra: Optional[Dict] = None):
        """Save checkpoint after completing a step."""
        with open(self.data_file, "wb") as f:
            pickle.dump(data, f)

        meta = {
            "step": step,
            "total": total,
            "timestamp": str(datetime.now()),
        }
        if extra:
            meta.update(extra)

        with open(self.progress_file, "w") as f:
            json.dump(meta, f, indent=2)

        logging.info(f"Checkpoint saved: step {step}/{total} -> {self.checkpoint_dir.name}")

    def backup(self):
        """Create a timestamped backup of the current checkpoint."""
        if not self.data_file.exists():
            return
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_path = self.backup_dir / f"backup_{ts}.pkl"
        shutil.copy(self.data_file, backup_path)
        # Keep last 5 backups
        backups = sorted(self.backup_dir.glob("backup_*.pkl"))
        for old in backups[:-5]:
            old.unlink()

    def load(self) -> Tuple[Any, int]:
        """Load checkpoint. Returns (data, last_completed_step)."""
        if not self.exists():
            return None, 0

        with open(self.data_file, "rb") as f:
            data = pickle.load(f)

        with open(self.progress_file, "r") as f:
            meta = json.load(f)

        step = meta.get("step", 0)
        total = meta.get("total", "?")
        logging.info(f"Resumed from checkpoint: step {step}/{total} ({self.checkpoint_dir})")
        return data, step

    def exists(self) -> bool:
        return self.data_file.exists() and self.progress_file.exists()

    def clear(self):
        if self.checkpoint_dir.exists():
            shutil.rmtree(self.checkpoint_dir)
            self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
            self.backup_dir.mkdir(parents=True, exist_ok=True)
            logging.info(f"Checkpoint cleared: {self.checkpoint_dir}")


# ============================================================
# HELPERS
# ============================================================

def load_jsonl_dataset(path: str) -> List[Dict]:
    """Load JSONL dataset matching your existing format."""
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def save_results_jsonl(results: List[Dict], path: str):
    """Save results list to JSONL."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")


# ============================================================
# RQ1: HIERARCHICAL EXPLANATION FIDELITY (multi-dataset)
# ============================================================

def run_rq1_on_dataset(
    ns_engine,
    dataset_key: str,
    dataset_path: str,
    dataset_label: str,
    max_queries: int = 50,
    checkpoint_every: int = 5,
    resume: bool = True,
    output_dir: str = MULTI_EVAL_CHECKPOINT_ROOT
) -> Dict:
    """
    Run RQ1 on a single dataset with per-dataset checkpointing.
    Reuses your existing RQ1Analyzer class.
    """
    logging.info(f"\n{'='*60}")
    logging.info(f"RQ1: {dataset_label} (resume={resume})")
    logging.info(f"{'='*60}")

    ckpt = DatasetCheckpoint(output_dir, dataset_key, "rq1")

    if not resume:
        ckpt.clear()
        logging.info("Starting from scratch (resume=False)")

    # Load dataset
    dataset = load_jsonl_dataset(dataset_path)
    queries = [
        row.get("question") or row.get("query", "")
        for row in dataset
        if row.get("question") or row.get("query")
    ]
    total = min(len(queries), max_queries)
    logging.info(f"Loaded {len(queries)} queries, capped at {total}")

    # Resume or start
    if resume and ckpt.exists():
        all_claims, start_idx = ckpt.load()
    else:
        all_claims, start_idx = [], 0

    analyzer = RQ1Analyzer(ns_engine)

    for i in range(start_idx, total):
        query = queries[i]
        logging.info(f"[{i+1}/{total}] {query[:60]}...")

        try:
            query_result = analyzer.analyze_query(query)
            all_claims.extend(query_result["claims"])
        except Exception as e:
            logging.error(f"RQ1 error on query {i}: {e}")
            ckpt.save(all_claims, i + 1, total)
            raise

        if (i + 1) % checkpoint_every == 0:
            ckpt.backup()
            ckpt.save(all_claims, i + 1, total)

    ckpt.backup()
    ckpt.save(all_claims, total, total)

    # Compute metrics (same as your existing run_rq1_fixed)
    result = RQ1Result()
    result.n_queries = total
    result.n_claims = len(all_claims)

    if all_claims:
        result.claims_with_l0_evidence = sum(1 for c in all_claims if c["has_l0"])
        result.claims_with_l1_evidence = sum(1 for c in all_claims if c["has_l1"])
        result.claims_with_l2_evidence = sum(1 for c in all_claims if c["has_l2"])
        result.l0_to_l1_consistent = sum(1 for c in all_claims if c.get("l0_l1_consistent", False))
        result.l0_evidence_rate = result.claims_with_l0_evidence / len(all_claims)
        result.cross_layer_consistency = result.l0_to_l1_consistent / len(all_claims)
        result.avg_vcs = safe_mean([c["vcs"] for c in all_claims])
        result.avg_msl = safe_mean([c["msl"] for c in all_claims])
        depths = [sum([c["has_l0"], c["has_l1"], c["has_l2"]]) for c in all_claims]
        result.avg_proof_depth = safe_mean(depths)

    final = {
        "dataset": dataset_label,
        "dataset_key": dataset_key,
        "experiment": "RQ1",
        "experiment_info": {
            "max_queries": max_queries,
            "timestamp": str(datetime.now())
        },
        "metrics": {
            "n_queries": result.n_queries,
            "n_claims": result.n_claims,
            "l0_evidence_rate": result.l0_evidence_rate,
            "cross_layer_consistency": result.cross_layer_consistency,
            "avg_proof_depth": result.avg_proof_depth,
            "avg_vcs": result.avg_vcs,
            "avg_msl": result.avg_msl
        },
        "evidence_distribution": {
            "claims_with_l0": result.claims_with_l0_evidence,
            "claims_with_l1": result.claims_with_l1_evidence,
            "claims_with_l2": result.claims_with_l2_evidence
        }
    }

    out_path = Path(output_dir) / dataset_key / f"rq1_results.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        json.dump(final, f, indent=2)

    logging.info(f"\nRQ1 [{dataset_label}] -> L0 rate={result.l0_evidence_rate:.2%}, "
                 f"VCS={result.avg_vcs:.3f}, saved to {out_path}")
    return final


# ============================================================
# RQ2: SYMBOLIC ATTRIBUTION (multi-dataset)
# ============================================================

def run_rq2_on_dataset(
    ns_engine,
    dataset_key: str,
    dataset_path: str,
    dataset_label: str,
    k: int = 5,
    limit: Optional[int] = None,
    resume: bool = True,
    output_dir: str = MULTI_EVAL_CHECKPOINT_ROOT
) -> Dict:
    """
    Run RQ2 on a single dataset with per-dataset checkpointing.
    Reuses your existing RealFeatureExtractor + AttributionModel classes.
    """
    logging.info(f"\n{'='*60}")
    logging.info(f"RQ2: {dataset_label} (resume={resume})")
    logging.info(f"{'='*60}")

    ckpt = DatasetCheckpoint(output_dir, dataset_key, "rq2")

    if not resume:
        ckpt.clear()
        logging.info("Starting from scratch (resume=False)")

    # Load queries from dataset
    dataset = load_jsonl_dataset(dataset_path)
    queries = [
        row.get("question") or row.get("query", "")
        for row in dataset
        if row.get("question") or row.get("query")
    ]
    if limit:
        queries = queries[:limit]
    total = len(queries)
    logging.info(f"Loaded {total} queries for RQ2")

    # Resume or start
    if resume and ckpt.exists():
        all_items, start_idx = ckpt.load()
        if all_items is None:
            all_items, start_idx = [], 0
    else:
        all_items, start_idx = [], 0

    extractor = RealFeatureExtractor(ns_engine)

    for q_idx in range(start_idx, total):
        query = queries[q_idx]
        logging.info(f"[{q_idx+1}/{total}] {query[:60]}...")

        try:
            query_entities = [
                w for w in query.split()
                if len(w) > 3 and w.lower() not in
                {"what", "when", "where", "which", "does", "the", "with", "from", "that", "this"}
            ]

            neural_scores = extractor.get_real_neural_scores(query, k=k)

            for layer, node_score_pairs in neural_scores.items():
                for rank, (node_id, neural_score) in enumerate(node_score_pairs, 1):
                    features = extractor.extract_symbolic_features(node_id, layer, query_entities)
                    features.pop("layer", None)

                    item = RetrievalItem(
                        node_id=node_id,
                        layer=layer,
                        neural_score=neural_score,
                        rank=rank,
                        query=query,
                        **features
                    )
                    all_items.append(item)

        except Exception as e:
            logging.error(f"RQ2 error on query {q_idx}: {e}")
            ckpt.save(all_items, q_idx + 1, total)
            continue

        if (q_idx + 1) % 5 == 0:
            ckpt.backup()
            ckpt.save(all_items, q_idx + 1, total)

    ckpt.backup()
    ckpt.save(all_items, total, total)

    if len(all_items) < 10:
        logging.warning(f"Only {len(all_items)} items collected for RQ2 [{dataset_label}] - too few for robust evaluation")
        return {"dataset": dataset_label, "error": "insufficient_items", "n_items": len(all_items)}

    # Train attribution models
    results = {}
    for model_type in ["decision_tree", "linear", "random_forest"]:
        try:
            model = AttributionModel(model_type=model_type)
            result = model.train_and_evaluate(all_items, test_size=0.2)
            results[model_type] = result
        except Exception as e:
            logging.error(f"RQ2 model training failed [{model_type}]: {e}")
            results[model_type] = {"error": str(e)}

    valid_results = {k: v for k, v in results.items() if "test_r2" in v}
    if not valid_results:
        logging.warning(f"All RQ2 models failed for [{dataset_label}]")
        return {"dataset": dataset_label, "error": "all_models_failed"}

    best_type = max(valid_results.keys(), key=lambda k: valid_results[k]["test_r2"])
    best_result = valid_results[best_type]

    final = {
        "dataset": dataset_label,
        "dataset_key": dataset_key,
        "experiment": "RQ2",
        "n_queries": total,
        "n_items": len(all_items),
        "best_model": {
            "type": best_type,
            "test_r2": best_result["test_r2"],
            "test_spearman": best_result["test_spearman"],
            "top_10_features": best_result["feature_importance"][:10]
        },
        "all_models": {
            k: {kk: vv for kk, vv in v.items() if kk != "feature_importance"}
            for k, v in results.items()
        }
    }

    out_path = Path(output_dir) / dataset_key / "rq2_results.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        json.dump(final, f, indent=2, default=str)

    logging.info(f"\nRQ2 [{dataset_label}] -> Best={best_type}, R²={best_result['test_r2']:.4f}, "
                 f"saved to {out_path}")
    return final


# ============================================================
# RQ3: VERIFICATION-GUIDED REFINEMENT (multi-dataset)
# ============================================================

def run_rq3_on_dataset(
    ns_engine,
    dataset_key: str,
    dataset_path: str,
    dataset_label: str,
    n_iterations: int = 3,
    queries_per_iteration: int = 10,
    enable_kg_lifting: bool = True,
    resume: bool = True,
    output_dir: str = MULTI_EVAL_CHECKPOINT_ROOT
) -> Dict:
    """
    Run RQ3 on a single dataset with per-dataset checkpointing.
    Reuses your existing RQ3RefinerDualLifting class.
    """
    logging.info(f"\n{'='*60}")
    logging.info(f"RQ3: {dataset_label} (resume={resume})")
    logging.info(f"{'='*60}")

    ckpt = DatasetCheckpoint(output_dir, dataset_key, "rq3")

    if not resume:
        ckpt.clear()
        logging.info("Starting from scratch (resume=False)")

    # Load queries
    dataset = load_jsonl_dataset(dataset_path)
    all_queries = [
        row.get("question") or row.get("query", "")
        for row in dataset
        if row.get("question") or row.get("query")
    ]
    logging.info(f"Loaded {len(all_queries)} queries for RQ3")

    # Resume or start
    if resume and ckpt.exists():
        saved, start_iteration = ckpt.load()
        if saved is not None:
            iteration_results = saved.get("iteration_results", [])
            kg_additions = saved.get("kg_additions", [])
        else:
            iteration_results, kg_additions, start_iteration = [], [], 0
    else:
        iteration_results, kg_additions, start_iteration = [], [], 0

    refiner = RQ3RefinerDualLifting(ns_engine, enable_kg_lifting=enable_kg_lifting)
    refiner.kg_additions = kg_additions

    for iteration in range(start_iteration, n_iterations):
        logging.info(f"\n--- RQ3 [{dataset_label}] Iteration {iteration+1}/{n_iterations} ---")

        start = (iteration * queries_per_iteration) % max(len(all_queries), 1)
        end = start + queries_per_iteration
        if end > len(all_queries):
            iter_queries = all_queries[start:] + all_queries[:end - len(all_queries)]
        else:
            iter_queries = all_queries[start:end]

        if not iter_queries:
            logging.warning(f"No queries for iteration {iteration+1}, skipping")
            continue

        try:
            result = refiner.run_iteration(iter_queries)
            result.iteration = iteration + 1
            iteration_results.append(result)

            logging.info(f"  Claims: {result.n_claims}, Support rate: {result.support_rate:.2%}, "
                         f"UMLS lift: {result.umls_lift_succeeded}, KG lift: {result.kg_lift_succeeded}")

            ckpt.backup()
            ckpt.save(
                {"iteration_results": iteration_results, "kg_additions": refiner.kg_additions},
                iteration + 1,
                n_iterations
            )

        except Exception as e:
            logging.error(f"RQ3 error in iteration {iteration+1}: {e}")
            ckpt.save(
                {"iteration_results": iteration_results, "kg_additions": refiner.kg_additions},
                iteration,
                n_iterations
            )
            raise

    # Compile results using same logic as your existing run_rq3_fixed
    if not iteration_results:
        return {"dataset": dataset_label, "error": "no_iterations_completed"}

    first = iteration_results[0]
    last = iteration_results[-1]
    total_umls_lift = sum(r.umls_lift_succeeded for r in iteration_results)
    total_kg_lift = sum(r.kg_lift_succeeded for r in iteration_results)

    final = {
        "dataset": dataset_label,
        "dataset_key": dataset_key,
        "experiment": "RQ3",
        "experiment_info": {
            "n_iterations": n_iterations,
            "queries_per_iteration": queries_per_iteration,
            "kg_lifting_enabled": enable_kg_lifting,
            "timestamp": str(datetime.now())
        },
        "improvement": {
            "initial_support_rate": first.support_rate,
            "final_support_rate": last.support_rate,
            "change": last.support_rate - first.support_rate,
            "initial_original_support_rate": first.original_support_rate,
            "final_original_support_rate": last.original_support_rate
        },
        "lifting": {
            "umls": {
                "total_attempted": sum(r.umls_lift_attempted for r in iteration_results),
                "total_succeeded": total_umls_lift,
                "success_rate": total_umls_lift / max(sum(r.umls_lift_attempted for r in iteration_results), 1)
            },
            "kg": {
                "total_attempted": sum(r.kg_lift_attempted for r in iteration_results),
                "total_succeeded": total_kg_lift,
                "success_rate": total_kg_lift / max(sum(r.kg_lift_attempted for r in iteration_results), 1)
            }
        },
        "kg_expansion": {
            "novel_claims": len(refiner.kg_additions),
            "candidates": refiner.kg_additions[:20]
        },
        "per_iteration": [asdict(r) for r in iteration_results]
    }

    out_path = Path(output_dir) / dataset_key / "rq3_results.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        json.dump(final, f, indent=2, default=str)

    logging.info(f"\nRQ3 [{dataset_label}] -> Support: {first.support_rate:.2%} -> {last.support_rate:.2%}, "
                 f"saved to {out_path}")
    return final


# ============================================================
# MASTER RUNNER: ALL DATASETS x ALL RQs
# ============================================================

def run_all_datasets(
    ns_engine,
    rqs_to_run: List[str] = ["rq1", "rq2", "rq3"],
    max_queries_rq1: int = 50,
    limit_rq2: int = 100,
    n_iterations_rq3: int = 3,
    queries_per_iteration_rq3: int = 10,
    resume: bool = True,
    output_dir: str = MULTI_EVAL_CHECKPOINT_ROOT
) -> Dict[str, Dict]:
    """
    Run all specified RQs across all registered datasets.

    Args:
        ns_engine: Your NeuroSymbolicHAGRAGRunner instance.
        rqs_to_run: List of RQ keys to run, e.g. ["rq1", "rq2", "rq3"].
        max_queries_rq1: Max queries for RQ1 per dataset.
        limit_rq2: Max queries for RQ2 per dataset.
        n_iterations_rq3: Number of iterations for RQ3.
        queries_per_iteration_rq3: Queries per iteration for RQ3.
        resume: If True, resume from checkpoints; if False, start fresh.
        output_dir: Root directory for all checkpoints and results.

    Returns:
        Nested dict: {dataset_key: {rq: results_dict}}
    """
    print("\n" + "=" * 70)
    print("MULTI-DATASET EVALUATION")
    print("=" * 70)
    print(f"Datasets : {list(MULTI_DATASET_REGISTRY.keys())}")
    print(f"RQs      : {rqs_to_run}")
    print(f"Resume   : {resume}")
    print(f"Output   : {output_dir}")
    print("=" * 70 + "\n")

    all_results = {}

    for dataset_key, dataset_info in MULTI_DATASET_REGISTRY.items():
        dataset_path = dataset_info["path"]
        dataset_label = dataset_info["label"]

        if not Path(dataset_path).exists():
            logging.warning(f"Dataset not found, skipping: {dataset_path}")
            continue

        n_rows = sum(1 for _ in open(dataset_path))
        if n_rows == 0:
            logging.warning(f"Dataset is empty, skipping: {dataset_path}")
            continue

        logging.info(f"\nProcessing dataset: {dataset_label} ({n_rows} QA pairs)")
        all_results[dataset_key] = {}

        # --- RQ1 ---
        if "rq1" in rqs_to_run:
            try:
                rq1_result = run_rq1_on_dataset(
                    ns_engine=ns_engine,
                    dataset_key=dataset_key,
                    dataset_path=dataset_path,
                    dataset_label=dataset_label,
                    max_queries=max_queries_rq1,
                    checkpoint_every=5,
                    resume=resume,
                    output_dir=output_dir
                )
                all_results[dataset_key]["rq1"] = rq1_result
            except Exception as e:
                logging.error(f"RQ1 failed for {dataset_label}: {e}")
                all_results[dataset_key]["rq1"] = {"error": str(e)}

        # --- RQ2 ---
        if "rq2" in rqs_to_run:
            try:
                rq2_result = run_rq2_on_dataset(
                    ns_engine=ns_engine,
                    dataset_key=dataset_key,
                    dataset_path=dataset_path,
                    dataset_label=dataset_label,
                    k=5,
                    limit=limit_rq2,
                    resume=resume,
                    output_dir=output_dir
                )
                all_results[dataset_key]["rq2"] = rq2_result
            except Exception as e:
                logging.error(f"RQ2 failed for {dataset_label}: {e}")
                all_results[dataset_key]["rq2"] = {"error": str(e)}

        # --- RQ3 ---
        if "rq3" in rqs_to_run:
            try:
                rq3_result = run_rq3_on_dataset(
                    ns_engine=ns_engine,
                    dataset_key=dataset_key,
                    dataset_path=dataset_path,
                    dataset_label=dataset_label,
                    n_iterations=n_iterations_rq3,
                    queries_per_iteration=queries_per_iteration_rq3,
                    enable_kg_lifting=True,
                    resume=resume,
                    output_dir=output_dir
                )
                all_results[dataset_key]["rq3"] = rq3_result
            except Exception as e:
                logging.error(f"RQ3 failed for {dataset_label}: {e}")
                all_results[dataset_key]["rq3"] = {"error": str(e)}

    # Save master summary
    summary_path = Path(output_dir) / "multi_dataset_summary.json"
    with open(summary_path, "w") as f:
        json.dump(all_results, f, indent=2, default=str)

    # Print comparison table
    print("\n" + "=" * 70)
    print("MULTI-DATASET RESULTS SUMMARY")
    print("=" * 70)

    for dataset_key, rq_results in all_results.items():
        label = MULTI_DATASET_REGISTRY[dataset_key]["label"]
        print(f"\n  {label}:")

        if "rq1" in rq_results and "metrics" in rq_results["rq1"]:
            m = rq_results["rq1"]["metrics"]
            print(f"    RQ1 -> L0 rate={m.get('l0_evidence_rate', 0):.2%}, "
                  f"VCS={m.get('avg_vcs', 0):.3f}, "
                  f"Proof depth={m.get('avg_proof_depth', 0):.2f}")

        if "rq2" in rq_results and "best_model" in rq_results["rq2"]:
            bm = rq_results["rq2"]["best_model"]
            print(f"    RQ2 -> Best={bm.get('type')}, R²={bm.get('test_r2', 0):.4f}")

        if "rq3" in rq_results and "improvement" in rq_results["rq3"]:
            imp = rq_results["rq3"]["improvement"]
            lft = rq_results["rq3"].get("lifting", {})
            print(f"    RQ3 -> Support: {imp.get('initial_support_rate', 0):.2%} "
                  f"-> {imp.get('final_support_rate', 0):.2%} "
                  f"({imp.get('change', 0):+.2%}), "
                  f"UMLS lift={lft.get('umls', {}).get('total_succeeded', 0)}, "
                  f"KG lift={lft.get('kg', {}).get('total_succeeded', 0)}")

    print(f"\n  Full results saved to: {summary_path}")
    print("=" * 70)

    return all_results


# ============================================================
# USAGE
# ============================================================

print("=" * 60)
print("MULTI-DATASET EVALUATION READY")
print("=" * 60)
print()
print("OPTION 1: Run all RQs on all datasets (auto-resume):")
print("  results = run_all_datasets(ns_engine)")
print()
print("OPTION 2: Run specific RQs only:")
print("  results = run_all_datasets(ns_engine, rqs_to_run=['rq1', 'rq3'])")
print()
print("OPTION 3: Start from scratch (ignore checkpoints):")
print("  results = run_all_datasets(ns_engine, resume=False)")
print()
print("OPTION 4: Single dataset, single RQ:")
print("  run_rq1_on_dataset(ns_engine, 'bioasq',")
print("    './data/bioasq_diabetes.jsonl', 'BioASQ')")
print("=" * 60)

# ============================================================
# RUN (uncomment to execute)
# ============================================================

results = run_all_datasets(
    ns_engine=ns_engine,
    rqs_to_run=["rq1", "rq2", "rq3"],
    max_queries_rq1=1000,
    limit_rq2=1000,
    n_iterations_rq3=3,
    queries_per_iteration_rq3=200,
    resume=True  # Change to False to start from scratch
)

In [ ]:
run_rq3_on_dataset(
    ns_engine=ns_engine,
    dataset_key="bioasq",
    dataset_path="./data/bioasq_diabetes.jsonl",
    dataset_label="BioASQ",
    n_iterations=3,
    queries_per_iteration=200,
    enable_kg_lifting=True,
    resume=False    # clears old 5-query checkpoint and starts fresh
)

In [ ]:
import json

# Master summary  all results in one place, start here
with open("./checkpoints/multi_dataset_eval/multi_dataset_summary.json") as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2))